# 1절 생태환경과 지리: 실습 4 경관진화모델링

- 계산기간: 120–0 ka BP 내 선택
- 시간간격: 1 kyr

- 모델: BIOME4–Pelletier
- 공간자료: 공통 가상유역 (20 m 격자)




In [ ]:
#@title 1.1 Google Colab 환경 준비 (최초 한 번 실행)
import os, sys, subprocess, importlib.util
PB4_THREADS=max(1,min(int(os.cpu_count() or 1),2))
os.environ["OMP_NUM_THREADS"] = str(PB4_THREADS)
os.environ["OMP_DYNAMIC"] = "FALSE"
os.environ["OMP_WAIT_POLICY"] = "PASSIVE"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
if subprocess.run(["bash","-lc","command -v gfortran"],capture_output=True).returncode != 0:
    subprocess.check_call(["apt-get","update","-qq"])
    subprocess.check_call(["apt-get","install","-y","-qq","gfortran"])
required={"numpy":"numpy","pandas":"pandas","matplotlib":"matplotlib","rasterio":"rasterio","scipy":"scipy","xarray":"xarray","geopandas":"geopandas>=1.0","shapely":"shapely>=2.0","pyogrio":"pyogrio","pyproj":"pyproj","netCDF4":"netCDF4","ipywidgets":"ipywidgets","xlsxwriter":"XlsxWriter"}
missing=[spec for module,spec in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])
def _configure_korean_plotting():
    from pathlib import Path
    import matplotlib.pyplot as plt
    import matplotlib.font_manager as fm
    from matplotlib.ft2font import FT2Font

    # 설치 직후에도 디스크의 글꼴을 직접 등록하여 오래된 글꼴 캐시를 피한다.
    font_paths=[
        globals().get("PB4_KOREAN_FONT_FILE", ""),
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/local/share/fonts/truetype/nanum/NanumGothic-Regular.ttf",
    ]
    font_paths.extend(
        path for path in fm.findSystemFonts()
        if any(token in Path(path).name.lower()
               for token in ("nanum", "noto", "malgun", "applegothic"))
    )
    required_chars=set(map(ord,"가상유역토심고도혼효림생물군계거리"))
    for font_path in dict.fromkeys(font_paths):
        if not font_path or not Path(font_path).is_file():
            continue
        if not required_chars.issubset(FT2Font(str(font_path)).get_charmap()):
            continue
        if not any(Path(f.fname)==Path(font_path) for f in fm.fontManager.ttflist):
            fm.fontManager.addfont(str(font_path))
        font_name=fm.FontProperties(fname=str(font_path)).get_name()
        globals()["PB4_KOREAN_FONT_FILE"]=str(font_path)
        plt.rcParams.update({
            "font.family":[font_name,"DejaVu Sans"],
            "font.sans-serif":[font_name,"DejaVu Sans"],
            "axes.unicode_minus":False,
            "pdf.fonttype":42,
            "ps.fonttype":42,
        })
        return font_name
    raise RuntimeError("한글 글꼴을 찾지 못했습니다. 1.1 환경 준비 셀을 다시 실행하십시오.")

try:
    _configure_korean_plotting()
except RuntimeError:
    subprocess.check_call(["apt-get","update","-qq"])
    subprocess.check_call(["apt-get","install","-y","-qq","fonts-nanum"])
    _configure_korean_plotting()
print(f"계산 자원: CPU {os.cpu_count()}개 | BIOME4 OpenMP {PB4_THREADS}스레드")

## 2절 입력자료


In [ ]:
#@title 2.1 모델 및 입력자료 준비
import base64, io, zipfile, hashlib, sys, re
from pathlib import Path
MODEL_DATA_COMMIT="df3949d7e94bc9a24b12a4d9a24487726554a520"
MODEL_BASE_URL=(
    "https://raw.githubusercontent.com/physical-geogrpahy-forever/"
    f"yongneup_nb/{MODEL_DATA_COMMIT}/"
    "pb4_chelsa21k/education/data/pb4studio_beyer_120ka"
)
import urllib.request, json
with urllib.request.urlopen(MODEL_BASE_URL+"/manifest.json",timeout=90) as response:
    MODEL_MANIFEST=json.loads(response.read().decode("utf-8"))
if MODEL_MANIFEST.get("format")!="pb4studio-beyer-120ka-zip-base64-v1":
    raise RuntimeError("지원하지 않는 PB4Studio Beyer 모델 형식입니다.")
_model_b64=[]
for part_name in MODEL_MANIFEST["parts"]:
    with urllib.request.urlopen(f"{MODEL_BASE_URL}/{part_name}",timeout=90) as response:
        _model_b64.append(response.read().decode("ascii").strip())
raw=base64.b64decode("".join(_model_b64))
checksum=hashlib.sha256(raw).hexdigest()
EXPECTED=MODEL_MANIFEST["sha256_zip"]
# SHA256 검증
if checksum != EXPECTED:
    raise RuntimeError("모델 파일 SHA256이 일치하지 않습니다.")
ROOT=Path("/content/yongneup_biome4_teaching") if Path("/content").exists() else Path.cwd()/"yongneup_biome4_teaching"
ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    archive.extractall(ROOT)
MODEL_DIR=ROOT/MODEL_MANIFEST["root_dir"]
assert (MODEL_DIR/"pb4studio"/"runner.py").is_file()
assert (MODEL_DIR/"fortran_src"/"biome4_original_4_2b2.f").is_file()
backend_path=MODEL_DIR/"pb4studio"/"biome4_backend.py"
backend_text=backend_path.read_text(encoding="utf-8")
if not re.search(r'biome\s*=\s*np\.rint\(out\[0\]\)\.astype\(np\.int16\)',backend_text):
    raise RuntimeError("BIOME4 생물군계가 원본 output(1)에서 읽히는지 확인할 수 없습니다.")
if not re.search(r'optpft\s*=\s*np\.rint\(out\[11\]\)\.astype\(np\.int16\)',backend_text):
    raise RuntimeError("BIOME4 optPFT가 원본 output(12)에서 읽히는지 확인할 수 없습니다.")
if re.search(r'out\[\s*489\s*\]',backend_text):
    raise RuntimeError("BIOME4 output(490)을 사용하는 비표준 경로가 감지되었습니다.")
# BIOME4 native biome ID를 스냅샷과 시계열 집계에 추가
runner_path=MODEL_DIR/"pb4studio"/"runner.py"
runner_text=runner_path.read_text(encoding="utf-8")

# Pelletier (2010) 하도·곡저/사면 분류를 스냅샷에 저장하기 위한 import 추가
if "pelletier2010_unit_contributing_area," not in runner_text:
    _geomorph_import_old='''from .pelletier_geomorph import (
    PelletierStrictConfig,
    update_pelletier_strict,
    pelletier_default_stable_dt_kyr,
)'''
    _geomorph_import_new='''from .pelletier_geomorph import (
    PelletierStrictConfig,
    update_pelletier_strict,
    pelletier_default_stable_dt_kyr,
    pelletier2010_unit_contributing_area,
)'''
    if runner_text.count(_geomorph_import_old)!=1:
        raise RuntimeError("runner.py의 Pelletier import 위치를 찾지 못했습니다.")
    runner_text=runner_text.replace(_geomorph_import_old,_geomorph_import_new,1)

summary_marker='        row = summarize_snapshot(int(round(igrid.resolution_m)), float(ka), z, h, veg_code, npp, eemt, igrid.land)\n'
if '"biome4_full_id_counts"' not in runner_text:
    if runner_text.count(summary_marker)!=1:
        raise RuntimeError("runner.py의 BIOME4 집계 위치를 찾지 못했습니다.")
    summary_insert=summary_marker+'''        native_biome=np.asarray(
            veg.get("biome4_full_id_node",np.full(igrid.shape,np.nan)),
            dtype="float32"
        )
        native_land=native_biome[igrid.land]
        native_valid=native_land[np.isfinite(native_land)]
        if native_valid.size:
            native_ids,native_counts=np.unique(native_valid.astype("int16"),return_counts=True)
            row["dominant_biome4_id"]=int(native_ids[np.argmax(native_counts)])
            row["biome4_full_id_counts"]=";".join(
                f"{int(k)}:{int(v)}" for k,v in zip(native_ids,native_counts)
            )
        else:
            row["dominant_biome4_id"]=-1
            row["biome4_full_id_counts"]=""
'''
    runner_text=runner_text.replace(summary_marker,summary_insert,1)

# summary_timeseries에 BIOME4/PFT 진단값 추가
diagnostic_marker='        row.update({\n'
if '"mean_agb_proxy_kg_m2"' not in runner_text:
    if runner_text.count(diagnostic_marker)!=1:
        raise RuntimeError("runner.py의 요약 진단 삽입 위치를 찾지 못했습니다.")
    diagnostic_insert='''        def _pb4_grid(field_name,fill=np.nan):
            arr=np.asarray(
                veg.get(field_name,np.full(igrid.shape,fill)),
                dtype="float64"
            )
            if arr.shape!=igrid.land.shape:
                if arr.size==igrid.land.size:
                    arr=arr.reshape(igrid.land.shape)
                else:
                    return np.full(igrid.land.shape,fill,dtype="float64")
            return arr

        def _pb4_land_mean(field_name):
            arr=_pb4_grid(field_name,np.nan)
            vals=arr[igrid.land & np.isfinite(arr)]
            return float(np.mean(vals)) if vals.size else np.nan

        row["mean_lai"]=_pb4_land_mean("lai_node")
        row["mean_aet_mm_yr"]=_pb4_land_mean("aet_node")
        row["mean_runoff_mm_yr"]=_pb4_land_mean("runoff_node")
        _agb_vals=np.asarray(agb,dtype="float64")
        if _agb_vals.shape!=igrid.land.shape and _agb_vals.size==igrid.land.size:
            _agb_vals=_agb_vals.reshape(igrid.land.shape)
        _agb_valid=_agb_vals[igrid.land & np.isfinite(_agb_vals)]
        row["mean_agb_proxy_kg_m2"]=float(np.mean(_agb_valid)) if _agb_valid.size else np.nan

        for _pid in range(1,14):
            _prefix=f"pft{_pid:02d}_"
            _npp_grid=_pb4_grid(_prefix+"mod_npp_node",np.nan)
            _lai_grid=_pb4_grid(_prefix+"mod_lai_node",np.nan)
            _pass=_pb4_grid(_prefix+"constraint_pass_node",0.0)

            _npp_vals=_npp_grid[igrid.land & np.isfinite(_npp_grid)]
            _lai_vals=_lai_grid[igrid.land & np.isfinite(_lai_grid)]
            _valid=igrid.land & np.isfinite(_pass)

            row[_prefix+"npp_mean"]=float(np.mean(_npp_vals)) if _npp_vals.size else np.nan
            row[_prefix+"lai_mean"]=float(np.mean(_lai_vals)) if _lai_vals.size else np.nan
            row[_prefix+"constraint_pass_fraction"]=(
                float(np.mean(_pass[_valid]>0)) if np.any(_valid) else np.nan
            )

        _opt=_pb4_grid("optpft_node",0.0)
        if np.any(igrid.land & ~np.isfinite(_opt)):
            raise RuntimeError("BIOME4 optPFT에 비유한 격자값이 있습니다.")
        _opt_codes=np.zeros(igrid.land.shape,dtype=int)
        _opt_codes[igrid.land]=np.rint(_opt[igrid.land]).astype(int)
        _bad_opt=igrid.land & ((_opt_codes<0)|(_opt_codes>14))
        if np.any(_bad_opt):
            bad_codes=np.unique(_opt_codes[_bad_opt]).tolist()
            raise RuntimeError(f"BIOME4 optPFT 원본 코드 범위를 벗어났습니다: {bad_codes}")
        _selected=_opt_codes
        for _pid in range(0,15):
            row[f"selected_pft{_pid:02d}_count"]=int(
                np.sum(_selected[igrid.land]==_pid)
            )
'''
    runner_text=runner_text.replace(diagnostic_marker,diagnostic_insert+diagnostic_marker,1)

snapshot_marker='            if "lai_node" in veg:\n'
if 'snapshot_fields["biome4_full_id"]' not in runner_text:
    if runner_text.count(snapshot_marker)<1:
        raise RuntimeError("runner.py의 스냅샷 출력 위치를 찾지 못했습니다.")
    runner_text=runner_text.replace(
        snapshot_marker,
        '            snapshot_fields["biome4_full_id"] = native_biome\n'+snapshot_marker,
        1
    )

# 각 스냅샷에서 모델이 실제 사용하는 Pelletier (2010) valley/hillslope 분류 계산
if 'snapshot_fields["valley_mask"]' not in runner_text:
    _valley_marker='            snapshot_fields["biome4_full_id"] = native_biome\n'
    if runner_text.count(_valley_marker)!=1:
        raise RuntimeError("runner.py의 valley 분류 삽입 위치를 찾지 못했습니다.")
    _valley_insert=_valley_marker+'''            (
                _unit_area_diag,
                _contrib_area_diag,
                _contrib_half_diag,
                _grid_ratio_diag,
                _valley_diag,
            )=pelletier2010_unit_contributing_area(
                cfg=geomorph_cfg,
                z=z,
                land=igrid.land,
                boundary_cfg=boundary_cfg,
                dx_m=float(igrid.resolution_m),
            )
            snapshot_fields["valley_mask"]=_valley_diag.astype("float32")
            snapshot_fields["hillslope_mask"]=(
                igrid.land & (~_valley_diag)
            ).astype("float32")
            snapshot_fields["contributing_area_m2"]=np.where(
                igrid.land,_contrib_area_diag,np.nan
            ).astype("float32")
            snapshot_fields["grid_resolution_ratio_f"]=np.where(
                igrid.land,_grid_ratio_diag,np.nan
            ).astype("float32")
'''
    runner_text=runner_text.replace(_valley_marker,_valley_insert,1)

# 월별 EPPT는 액상수의 열전달 기여만 합산한다.
# 월평균기온 T_m이 0°C 이하이거나 유효강수 P_eff=PPT-AET가 0 이하인 달은
# Rasmussen & Tabor (2007)의 월별 규칙에 따라 EPPT 기여를 0으로 둔다.
# 이 실습은 BIOME4의 월별 AET를 사용하며, 적설 저장·융설수 이월은 모의하지 않는다.
EEMT_COLD_PHASE_METHOD = "sum(Cw * max(T_monthly,0) * max(PPT-AET,0)); snow storage/melt carryover not modeled"
climate_path=MODEL_DIR/"pb4studio"/"climate.py"
climate_text=climate_path.read_text(encoding="utf-8")
_eemt_old_line='    eppt_mj = np.nansum(np.asarray(temp_monthly_C, dtype="float64") * float(cfg.eemt_cw_j_kg_k) * peff_mm, axis=0) / 1.0e6'
_eemt_new_block='''    # Apply both monthly liquid-temperature and positive-effective-precipitation limits.
    _temp_delta_c = np.asarray(temp_monthly_C, dtype="float64")
    _liquid_delta_t = np.maximum(_temp_delta_c, 0.0)
    _liquid_peff_mm = np.maximum(peff_mm, 0.0)
    eppt_mj = np.nansum(
        _liquid_delta_t * float(cfg.eemt_cw_j_kg_k) * _liquid_peff_mm,
        axis=0,
    ) / 1.0e6'''
if climate_text.count(_eemt_old_line)!=1: raise RuntimeError("Expected exactly one original EEMT EPPT expression; found "+climate_text.count(_eemt_old_line));
climate_text=climate_text.replace(_eemt_old_line,_eemt_new_block,1);
compile(climate_text,str(climate_path),"exec");
climate_path.write_text(climate_text,encoding="utf-8");
if (
    _eemt_old_line in climate_text
    or "_liquid_delta_t = np.maximum(_temp_delta_c, 0.0)" not in climate_text
    or "_liquid_peff_mm = np.maximum(peff_mm, 0.0)" not in climate_text
):
    raise RuntimeError("EEMT runtime patch verification failed: both T and P_eff must be non-negative.")

# EEMT가 0 이하인 셀은 지형 계산에 들어가기 전에 원인을 구분해 보고한다.
_runner_eemt_line='        eemt = np.where(igrid.land & np.isfinite(eemt), eemt, np.nan).astype("float32")'
if runner_text.count(_runner_eemt_line)!=1: raise RuntimeError("Expected exactly one runner EEMT line; found "+runner_text.count(_runner_eemt_line));
_runner_eemt_diag='''        eemt = np.where(igrid.land & np.isfinite(eemt), eemt, np.nan).astype("float32")
        _bad_eemt_preflight = igrid.land & (~np.isfinite(eemt) | (eemt <= 0.0))
        if np.any(_bad_eemt_preflight):
            _bad_rc = tuple(int(v) for v in np.argwhere(_bad_eemt_preflight)[0])
            _bad_vals = eemt[_bad_eemt_preflight]
            _bad_bare = int(np.count_nonzero(_bad_eemt_preflight & bare_bedrock))
            _bad_nonfinite = int(np.count_nonzero(~np.isfinite(_bad_vals)))
            _bad_nonpositive = int(np.count_nonzero(np.isfinite(_bad_vals) & (_bad_vals <= 0.0)))
            raise ValueError(
                "EEMT preflight failed after liquid-water and positive-P_eff filtering: "
                f"{int(np.count_nonzero(_bad_eemt_preflight))} bad active-land cell(s); "
                f"bare bedrock={_bad_bare}, nonfinite={_bad_nonfinite}, "
                f"finite nonpositive={_bad_nonpositive}, "
                f"first row={_bad_rc[0]}, col={_bad_rc[1]}, "
                f"finite range=({float(np.nanmin(_bad_vals)) if np.any(np.isfinite(_bad_vals)) else float('nan')}, "
                f"{float(np.nanmax(_bad_vals)) if np.any(np.isfinite(_bad_vals)) else float('nan')})"
            )'''
runner_text=runner_text.replace(_runner_eemt_line,_runner_eemt_diag,1)
if runner_text.count("_bad_eemt_preflight")!=2: raise RuntimeError("Runner EEMT diagnostic insertion failed.");

# 패치한 모델을 새로 불러옴
compile(runner_text,str(runner_path),"exec")
assert 'snapshot_fields["biome4_full_id"]' in runner_text
assert 'snapshot_fields["valley_mask"]' in runner_text
assert 'snapshot_fields["hillslope_mask"]' in runner_text
assert 'pelletier2010_unit_contributing_area' in runner_text
assert '"biome4_full_id_counts"' in runner_text
runner_path.write_text(runner_text,encoding="utf-8")
# 같은 Colab 런타임에서 과거 pb4studio 모듈이 캐시되는 문제 방지
for module_name in list(sys.modules):
    if module_name=="pb4studio" or module_name.startswith("pb4studio."):
        del sys.modules[module_name]
if str(MODEL_DIR) in sys.path:
    sys.path.remove(str(MODEL_DIR))
sys.path.insert(0,str(MODEL_DIR))
from pb4studio.config import RunConfig, RunMode, InputPaths, TimeConfig, ScienceConfig
from pb4studio import runner
# 계산기간 확인
_time_contract=TimeConfig(
    start_ka=120.0,end_ka=0.0,interval_kyr=1.0,
    snapshot_ka=(120.0,90.0,60.0,30.0,0.0)
)
assert _time_contract.snapshot_times_ka()[0]==120.0
assert _time_contract.snapshot_times_ka()[-1]==0.0
# 기후 입력
# 추가 고도감률을 적용하지 않는다.
_original_global_load_climate=runner.load_climate

def _load_global_representative_climate(path,*args,**kwargs):
    climate_path=Path(path)
    if climate_path.suffix.lower()==".csv" and climate_path.is_file():
        probe=pd.read_csv(climate_path)
        required={
            "ka_bp","month","temp_C","precip_mm","cloud_pct","tmin_C"
        }
        if required.issubset(probe.columns):
            df=probe.copy()
            df["ka_bp"]=pd.to_numeric(df["ka_bp"],errors="raise").astype(float)
            df["month"]=pd.to_numeric(df["month"],errors="raise").astype(int)
            for col in ("temp_C","precip_mm","cloud_pct","tmin_C"):
                df[col]=pd.to_numeric(df[col],errors="raise").astype(float)

            if not df["month"].between(1,12).all():
                raise ValueError("기후자료의 month는 1..12여야 합니다.")
            monthly_counts=df.groupby("ka_bp")["month"].nunique()
            if not monthly_counts.eq(12).all():
                bad=monthly_counts.loc[~monthly_counts.eq(12)].to_dict()
                raise ValueError(
                    f"기후자료는 각 시점마다 12개월이 필요합니다: {bad}"
                )
            if df[["ka_bp","month"]].duplicated().any():
                raise ValueError("기후자료에 중복 ka_bp-month 행이 있습니다.")
            if not np.isfinite(
                df[["ka_bp","temp_C","precip_mm","cloud_pct","tmin_C"]].to_numpy(float)
            ).all():
                raise ValueError("기후자료에 비정상 값이 있습니다.")

            df["precip_mm"]=df["precip_mm"].clip(lower=0.0)
            df["cloud_pct"]=df["cloud_pct"].clip(lower=0.0,upper=100.0)
            df["lapse_rate_C_per_m"]=0.0
            df["tmin_lapse_rate_C_per_m"]=0.0
            df["reference_elevation_m"]=0.0
            return df.sort_values(["ka_bp","month"]).reset_index(drop=True)

    # 그 밖의 입력은 PB4Studio 로더 사용
    df=_original_global_load_climate(path,*args,**kwargs).copy()
    for col in ("lapse_rate_C_per_m","tmin_lapse_rate_C_per_m"):
        df[col]=0.0
    df["reference_elevation_m"]=0.0
    return df

runner.load_climate=_load_global_representative_climate
import inspect
assert Path(runner.__file__).resolve()==runner_path.resolve()
runner_source=inspect.getsource(runner.run_single_case)
assert 'snapshot_fields["biome4_full_id"]' in runner_source
assert 'snapshot_fields["valley_mask"]' in runner_source
assert 'snapshot_fields["hillslope_mask"]' in runner_source
assert '"biome4_full_id_counts"' in runner_source
print("모델 및 입력자료 준비 완료")



### 월별 EEMT의 액상수 열항 처리

월별 EPPT는 액상수의 열전달 기여만 합산합니다.

\[
E_{PPT}=\sum_{m=1}^{12} C_w\,\max(T_m,0)\,\max(PPT_m-AET_m,0)
\]

여기서 \(T_m\)은 월평균 기온(°C), \(PPT_m\)는 월강수량, \(AET_m\)는 BIOME4에서 계산한 월별 실제증발산량입니다. 따라서 월평균 기온이 0°C 이하인 달과 유효강수 \(P_{eff}=PPT-AET\)가 0 이하인 달은 EPPT에 기여하지 않습니다. 이 두 조건을 모두 적용하므로 영하 월의 강수나 건조한 달의 음의 유효강수가 연간 EEMT를 낮추는 항으로 들어가지 않습니다.

이 실습은 Rasmussen & Tabor (2007)의 월별 양의 유효강수 처리 원칙을 따르되, 원 논문의 잠재증발산량(PET) 대신 BIOME4의 월별 AET를 사용한 결합 구현입니다. 적설 저장과 융설수의 다음 달 이월은 별도로 모의하지 않습니다.

근거: Rasmussen & Tabor (2007), *Applying a quantitative pedogenic energy model across a range of environmental gradients*, DOI: [10.2136/sssaj2007.0051](https://doi.org/10.2136/sssaj2007.0051); Pelletier et al. (2013), Eqs. (1)–(2), DOI: [10.1002/jgrf.20046](https://doi.org/10.1002/jgrf.20046). 적설 저장이 적은 유역을 대상으로 한 EEMT 적용 범위는 Rasmussen & Gallo (2013), DOI: [10.5194/hess-17-3389-2013](https://doi.org/10.5194/hess-17-3389-2013)도 함께 참조하십시오.


In [ ]:
#@title 2.2 기후자료 불러오기
import io, lzma, hashlib, urllib.request, base64, json
import pandas as pd
import numpy as np

CLIMATE_DATA_COMMIT="742398db789c2be4e96368532994a3ba77c11e57"
CLIMATE_BASE_URL=(
    "https://raw.githubusercontent.com/physical-geogrpahy-forever/"
    f"yongneup_nb/{CLIMATE_DATA_COMMIT}/"
    "pb4_chelsa21k/education/data/beyer_landmark_global_120ka"
)

with urllib.request.urlopen(CLIMATE_BASE_URL+"/manifest.json",timeout=90) as response:
    climate_manifest=json.loads(response.read().decode("utf-8"))

if climate_manifest.get("format")!="pb4-beyer-landmark-xor-f32-lzma-base64-v1":
    raise RuntimeError("지원하지 않는 기후자료 형식입니다.")

CLIMATE_CACHE=ROOT/"beyer_landmark_global_120ka.xor.xz"
compressed=None
if CLIMATE_CACHE.is_file():
    cached=CLIMATE_CACHE.read_bytes()
    if hashlib.sha256(cached).hexdigest()==climate_manifest["sha256_lzma"]:
        compressed=cached
    else:
        CLIMATE_CACHE.unlink()

if compressed is None:
    encoded_parts=[]
    for part_name in climate_manifest["parts"]:
        with urllib.request.urlopen(
            f"{CLIMATE_BASE_URL}/{part_name}",timeout=90
        ) as response:
            encoded_parts.append(response.read().decode("ascii").strip())
    compressed=base64.b64decode("".join(encoded_parts))
    if hashlib.sha256(compressed).hexdigest()!=climate_manifest["sha256_lzma"]:
        raise RuntimeError("기후자료 SHA256 불일치")
    CLIMATE_CACHE.write_bytes(compressed)

raw_xor=lzma.decompress(compressed)
if hashlib.sha256(raw_xor).hexdigest()!=climate_manifest["sha256_raw_xor"]:
    raise RuntimeError("기후자료 압축 해제 후 SHA256 불일치")

shape=tuple(int(x) for x in climate_manifest["shape"])
xor_values=np.frombuffer(raw_xor,dtype="<u4").reshape(shape)
value_bits=np.bitwise_xor.accumulate(xor_values,axis=-1)
climate_values=value_bits.view("<f4")

variable_names=list(climate_manifest["variable_names"])
climate_ages=np.asarray(climate_manifest["ka_bp"],dtype=int)
climate_months=np.asarray(climate_manifest["months"],dtype=int)
representatives=list(climate_manifest["representatives"])

if climate_values.shape!=(
    len(representatives),len(variable_names),
    len(climate_months),len(climate_ages)
):
    raise RuntimeError("기후자료 배열 크기와 manifest가 일치하지 않습니다.")

records=[]
for ri,meta in enumerate(representatives):
    for ai,age in enumerate(climate_ages):
        for mi,month in enumerate(climate_months):
            rec=dict(meta)
            rec["ka_bp"]=int(age)
            rec["month"]=int(month)
            for vi,var_name in enumerate(variable_names):
                rec[var_name]=float(climate_values[ri,vi,mi,ai])
            records.append(rec)

CLIMATE_MASTER=pd.DataFrame.from_records(records)
required_columns={
    "representative_id","landmark_name_ko","landmark_type",
    "koppen_reference_0ka","koppen_name",
    "source_grid_lon","source_grid_lat",
    "distance_from_anchor_km","robustness_pct",
    "ka_bp","month",
    "temp_C","precip_mm","cloud_pct","relative_humidity","wind_speed"
}
missing=required_columns-set(CLIMATE_MASTER.columns)
if missing:
    raise ValueError(f"기후자료 열 누락: {sorted(missing)}")

CLIMATE_MASTER["ka_bp"]=CLIMATE_MASTER["ka_bp"].astype(int)
CLIMATE_MASTER["month"]=CLIMATE_MASTER["month"].astype(int)

climate_places=(
    CLIMATE_MASTER.sort_values(["representative_id","ka_bp"])
    .drop_duplicates("representative_id")
    [[
        "representative_id","landmark_name_ko","landmark_type",
        "koppen_reference_0ka","koppen_name",
        "source_grid_lon","source_grid_lat",
        "distance_from_anchor_km","robustness_pct"
    ]]
    .reset_index(drop=True)
)

# 대한민국을 먼저, 이후 지명을 가나다/문자열 순으로 표시
climate_places["_display_priority"]=np.select(
    [
        climate_places.representative_id.eq("KR_SEOUL"),
        climate_places.representative_id.eq("KR_JEJU")
    ],
    [0,1],
    default=2
)
climate_places=(
    climate_places
    .sort_values(["_display_priority","landmark_name_ko","representative_id"])
    .drop(columns="_display_priority")
    .reset_index(drop=True)
)

def _coord_label(lat,lon):
    ns="N" if float(lat)>=0 else "S"
    ew="E" if float(lon)>=0 else "W"
    return f"{abs(float(lat)):.2f}°{ns}, {abs(float(lon)):.2f}°{ew}"

climate_places["region_name"]=climate_places["landmark_name_ko"]
climate_places["place_label"]=[
    f"{row.landmark_name_ko} ({_coord_label(row.source_grid_lat,row.source_grid_lon)}) · {row.representative_id}"
    for row in climate_places.itertuples(index=False)
]

# 쾨펜 기후형 설명
_KOPPEN_KO={
    "Af":"열대 우림 기후",
    "Am":"열대 몬순 기후",
    "Aw":"사바나 기후",
    "BWh":"사막 기후",
    "BWk":"사막 기후",
    "BSh":"스텝 기후",
    "BSk":"스텝 기후",
    "Csa":"지중해성 기후",
    "Csb":"지중해성 기후",
    "Csc":"지중해성 기후",
    "Cwa":"온대 겨울 건조 기후",
    "Cwb":"온대 겨울 건조 기후",
    "Cwc":"온대 겨울 건조 기후",
    "Cfa":"온난 습윤 기후",
    "Cfb":"서안 해양성 기후",
    "Cfc":"아극 해양성 기후",
    "Dsa":"고지 지중해성 기후",
    "Dsb":"고지 지중해성 기후",
    "Dsc":"고지 지중해성 기후",
    "Dsd":"고지 지중해성 기후",
    "Dwa":"냉대 겨울 건조 기후",
    "Dwb":"냉대 겨울 건조 기후",
    "Dwc":"냉대 겨울 건조 기후",
    "Dwd":"냉대 겨울 건조 기후",
    "Dfa":"냉대 습윤 기후",
    "Dfb":"냉대 습윤 기후",
    "Dfc":"냉대 습윤 기후",
    "Dfd":"냉대 습윤 기후",
    "ET":"툰드라 기후",
    "EF":"빙설 기후"
}

_KOPPEN_SUBTYPE={
    "Af":"건기 없음",
    "Am":"짧은 건기",
    "Aw":"겨울 건조",
    "BWh":"고온형 (h)",
    "BWk":"한랭형 (k)",
    "BSh":"고온형 (h)",
    "BSk":"한랭형 (k)",
    "Csa":"고온 여름형 (a)",
    "Csb":"온난 여름형 (b)",
    "Csc":"냉량 여름형 (c)",
    "Cwa":"고온 여름형 (a)",
    "Cwb":"온난 여름형 (b)",
    "Cwc":"냉량 여름형 (c)",
    "Cfa":"고온 여름형 (a)",
    "Cfb":"온난 여름형 (b)",
    "Cfc":"냉량 여름형 (c)",
    "Dsa":"고온 여름형 (a)",
    "Dsb":"온난 여름형 (b)",
    "Dsc":"냉량 여름형 (c)",
    "Dsd":"극한 겨울형 (d)",
    "Dwa":"고온 여름형 (a)",
    "Dwb":"온난 여름형 (b)",
    "Dwc":"냉량 여름형 (c)",
    "Dwd":"극한 겨울형 (d)",
    "Dfa":"고온 여름형 (a)",
    "Dfb":"온난 여름형 (b)",
    "Dfc":"냉량 여름형 (c)",
    "Dfd":"극한 겨울형 (d)",
    "ET":"툰드라형",
    "EF":"빙설형"
}

_KOPPEN_DESC={
    "Af":"연중 고온다습하며 뚜렷한 건기가 없다.",
    "Am":"연중 고온이며 짧은 건기와 뚜렷한 우기가 나타난다.",
    "Aw":"연중 고온이며 겨울철 건기가 뚜렷한 사바나 기후이다.",
    "BWh":"사막 기후 가운데 연평균 기온이 높은 고온형이다.",
    "BWk":"사막 기후 가운데 연평균 기온이 낮은 한랭형이다.",
    "BSh":"스텝 기후 가운데 연평균 기온이 높은 고온형이다.",
    "BSk":"스텝 기후 가운데 연평균 기온이 낮은 한랭형이다.",
    "Csa":"여름이 덥고 건조하며 겨울은 온화하고 상대적으로 습하다.",
    "Csb":"여름이 온난하고 건조하며 겨울은 온화하고 상대적으로 습하다.",
    "Csc":"여름이 짧고 서늘하며 건조한 지중해성 기후의 냉량형이다.",
    "Cwa":"겨울이 건조하고 여름이 더운 온대 기후이다.",
    "Cwb":"겨울이 건조하고 여름이 온난한 온대 기후이다.",
    "Cwc":"겨울이 건조하고 여름이 짧고 서늘한 온대 기후이다.",
    "Cfa":"건기가 뚜렷하지 않고 여름이 더운 온난 습윤 기후이다.",
    "Cfb":"연중 강수가 비교적 고르게 나타나며 여름이 온난한 서안 해양성 기후이다.",
    "Cfc":"연중 습윤하지만 여름이 짧고 서늘한 아극 해양성 기후이다.",
    "Dsa":"겨울이 춥고 여름이 덥고 건조한 냉대 기후이다.",
    "Dsb":"겨울이 춥고 여름이 온난하고 건조한 냉대 기후이다.",
    "Dsc":"겨울이 춥고 여름이 짧고 서늘하며 건조한 냉대 기후이다.",
    "Dsd":"겨울이 매우 춥고 여름이 짧고 서늘하며 건조한 냉대 기후이다.",
    "Dwa":"겨울이 춥고 건조하며 여름이 더운 냉대 기후이다.",
    "Dwb":"겨울이 춥고 건조하며 여름이 온난한 냉대 기후이다.",
    "Dwc":"겨울이 매우 춥고 건조하며 여름이 짧고 서늘한 냉대 기후이다.",
    "Dwd":"겨울이 극도로 춥고 건조하며 여름이 짧고 서늘한 냉대 기후이다.",
    "Dfa":"겨울이 춥고 건기가 뚜렷하지 않으며 여름이 더운 냉대 습윤 기후이다.",
    "Dfb":"겨울이 춥고 연중 습윤하며 여름이 온난한 냉대 습윤 기후이다.",
    "Dfc":"겨울이 길고 춥고 연중 습윤하며 여름이 짧고 서늘한 냉대 습윤 기후이다.",
    "Dfd":"겨울이 극도로 춥고 연중 습윤하며 여름이 짧고 서늘한 냉대 습윤 기후이다.",
    "ET":"최난월 평균기온이 0°C 이상 10°C 미만인 한대 기후이다.",
    "EF":"최난월 평균기온이 0°C 미만인 빙설 기후이다."
}

assert CLIMATE_MASTER.groupby(
    ["representative_id","ka_bp"]
)["month"].nunique().eq(12).all()
assert CLIMATE_MASTER.ka_bp.min()==0 and CLIMATE_MASTER.ka_bp.max()==120
climate_types=sorted(climate_places.koppen_reference_0ka.unique())
print(
    "기후자료: 120–0 ka BP"
    ""
)
koppen_explanation=pd.DataFrame([
    {
        "쾨펜 코드":z,
        "영문명":climate_places.loc[
            climate_places.koppen_reference_0ka==z,"koppen_name"
        ].iloc[0],
        "국문명":_KOPPEN_KO.get(z,z),
        "세부형":_KOPPEN_SUBTYPE.get(z,""),
        "설명":_KOPPEN_DESC.get(z,"")
    }
    for z in climate_types
])
print("\n쾨펜 기후형 설명")
print("a/b/c/d 및 h/k는 세부형으로 구분.")
display(koppen_explanation)

# 0 ka BP 지점별 월별 기후
import matplotlib.pyplot as plt
_configure_korean_plotting()
import math

print("\n0 ka BP 지점별 월별 기온·강수량")
_ncols=4
_nrows=math.ceil(len(climate_places)/_ncols)
_fig,_axes=plt.subplots(
    _nrows,_ncols,
    figsize=(16,3.1*_nrows),
    constrained_layout=True
)
_axes=np.atleast_1d(_axes).ravel()

for ax,row in zip(_axes,climate_places.itertuples(index=False)):
    current=(
        CLIMATE_MASTER.loc[
            (CLIMATE_MASTER.representative_id==row.representative_id) &
            (CLIMATE_MASTER.ka_bp==0),
            ["month","temp_C","precip_mm"]
        ]
        .sort_values("month")
    )
    if len(current)!=12:
        raise ValueError(f"{row.representative_id}: 0 ka BP 월별 기후자료가 12개월이 아닙니다.")

    ax2=ax.twinx()
    ax2.bar(
        current["month"],current["precip_mm"],
        width=0.72,alpha=0.30
    )
    ax.plot(
        current["month"],current["temp_C"],
        marker="o",markersize=2.8,linewidth=1.5
    )
    ax.set_xticks(range(1,13))
    ax.set_xlim(0.4,12.6)
    ax.set_xlabel("월")
    ax.set_ylabel("기온 (°C)")
    ax2.set_ylabel("강수량 (mm)")
    ax.set_title(
        f"{row.region_name} · {row.koppen_reference_0ka}\n"
        f"{_coord_label(row.source_grid_lat,row.source_grid_lon)} · {row.representative_id}",
        fontsize=9
    )
    ax.grid(axis="y",alpha=.2)

for ax in _axes[len(climate_places):]:
    ax.remove()

plt.show()

print("\n기후 지점")
display(
    climate_places[[
        "representative_id","landmark_name_ko","landmark_type",
        "koppen_reference_0ka","koppen_name",
        "source_grid_lat","source_grid_lon",
        "distance_from_anchor_km","robustness_pct"
    ]].rename(columns={
        "representative_id":"기후 지점 ID",
        "landmark_name_ko":"지명",
        "landmark_type":"유형",
        "koppen_reference_0ka":"쾨펜 코드",
        "koppen_name":"기후형",
        "source_grid_lat":"위도",
        "source_grid_lon":"경도",
        "distance_from_anchor_km":"기준좌표와 거리 (km)",
        "robustness_pct":"기후형 안정성 (%)"
    })
)


## 3절 가상 유역 설정

DEM에서 D8 유향과 집수면적을 계산하고, 목표 유역면적에 가장 가까운 하도 셀을 유출구로 선택한 뒤 그 상류 기여역을 가상유역으로 사용합니다. 모든 시나리오에는 동일한 유역과 유출구를 적용합니다.


In [ ]:
#@title 3.1 가상 유역 설정
import ipywidgets as widgets
import numpy as np
import matplotlib.pyplot as plt
_configure_korean_plotting()
from scipy.ndimage import gaussian_filter
from IPython.display import display, clear_output
import heapq

MAX_GRID_N=40
MIN_GRID_N=12
MAX_BASIN_AREA_KM2=0.375

basin_area_widget=widgets.BoundedFloatText(
    value=0.0936,min=0.05,max=MAX_BASIN_AREA_KM2,step=0.01,description="목표 유역면적 (km²)",
    style={"description_width":"185px"},layout=widgets.Layout(width="380px"))
basin_cellsize_widget=widgets.BoundedFloatText(
    value=20.0,min=5.0,max=100.0,step=5.0,description="격자 해상도 (m)",
    style={"description_width":"185px"},layout=widgets.Layout(width="380px"))
basin_base_elev_widget=widgets.BoundedFloatText(
    value=350.0,min=0.0,max=3000.0,step=10.0,description="기준 고도 (m)",
    style={"description_width":"185px"},layout=widgets.Layout(width="380px"))
basin_long_relief_widget=widgets.BoundedFloatText(
    value=55.0,min=5.0,max=1000.0,step=5.0,description="유역 기복량 (m)",
    style={"description_width":"185px"},layout=widgets.Layout(width="380px"))
basin_soil_depth_widget=widgets.BoundedFloatText(
    value=0.55,min=0.05,max=2.0,step=0.05,description="기본 토심 (m)",
    style={"description_width":"185px"},layout=widgets.Layout(width="380px"))

basin_seed_widget=widgets.BoundedIntText(
    value=12,min=0,max=9999,step=1,description="지형 시드",
    style={"description_width":"185px"},layout=widgets.Layout(width="380px"))
basin_elongation_widget=widgets.FloatSlider(
    value=1.15,min=0.70,max=1.80,step=0.05,description="유역 길쭉함",
    continuous_update=False,readout_format=".2f",
    style={"description_width":"185px"},layout=widgets.Layout(width="380px"))
basin_irregularity_widget=widgets.FloatSlider(
    value=0.12,min=0.0,max=0.30,step=0.01,description="능선·하곡 불규칙성",
    continuous_update=False,readout_format=".2f",
    style={"description_width":"185px"},layout=widgets.Layout(width="380px"))
basin_noise_widget=widgets.FloatSlider(
    value=3.0,min=0.0,max=20.0,step=0.5,description="미세지형 요철 (m)",
    continuous_update=False,readout_format=".1f",
    style={"description_width":"185px"},layout=widgets.Layout(width="380px"))
basin_grid_info=widgets.HTML()

basin_help=widgets.HTML(
    value="""
    <table style="border-collapse:collapse; margin-top:8px; font-size:13px;">
      <tr>
        <th style="text-align:left; padding:4px 12px 4px 0;">항목</th>
        <th style="text-align:left; padding:4px 18px 4px 0;">의미</th>
        <th style="text-align:left; padding:4px 0;">값 증가 시</th>
      </tr>
      <tr><td>목표 유역면적</td><td>유출구의 목표 상류 기여면적</td><td>유역 확대</td></tr>
      <tr><td>격자 해상도</td><td>DEM 셀 크기</td><td>공간 해상도 감소, 계산량 감소</td></tr>
      <tr><td>기준 고도</td><td>유출구 기준 고도</td><td>DEM 전체 고도 상승</td></tr>
      <tr><td>유역 기복량</td><td>가상 DEM의 고저차</td><td>사면경사 증가</td></tr>
      <tr><td>기본 토심</td><td>초기 토양 두께</td><td>초기 토심 증가</td></tr>
      <tr><td>지형 시드</td><td>가상 DEM 생성 난수</td><td>값의 크기 자체에 물리적 의미 없음</td></tr>
      <tr><td>유역 길쭉함</td><td>주곡 방향 장축 강조 정도</td><td>길고 좁은 지형</td></tr>
      <tr><td>능선·하곡 불규칙성</td><td>대규모 지형 굴곡</td><td>능선·하곡 굴곡 증가</td></tr>
      <tr><td>미세지형 요철</td><td>DEM의 소규모 고도 변동</td><td>국지 기복 증가</td></tr>
    </table>
    """
)

_D8=[
    (-1,-1,np.sqrt(2.0)),(-1,0,1.0),(-1,1,np.sqrt(2.0)),
    (0,-1,1.0),                         (0,1,1.0),
    (1,-1,np.sqrt(2.0)), (1,0,1.0), (1,1,np.sqrt(2.0)),
]

def _provisional_dem(n,seed,elong,irr,noise_m,base,relief):
    rng=np.random.default_rng(int(seed))
    rr,cc=np.indices((n,n),dtype=float)

    y_up=(n-1-rr)/max(n-1,1)
    x=(cc-(n-1)/2.0)/max((n-1)/2.0,1.0)

    phase=rng.uniform(0,2*np.pi)
    meander_amp=min(0.28,max(0.0,1.5*float(irr)))
    valley_center=(
        meander_amp
        * np.sin(np.pi*y_up+phase)
        * np.sin(np.pi*y_up)
    )

    lateral_scale=max(0.55,1.0/max(float(elong),0.1))
    lateral=np.abs((x-valley_center)/lateral_scale)**1.65

    phase2=rng.uniform(0,2*np.pi)
    broad_texture=np.sin(
        2*np.pi*(0.65*x+0.35*y_up)+phase2
    )

    noise=rng.normal(size=(n,n))
    noise=gaussian_filter(
        noise,sigma=max(0.8,n/22.0),mode="reflect"
    )
    noise-=float(np.mean(noise))
    sd=float(np.std(noise))
    if sd>0:
        noise/=sd

    z=(
        float(base)
        + float(relief)*(
            0.62*y_up
            +0.30*lateral
            +0.55*float(irr)*broad_texture
        )
        + float(noise_m)*noise
    )
    return z.astype("float64")

def _priority_flood_fill(dem):
    z=np.asarray(dem,dtype="float64").copy()
    nrow,ncol=z.shape
    visited=np.zeros(z.shape,dtype=bool)
    heap=[]

    for r in range(nrow):
        for c in (0,ncol-1):
            if not visited[r,c]:
                visited[r,c]=True
                heapq.heappush(heap,(float(z[r,c]),r,c))
    for c in range(ncol):
        for r in (0,nrow-1):
            if not visited[r,c]:
                visited[r,c]=True
                heapq.heappush(heap,(float(z[r,c]),r,c))

    eps=max(float(np.nanmax(z)-np.nanmin(z)),1.0)*1e-9
    while heap:
        elev,r,c=heapq.heappop(heap)
        for dr,dc,_ in _D8:
            rr=r+dr
            cc=c+dc
            if rr<0 or rr>=nrow or cc<0 or cc>=ncol or visited[rr,cc]:
                continue
            visited[rr,cc]=True
            if z[rr,cc]<=elev:
                z[rr,cc]=elev+eps
            heapq.heappush(heap,(float(z[rr,cc]),rr,cc))
    return z

def _d8_route_and_accumulation(dem_filled):
    z=np.asarray(dem_filled,dtype="float64")
    nrow,ncol=z.shape
    receiver=np.full((nrow,ncol,2),-1,dtype=np.int16)

    # 임시 DEM의 외곽은 유역분석 단계에서만 열린 경계로 취급한다.
    for r in range(1,nrow-1):
        for c in range(1,ncol-1):
            best=None
            best_slope=0.0
            for dr,dc,distance in _D8:
                rr=r+dr
                cc=c+dc
                slope=(z[r,c]-z[rr,cc])/distance
                if slope>best_slope:
                    best_slope=float(slope)
                    best=(rr,cc)
            if best is None:
                raise RuntimeError(
                    f"D8 유향을 결정할 수 없는 내부 셀이 있습니다: ({r}, {c})"
                )
            receiver[r,c]=best

    accumulation=np.ones((nrow,ncol),dtype=np.int32)
    order=np.argsort(z.ravel())[::-1]
    for flat in order:
        r,c=np.unravel_index(int(flat),z.shape)
        rr,cc=receiver[r,c]
        if rr>=0:
            accumulation[rr,cc]+=accumulation[r,c]
    return receiver,accumulation

def _upstream_mask(receiver,outlet):
    nrow,ncol=receiver.shape[:2]
    children=[[] for _ in range(nrow*ncol)]
    for r in range(nrow):
        for c in range(ncol):
            rr,cc=receiver[r,c]
            if rr>=0:
                children[int(rr)*ncol+int(cc)].append(r*ncol+c)

    start=int(outlet[0])*ncol+int(outlet[1])
    mask_flat=np.zeros(nrow*ncol,dtype=bool)
    mask_flat[start]=True
    stack=[start]
    while stack:
        parent=stack.pop()
        for child in children[parent]:
            if not mask_flat[child]:
                mask_flat[child]=True
                stack.append(child)
    return mask_flat.reshape((nrow,ncol))

def _build_virtual_basin(return_diagnostics=False):
    target_km2=float(basin_area_widget.value)
    cell=float(basin_cellsize_widget.value)
    target_cells=target_km2*1_000_000.0/(cell*cell)

    if target_cells>MAX_GRID_N*MAX_GRID_N:
        max_area=MAX_GRID_N*MAX_GRID_N*cell*cell/1_000_000.0
        raise ValueError(
            f"현재 {cell:g} m 해상도와 최대 {MAX_GRID_N}×{MAX_GRID_N} 격자에서는 "
            f"{target_km2:g} km² 유역을 만들 수 없습니다. "
            f"이론적 최대 면적은 {max_area:.4f} km²입니다."
        )

    seed=int(basin_seed_widget.value)
    elong=float(basin_elongation_widget.value)
    irr=float(basin_irregularity_widget.value)
    noise_m=float(basin_noise_widget.value)
    base=float(basin_base_elev_widget.value)
    relief=float(basin_long_relief_widget.value)

    best=None
    for n in range(MIN_GRID_N,MAX_GRID_N+1):
        raw=_provisional_dem(
            n,seed,elong,irr,noise_m,base,relief
        )
        filled=_priority_flood_fill(raw)
        receiver,accumulation=_d8_route_and_accumulation(filled)

        # 임시 DEM 내부의 실제 하도 셀을 outlet 후보로 평가한다.
        # 해당 셀의 D8 집수면적이 사용자의 목표 유역면적에 가장 가까운 지점을 선택한다.
        for r in range(1,n-1):
            for c in range(1,n-1):
                if receiver[r,c,0]<0:
                    continue
                count=int(accumulation[r,c])
                if count<8:
                    continue
                key=(abs(count-target_cells),n,-r)
                if best is None or key<best[0]:
                    best=(
                        key,n,raw,filled,receiver,accumulation,
                        (int(r),int(c)),count
                    )

    if best is None:
        raise RuntimeError("D8 유역분석에서 유출구 후보를 찾지 못했습니다.")

    _,n,raw,filled,receiver,accumulation,outlet,count=best
    basin_mask=_upstream_mask(receiver,outlet)

    if int(basin_mask.sum())!=int(count):
        raise RuntimeError(
            "D8 집수면적과 delineation된 상류 유역의 격자 수가 일치하지 않습니다."
        )

    outlet_r,outlet_c=outlet
    down_r,down_c=receiver[outlet_r,outlet_c]
    if down_r<0 or basin_mask[int(down_r),int(down_c)]:
        raise RuntimeError("선택한 유출구가 최종 유역의 하류 경계에 있지 않습니다.")

    actual_km2=float(basin_mask.sum())*cell*cell/1_000_000.0
    relative_error=abs(actual_km2-target_km2)/target_km2
    if relative_error>0.10:
        raise ValueError(
            f"현재 지형·해상도에서 목표 {target_km2:.4f} km²에 가까운 유역을 찾지 못했습니다. "
            f"가장 가까운 D8 유역은 {actual_km2:.4f} km²입니다. "
            "격자 해상도 또는 지형 시드를 조정하십시오."
        )

    # 유역분석에 사용한 수문보정 DEM을 그대로 모델 입력으로 사용한다.
    # outlet 고도만 사용자가 지정한 값으로 평행이동하여 유향은 유지한다.
    shift=base-float(filled[outlet_r,outlet_c])
    dem=filled+shift
    dem=np.where(basin_mask,dem,np.nan).astype("float32")

    if return_diagnostics:
        return (
            dem,basin_mask,(outlet_r,outlet_c),
            accumulation.astype("int32"),
            receiver,
            filled.astype("float64"),
        )
    return dem,basin_mask,(outlet_r,outlet_c)

basin_preview=widgets.Output()

def _update_basin_preview(change=None):
    with basin_preview:
        clear_output(wait=True)
        try:
            (
                dem,mask,(outlet_r,outlet_c),
                flow_acc,receiver,filled
            )=_build_virtual_basin(return_diagnostics=True)
        except Exception as exc:
            basin_grid_info.value="<b>격자 계산 불가</b>"
            print("설정 오류:",exc)
            return

        n=dem.shape[0]
        cell=float(basin_cellsize_widget.value)
        actual_area_km2=float(mask.sum())*cell*cell/1_000_000.0
        target_area_km2=float(basin_area_widget.value)
        width=n*cell
        masked=np.ma.masked_invalid(dem)

        gy,gx=np.gradient(np.where(mask,dem,np.nan))
        slope=np.degrees(np.arctan(np.sqrt((gx/cell)**2+(gy/cell)**2)))
        slope_vals=slope[mask & np.isfinite(slope)]
        mean_slope=float(np.mean(slope_vals)) if slope_vals.size else np.nan
        max_slope=float(np.max(slope_vals)) if slope_vals.size else np.nan
        rr_valid,cc_valid=np.where(mask)
        basin_height=(rr_valid.max()-rr_valid.min()+1)*cell if rr_valid.size else np.nan
        basin_width=(cc_valid.max()-cc_valid.min()+1)*cell if cc_valid.size else np.nan
        shape_ratio=max(basin_height,basin_width)/max(min(basin_height,basin_width),cell)

        basin_grid_info.value=(
            f"<b>자동 격자:</b> {n}×{n} &nbsp; "
            f"<b>D8 유역면적:</b> {actual_area_km2:.4f} km²<br>"
            f"목표 {target_area_km2:.4f} km² · "
            f"차이 {100*(actual_area_km2-target_area_km2)/target_area_km2:+.1f}%<br>"
            f"평균경사 {mean_slope:.1f}° · 최대경사 {max_slope:.1f}° · "
            f"경계상자 장단축비 {shape_ratio:.2f}"
        )

        x=np.arange(n,dtype=float)*cell
        y=np.arange(n,dtype=float)*cell
        extent=[
            -0.5*cell,(n-0.5)*cell,
            (n-0.5)*cell,-0.5*cell
        ]

        fig,axes=plt.subplots(
            1,3,figsize=(16.5,5.6),constrained_layout=True
        )

        # DEM
        ax=axes[0]
        im0=ax.imshow(
            masked,cmap="terrain",origin="upper",extent=extent
        )
        ax.contour(
            x,y,mask.astype(float),
            levels=[0.5],colors="black",linewidths=1.2
        )

        channel_threshold=max(5,int(round(0.05*mask.sum())))
        channel=mask & (flow_acc>=channel_threshold)
        cr,cc=np.where(channel)
        if len(cr):
            ax.scatter(
                cc*cell,cr*cell,
                s=8,marker="s",linewidths=0,
                alpha=0.75,label="D8 하도"
            )

        ax.plot(
            outlet_c*cell,outlet_r*cell,
            marker="v",markersize=11,label="open 유출구"
        )
        ax.set_title(
            f"DEM · 목표 {target_area_km2:g} km² · 실제 {actual_area_km2:.3f} km²"
        )
        ax.set_xlabel("거리 (m)")
        ax.set_ylabel("거리 (m)")
        ax.set_aspect("equal")
        ax.legend(frameon=False,loc="lower left",fontsize=8)
        fig.colorbar(im0,ax=ax,label="고도 (m)",fraction=.05)

        # D8 유향도
        ax=axes[1]
        ax.imshow(
            np.where(mask,1.0,np.nan),
            cmap="Greys",origin="upper",extent=extent,
            vmin=0,vmax=1,alpha=0.08
        )
        ax.contour(
            x,y,mask.astype(float),
            levels=[0.5],colors="black",linewidths=1.2
        )

        qx=[]; qy=[]; qu=[]; qv=[]
        for r in range(n):
            for c in range(n):
                if not mask[r,c]:
                    continue
                rr,cc=receiver[r,c]
                if rr<0 or cc<0:
                    continue
                if not mask[int(rr),int(cc)] and not (
                    r==outlet_r and c==outlet_c
                ):
                    continue
                qx.append(c*cell)
                qy.append(r*cell)
                qu.append((int(cc)-c)*cell)
                qv.append((int(rr)-r)*cell)

        if qx:
            ax.quiver(
                qx,qy,qu,qv,
                angles="xy",scale_units="xy",scale=1,
                width=0.0030,headwidth=3.8,headlength=5.0,
                headaxislength=4.5,minlength=0.1
            )

        ax.plot(
            outlet_c*cell,outlet_r*cell,
            marker="v",markersize=11,label="open 유출구"
        )
        ax.set_title("D8 유향도")
        ax.set_xlabel("거리 (m)")
        ax.set_ylabel("거리 (m)")
        ax.set_aspect("equal")
        ax.legend(frameon=False,loc="lower left",fontsize=8)

        # D8 흐름누적도
        ax=axes[2]
        upstream_area_m2=np.where(
            mask,flow_acc.astype(float)*(cell**2),np.nan
        )
        log_upstream_area=np.log10(upstream_area_m2)
        im2=ax.imshow(
            log_upstream_area,
            cmap="Blues",origin="upper",extent=extent
        )
        ax.contour(
            x,y,mask.astype(float),
            levels=[0.5],colors="black",linewidths=1.2
        )
        ax.plot(
            outlet_c*cell,outlet_r*cell,
            marker="v",markersize=11,label="open 유출구"
        )
        ax.set_title("D8 흐름누적도")
        ax.set_xlabel("거리 (m)")
        ax.set_ylabel("거리 (m)")
        ax.set_aspect("equal")
        ax.legend(frameon=False,loc="lower left",fontsize=8)
        fig.colorbar(
            im2,ax=ax,
            label="log10(상류 기여면적, m²)",
            fraction=.05
        )

        plt.show()

        vals=dem[mask]
        print(
            f"유역 면적: {actual_area_km2:.3f} km² "
            f"({actual_area_km2*100:.1f} ha) | "
            f"유효 격자: {int(mask.sum()):,}개 | "
            f"경계상자: {width/1000:.2f} × {width/1000:.2f} km | "
            f"유출구: row {outlet_r}, col {outlet_c} | "
            f"고도 범위: {np.nanmin(vals):.1f}–{np.nanmax(vals):.1f} m"
        )

for _w in (
    basin_area_widget,basin_cellsize_widget,basin_base_elev_widget,
    basin_long_relief_widget,basin_soil_depth_widget,
    basin_seed_widget,basin_elongation_widget,
    basin_irregularity_widget,basin_noise_widget
):
    _w.observe(_update_basin_preview,names="value")

display(widgets.HBox([
    widgets.VBox([
        basin_area_widget,basin_cellsize_widget,basin_base_elev_widget,
        basin_long_relief_widget,basin_soil_depth_widget
    ]),
    widgets.VBox([
        basin_seed_widget,basin_elongation_widget,
        basin_irregularity_widget,basin_noise_widget,
        basin_grid_info
    ])
]))
display(basin_help)
display(basin_preview)
_update_basin_preview()


## 4절 시나리오 설정

| 항목 | 내용 |
|---|---|
| 계산기간 | 120–0 ka BP, 1 kyr 간격 |
| 공간자료 | 동일한 가상유역 |
| 실행모드 | 시나리오별 정적 또는 동적 |
| 지형과정 | 융기율, 토양생산, 사면수송, 유수침식 |


In [ ]:
#@title 4.1 시나리오 설정
import ipywidgets as widgets
from IPython.display import display

scenario_count_widget=widgets.Dropdown(
    options=[2,3,4],value=2,description="시나리오 수",
    style={"description_width":"115px"},layout=widgets.Layout(width="280px"))
start_ka_widget=widgets.BoundedIntText(
    value=120,min=1,max=120,description="시작 (ka BP)",
    style={"description_width":"110px"},layout=widgets.Layout(width="225px"))
end_ka_widget=widgets.BoundedIntText(
    value=0,min=0,max=119,description="종료 (ka BP)",
    style={"description_width":"110px"},layout=widgets.Layout(width="225px"))

def _slider(desc,value,minv,maxv,step):
    return widgets.FloatSlider(
        description=desc,value=value,min=minv,max=maxv,step=step,
        continuous_update=False,style={"description_width":"175px"},
        layout=widgets.Layout(width="525px"))

def _reps_for_zone(zone):
    rows=climate_places.loc[climate_places.koppen_reference_0ka==zone]
    return [(x.place_label,x.representative_id)
            for x in rows.itertuples(index=False)]

scenario_controls=[]
for i in range(4):
    controls={
        "name":widgets.Text(value="baseline" if i==0 else f"scenario{i+1}",
              description="시나리오 이름",style={"description_width":"175px"},
              layout=widgets.Layout(width="525px")),
        "model":widgets.Dropdown(options=["정적","동적"],value="동적",
              description="모델",style={"description_width":"175px"},
              layout=widgets.Layout(width="525px")),
        "koppen":widgets.Dropdown(
              options=[(
                  f'{z} — {_KOPPEN_KO.get(z,z)}'
                  f' [{_KOPPEN_SUBTYPE.get(z,"")}] — '
                  f'{climate_places.loc[climate_places.koppen_reference_0ka==z,"koppen_name"].iloc[0]}',z
              ) for z in climate_types],
              value="Dwa" if "Dwa" in climate_types else climate_types[0],
              description="쾨펜 기후형",style={"description_width":"175px"},
              layout=widgets.Layout(width="525px")),
        "representative":widgets.Dropdown(
              description="도시·지역",style={"description_width":"175px"},
              layout=widgets.Layout(width="650px")),
        "initial_soil_depth_multiplier":_slider("초기 토심 배율",1.0,0.2,2.0,0.1),
        "uplift_m_per_kyr":_slider("융기율 (m/kyr)",0.08,0,0.16,0.01),
        "soil_production_a":_slider("토양생산 계수 a",0.037,0.01,0.08,0.001),
        "hillslope_eemt_c":_slider("사면수송 계수 c",0.033,0.005,0.08,0.001),
        "hillslope_biomass_d":_slider("사면수송 계수 d",0.050,0.005,0.12,0.005),
        "fluvial_erodibility_k0":_slider("유수침식 계수 K0",0.020,0.005,0.05,0.001),
    }
    def _sync(change,control=controls):
        previous=control["representative"].value
        options=_reps_for_zone(control["koppen"].value)
        control["representative"].options=options
        valid_ids=[value for _,value in options]
        control["representative"].value=(
            previous if previous in valid_ids else (valid_ids[0] if valid_ids else None)
        )
    controls["koppen"].observe(_sync,names="value")
    _sync(None)
    controls["_panel"]=widgets.VBox(
        [widgets.HTML(value=f"<h3>시나리오 {i+1}</h3>")]+
        [c for k,c in controls.items() if not k.startswith("_")],
        layout=widgets.Layout(width="100%",border="1px solid #dddddd",padding="12px",margin="8px 0"))
    scenario_controls.append(controls)

scenario_box=widgets.VBox([w["_panel"] for w in scenario_controls])
def _display_scenarios(change=None):
    for i,w in enumerate(scenario_controls):
        w["_panel"].layout.display="flex" if i<scenario_count_widget.value else "none"
scenario_count_widget.observe(_display_scenarios,names="value")
_display_scenarios()
display(widgets.HBox([scenario_count_widget,start_ka_widget,end_ka_widget]))
display(scenario_box)


## 5절 시나리오 입력자료 생성


In [ ]:
#@title 5.1 시나리오 입력자료 생성
import re, shutil, json, numpy as np, pandas as pd, rasterio
from rasterio.transform import from_origin
from pyproj import Transformer
from pathlib import Path

scenario_count=int(scenario_count_widget.value)
start_ka=int(start_ka_widget.value)
end_ka=int(end_ka_widget.value)
if not (0<=end_ka<start_ka<=120):
    raise ValueError("시작시점과 종료시점을 120–0 ka BP 범위에서 지정하십시오.")
period=f"{start_ka}–{end_ka} ka BP"

scenario_configs=[]
for i in range(scenario_count):
    w=scenario_controls[i]
    name=w["name"].value.strip()
    if not re.fullmatch(r"[A-Za-z0-9_-]{1,40}",name):
        raise ValueError(f"시나리오 {i+1} 이름에는 영문·숫자·_·-만 사용할 수 있습니다.")
    scenario_configs.append({
        "name":name,"model":w["model"].value,
        "koppen":w["koppen"].value,
        "representative_id":w["representative"].value,
        "soil_depth_multiplier":float(w["initial_soil_depth_multiplier"].value),
        "uplift_m_per_kyr":float(w["uplift_m_per_kyr"].value),
        "soil_production_a":float(w["soil_production_a"].value),
        "hillslope_c":float(w["hillslope_eemt_c"].value),
        "hillslope_d":float(w["hillslope_biomass_d"].value),
        "fluvial_K0":float(w["fluvial_erodibility_k0"].value)
    })
if len(set(x["name"] for x in scenario_configs))!=len(scenario_configs):
    raise ValueError("시나리오 이름은 중복될 수 없습니다.")

exp_root=ROOT/"teaching_experiments_global120"
exp_root.mkdir(parents=True,exist_ok=True)

# 공통 가상유역
cell_size_m=float(basin_cellsize_widget.value)
target_basin_area_km2=float(basin_area_widget.value)
base_elevation_m=float(basin_base_elev_widget.value)
longitudinal_relief_m=float(basin_long_relief_widget.value)
base_soil_depth_m=float(basin_soil_depth_widget.value)
basin_seed=int(basin_seed_widget.value)
basin_elongation=float(basin_elongation_widget.value)
basin_irregularity=float(basin_irregularity_widget.value)
basin_noise_m=float(basin_noise_widget.value)

(
    dem_common,basin_mask,(outlet_row,outlet_col),
    flow_accumulation_common,flow_receiver_common,dem_filled_common
)=_build_virtual_basin(return_diagnostics=True)
grid_size=int(dem_common.shape[0])
actual_basin_area_m2=float(basin_mask.sum()*cell_size_m*cell_size_m)
actual_basin_area_km2=actual_basin_area_m2/1_000_000.0

soil_common=np.where(
    basin_mask,base_soil_depth_m,np.nan
).astype("float32")
texture_common=np.where(
    basin_mask,2.0,np.nan
).astype("float32")

virtual_basin_config={
    "target_basin_area_km2":target_basin_area_km2,
    "actual_basin_area_km2":actual_basin_area_km2,
    "grid_size":grid_size,
    "cell_size_m":cell_size_m,
    "bounding_box_width_m":grid_size*cell_size_m,
    "valid_cell_count":int(basin_mask.sum()),
    "basin_area_m2":actual_basin_area_m2,
    "outlet_row":int(outlet_row),
    "outlet_col":int(outlet_col),
    "base_elevation_m":base_elevation_m,
    "relief_control_m":longitudinal_relief_m,
    "base_soil_depth_m":base_soil_depth_m,
    "shape_seed":basin_seed,
    "elongation":basin_elongation,
    "boundary_irregularity":basin_irregularity,
    "elevation_noise_m":basin_noise_m,
    "flow_routing_method":"D8 steepest descent",
    "depression_handling":"Priority-Flood with epsilon drainage",
    "outlet_selection":"target-area nearest contributing-area cell",
    "outlet_contributing_cells":int(basin_mask.sum()),
    "area_error_percent":100.0*(actual_basin_area_km2-target_basin_area_km2)/target_basin_area_km2,
    "boundary_condition":"selected outlet open; all remaining delineated watershed boundary closed",
}
virtual_basin_table=pd.DataFrame([virtual_basin_config])
print("가상유역 설정")
display(virtual_basin_table)

def make_climate_csv(representative,output_path):
    raw=CLIMATE_MASTER[CLIMATE_MASTER.representative_id==representative].copy()
    if raw.empty: raise ValueError(f"기후 지점이 없습니다: {representative}")
    newages=np.arange(0,121,dtype=int)
    data=[]
    for month in range(1,13):
        source=raw.loc[raw.month==month].sort_values("ka_bp")
        age=source["ka_bp"].to_numpy(float)
        if len(age)<2 or age[0]!=0 or age[-1]!=120:
            raise ValueError(f"{representative} {month}월: 기후 시점 범위 오류")
        for k in newages:
            rec={"ka_bp":int(k),"month":month}
            for key in ("temp_C","precip_mm","cloud_pct","relative_humidity","wind_speed"):
                if key in source:
                    rec[key]=float(np.interp(k,age,source[key].to_numpy(float)))
            data.append(rec)
    df=pd.DataFrame(data)
    # BIOME4 원본 climdata 회귀식으로 연중 최저 월평균기온에서 Tmin을 추정한다.
    coldest_month_mean=df.groupby("ka_bp")["temp_C"].transform("min").to_numpy(float)
    df["tmin_C"]=(
        0.006*coldest_month_mean**2
        + 1.316*coldest_month_mean
        - 21.9
    )
    df["precip_mm"]=df["precip_mm"].clip(lower=0)
    df["cloud_pct"]=df["cloud_pct"].clip(lower=0,upper=100)
    assert len(df)==121*12 and not df[["temp_C","precip_mm","cloud_pct","tmin_C"]].isna().any().any()
    assert df.groupby("ka_bp")["tmin_C"].nunique().eq(1).all()
    assert np.isfinite(df["tmin_C"].to_numpy(float)).all()
    df.sort_values(["ka_bp","month"],ascending=[False,True]).to_csv(
        output_path,index=False,float_format="%.6f")
    return df

def generate_virtual_basin(folder,lon,lat,soil_factor):
    from rasterio.features import shapes
    folder.mkdir(parents=True,exist_ok=True)

    # 가상유역의 지리좌표는 선택한 기후 지점에 두되,
    # 지형·유역형상은 모든 시나리오에서 동일하게 사용
    if lat>=84: epsg=3413
    elif lat<=-80: epsg=3031
    else:
        zone=max(1,min(60,int((lon+180)//6+1)))
        epsg=(32600 if lat>=0 else 32700)+zone

    xx,yy=Transformer.from_crs(4326,epsg,always_xy=True).transform(lon,lat)
    width=grid_size*cell_size_m
    left=xx-width/2
    top=yy+width/2
    transform=from_origin(left,top,cell_size_m,cell_size_m)

    dem_write=np.where(basin_mask,dem_common,-9999.0).astype("float32")
    soil_write=np.where(
        basin_mask,base_soil_depth_m*soil_factor,-9999.0
    ).astype("float32")
    texture_write=np.where(basin_mask,2.0,-9999.0).astype("float32")

    for filename,data in (
        ("dem_virtual.tif",dem_write),
        ("soil_depth_virtual_m.tif",soil_write),
        ("soil_texture_virtual.tif",texture_write)
    ):
        with rasterio.open(
            folder/filename,"w",driver="GTiff",
            width=grid_size,height=grid_size,count=1,
            dtype="float32",crs=f"EPSG:{epsg}",
            transform=transform,nodata=-9999.0
        ) as dst:
            dst.write(data,1)

    # 마스크의 실제 불규칙 경계를 basin polygon으로 저장
    mask_u8=basin_mask.astype("uint8")
    geoms=[
        geom for geom,val in shapes(mask_u8,mask=basin_mask,connectivity=8,transform=transform)
        if int(val)==1
    ]
    if len(geoms)!=1:
        raise RuntimeError(f"가상유역 마스크가 단일 연결 유역이 아닙니다: {len(geoms)}개")
    basin_geometry=geoms[0]

    # D8 delineation으로 정한 outlet만 open, 나머지 최종 유역 경계는 closed
    outlet_x=left+(outlet_col+0.5)*cell_size_m
    outlet_y=top-(outlet_row+0.5)*cell_size_m
    outlet_geometry={"type":"Point","coordinates":[outlet_x,outlet_y]}

    for fname,geometry in (
        ("basin_virtual.geojson",basin_geometry),
        ("outlet_open_free_virtual.geojson",outlet_geometry)
    ):
        feature={
            "type":"FeatureCollection",
            "crs":{"type":"name","properties":{"name":f"EPSG:{epsg}"}},
            "features":[{"type":"Feature","properties":{}, "geometry":geometry}]
        }
        (folder/fname).write_text(
            json.dumps(feature,ensure_ascii=False),encoding="utf-8"
        )

scenario_paths={}
for sc in scenario_configs:
    name=sc["name"]
    row=climate_places.loc[climate_places.representative_id==sc["representative_id"]].iloc[0]
    work=exp_root/name
    inputs=work/"inputs"
    if inputs.exists():shutil.rmtree(inputs)
    inputs.mkdir(parents=True)
    generate_virtual_basin(
        inputs,float(row.source_grid_lon),float(row.source_grid_lat),
        sc["soil_depth_multiplier"]
    )
    clim_path=inputs/"beyer_global_120ka_monthly.csv"
    make_climate_csv(sc["representative_id"],clim_path)
    scenario_paths[name]={"work":work,"inputs":inputs,"rawcsv":clim_path,
                          "soil_path":inputs/"soil_depth_virtual_m.tif"}

scenario_table=pd.DataFrame(scenario_configs)
scenario_table["기후 지역"]=scenario_table["representative_id"].map(
    climate_places.set_index("representative_id")["region_name"]
)
scenario_table["기후 지점"]=scenario_table["representative_id"].map(
    climate_places.set_index("representative_id")["place_label"]
)
display(scenario_table)
print(
    f"기간 {period} | 시간간격 1 kyr | "
    f"가상유역 {actual_basin_area_km2:.3f} km² | "
    f"{grid_size}×{grid_size} 격자 | "
    f"해상도 {cell_size_m:g} m | "
    f"경계상자 {grid_size*cell_size_m/1000:.2f} km"
)

import matplotlib.pyplot as plt
_configure_korean_plotting()
from IPython.display import display

annual_rows=[]
for sc in scenario_configs:
    cp=scenario_paths[sc["name"]]["rawcsv"]
    forcing=pd.read_csv(cp)
    yearly=forcing.groupby("ka_bp",as_index=False).agg(
        mean_temp_C=("temp_C","mean"),
        annual_precip_mm=("precip_mm","sum"),
        mean_cloud_pct=("cloud_pct","mean")
    )
    yearly.insert(0,"scenario",sc["name"])
    yearly.insert(1,"model",sc["model"])
    yearly.insert(2,"koppen",sc["koppen"])
    yearly.insert(3,"representative_id",sc["representative_id"])
    annual_rows.append(yearly)
scenario_climate_annual=pd.concat(annual_rows,ignore_index=True)
scenario_climate_annual.to_csv(
    exp_root/"climate_forcing_annual.csv",index=False,encoding="utf-8-sig"
)

fig,axes=plt.subplots(1,3,figsize=(16,4.3),constrained_layout=True)
_x=np.arange(grid_size,dtype=float)*cell_size_m
_y=np.arange(grid_size,dtype=float)*cell_size_m
_extent=[
    -0.5*cell_size_m,(grid_size-0.5)*cell_size_m,
    (grid_size-0.5)*cell_size_m,-0.5*cell_size_m
]
im=axes[0].imshow(
    np.ma.masked_invalid(dem_common),
    cmap="terrain",origin="upper",extent=_extent
)
axes[0].contour(
    _x,_y,basin_mask.astype(float),
    levels=[0.5],colors="black",linewidths=1.0
)
_channel_threshold=max(5,int(round(0.05*basin_mask.sum())))
_channel=basin_mask & (flow_accumulation_common>=_channel_threshold)
_cr,_cc=np.where(_channel)
if len(_cr):
    axes[0].scatter(
        _cc*cell_size_m,_cr*cell_size_m,
        s=7,marker="s",linewidths=0,alpha=0.75,label="D8 하도"
    )
axes[0].plot(
    outlet_col*cell_size_m,outlet_row*cell_size_m,
    marker="v",markersize=9,label="open 유출구"
)
axes[0].legend(frameon=False,fontsize=8,loc="lower left")
axes[0].set_aspect("equal")
axes[0].set_xlabel("거리 (m)")
axes[0].set_ylabel("거리 (m)")
axes[0].set_title(
    f"D8 유역분석 DEM 및 단일 open 유출구\n"
    f"{actual_basin_area_km2:.3f} km²"
)
fig.colorbar(im,ax=axes[0],fraction=.05,label="고도 (m)")
for sc in scenario_configs:
    data=scenario_climate_annual[scenario_climate_annual.scenario==sc["name"]]
    axes[1].plot(data.ka_bp,data.mean_temp_C,label=f'{sc["name"]} ({sc["koppen"]})')
    axes[2].plot(data.ka_bp,data.annual_precip_mm,label=f'{sc["name"]} ({sc["koppen"]})')
axes[1].set_ylabel("연평균 기온 (°C)")
axes[2].set_ylabel("연강수량 (mm)")
for ax in axes[1:]:
    ax.set_xlabel("시기 (ka BP)")
    ax.set_xlim(start_ka,end_ka)
    ax.grid(alpha=.2)
axes[1].legend(frameon=False,fontsize=8)
plt.show()



## 6절 시나리오 일괄 실행


In [ ]:
#@title 6.1 시나리오 실행
import os, time, warnings
import numpy as np
import pandas as pd
from pb4studio.pelletier_geomorph import PelletierStrictConfig
from pb4studio.snapshots import snapshot_dir_name

pb4_threads=int(globals().get("PB4_THREADS",max(1,min(int(os.cpu_count() or 1),2))))
times=np.round(np.arange(start_ka,end_ka-0.01,-1.0),6)
assert len(times)>1 and abs(float(times[-1])-end_ka)<1e-5
# 5 kyr 간격으로 자동 저장. 시작·종료시점은 항상 포함
_snap_list=[float(start_ka)]
_next=float(start_ka)-5.0
while _next>float(end_ka)+1e-9:
    _snap_list.append(float(_next))
    _next-=5.0
if abs(_snap_list[-1]-float(end_ka))>1e-9:
    _snap_list.append(float(end_ka))
snaps=tuple(round(x,6) for x in _snap_list)
print("지도·공간분포 저장시점 (ka BP):",", ".join(f"{x:g}" for x in snaps))

def _fmt_seconds(seconds):
    seconds=max(0,int(round(seconds)))
    h,rem=divmod(seconds,3600)
    m,s=divmod(rem,60)
    return f"{h:d}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"

scenario_outputs={}
scenario_runtime=[]
overall_start=time.perf_counter()

for scenario_no,sc in enumerate(scenario_configs,1):
    name=sc["name"]
    paths=scenario_paths[name]
    work=paths["work"]
    inputs=paths["inputs"]
    rawcsv=paths["rawcsv"]
    soil_path=paths["soil_path"]

    out_dir=work/"output"
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True,exist_ok=True)

    cfg=RunConfig(
        mode=RunMode.STATIC if sc["model"]=="정적" else RunMode.DYNAMIC,
        inputs=InputPaths(
            dem_path=str(inputs/"dem_virtual.tif"),
            climate_nc_path=str(rawcsv),
            soil_depth_path=str(soil_path),
            soil_texture_path=str(inputs/"soil_texture_virtual.tif"),
            basin_polygon_path=str(inputs/"basin_virtual.geojson"),
            open_free_shp=str(inputs/"outlet_open_free_virtual.geojson"),
            open_fixed_slope_shp=None,
            soil_depth_unit="m",
            validation_points_csv="",
        ),
        time=TimeConfig(
            start_ka=float(start_ka),
            end_ka=float(end_ka),
            interval_kyr=1.0,
            snapshot_ka=snaps,
        ),
        resolution_m_override=float(cell_size_m),
        requested_core_count=pb4_threads,
        pelletier=PelletierStrictConfig(
            uplift_m_per_kyr=sc["uplift_m_per_kyr"],
            p0_base_m_per_kyr=sc["soil_production_a"],
            kd_c_eemt_m_per_kyr_per_mj=sc["hillslope_c"],
            kd_d_agb_m_per_kyr_per_kgm2=sc["hillslope_d"],
            k0_regolith=sc["fluvial_K0"],
            hillslope_critical_slope=1.50,
        ),
        science=ScienceConfig(
            biome4_variant="mckenzie2003",
            force_compile_biome4=False,
            validation_accuracy_method="per_output_time",
            basin_presence_min_fraction=0.01,
        ),
        output_dir=str(out_dir),
    )

    cfg.to_json(work/"run_config.json")
    settings={
        "scenario_number":scenario_no,
        "scenario_count":scenario_count,
        "scenario_name":name,
        "model":sc["model"],
        "koppen":sc["koppen"],
        "representative_id":sc["representative_id"],
        "soil_depth_multiplier":sc["soil_depth_multiplier"],
        "start_ka":start_ka,
        "end_ka":end_ka,
        "u_m_per_kyr":sc["uplift_m_per_kyr"],
        "p0_a":sc["soil_production_a"],
        "kd_c":sc["hillslope_c"],
        "kd_d":sc["hillslope_d"],
        "K0":sc["fluvial_K0"],
        "model_sha256":checksum,
        "eemt_cold_phase_method":EEMT_COLD_PHASE_METHOD,
        "biome4_openmp_threads":pb4_threads,
    }
    (work/"teaching_settings.json").write_text(
        json.dumps(settings,ensure_ascii=False,indent=2),encoding="utf-8"
    )

    state={"case":None,"case_start":None}
    def progress(case_label,step_index,n_steps,ka,_name=name,_state=state):
        now=time.perf_counter()
        if _state["case"]!=case_label:
            _state["case"]=case_label
            _state["case_start"]=now
            print(f"\n[{_name} / {case_label}] 계산 시작 | {n_steps}시점",flush=True)
        elapsed=now-_state["case_start"]
        completed=max(int(step_index),0)
        if step_index==0 or step_index==n_steps-1 or step_index<10 or step_index%10==0:
            if completed>0:
                sec_per_step=elapsed/completed
                eta=sec_per_step*max(n_steps-completed,0)
                print(
                    f"[{_name} / {case_label}] {step_index+1}/{n_steps}, {ka:.1f} ka BP"
                    f" | 경과 {_fmt_seconds(elapsed)} | 남은 시간 약 {_fmt_seconds(eta)}",
                    flush=True
                )
            else:
                print(f"[{_name} / {case_label}] 1/{n_steps}, {ka:.1f} ka BP",flush=True)

    _place_name=climate_places.loc[
        climate_places.representative_id==sc["representative_id"],"landmark_name_ko"
    ].iloc[0]
    print(
        f"\n===== 시나리오 {scenario_no}/{scenario_count}: {name} ====="
        f"\n쾨펜 기후형 {sc['koppen']} | 도시·지역 {_place_name} ({sc['representative_id']}) | 모델 {sc['model']}",
        flush=True
    )
    start=time.perf_counter()
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",message="Mean of empty slice",
                category=RuntimeWarning,module=r"pb4studio\.runner"
            )
            runner.run(cfg,progress_callback=progress)
    except KeyboardInterrupt:
        print(f"\n{name} 실행이 중단되었습니다. 계산 예외가 아니라 Jupyter 중단 신호입니다.",flush=True)
        raise

    elapsed=time.perf_counter()-start

    for label,subdir in [(sc["model"],"model_static" if sc["model"]=="정적" else "model_dynamic")]:
        summary_file=out_dir/subdir/"summary_timeseries.csv"
        assert summary_file.is_file(),f"{name}/{label}: summary_timeseries.csv 없음"
        summary=pd.read_csv(summary_file)
        assert len(summary)==len(times),f"{name}/{label}: 시점 수 불일치"
        assert summary["ka_bp"].is_unique,f"{name}/{label}: 중복 시점"
        assert np.allclose(summary["ka_bp"].to_numpy(float),times,atol=1e-6,rtol=0)
        for col in ("mean_elevation_m","mean_soil_depth_m","mean_npp"):
            assert np.isfinite(summary[col].to_numpy(float)).all(),f"{name}/{label}: {col} 비정상 값"

    # BIOME4 native biome 출력 누락은 지도 단계 전에 확인
    for age in snaps:
        biome_file=out_dir/("model_static" if sc["model"]=="정적" else "model_dynamic")/snapshot_dir_name(float(age))/"biome4_full_id.tif"
        if not biome_file.is_file():
            raise FileNotFoundError(f"BIOME4 생물군계 GeoTIFF 저장 실패: {biome_file}")
    if "biome4_full_id_counts" not in summary.columns:
        raise RuntimeError(f"{name}: BIOME4 생물군계 시계열 집계 누락")

    scenario_outputs[name]=out_dir
    scenario_runtime.append({"scenario":name,"model":sc["model"],"koppen":sc["koppen"],"representative_id":sc["representative_id"],"elapsed_s":elapsed})
    print(f"{name} 완료 | {_fmt_seconds(elapsed)}",flush=True)

reference_scenario=scenario_configs[0]["name"]
work=scenario_paths[reference_scenario]["work"]
inputs=scenario_paths[reference_scenario]["inputs"]
rawcsv=scenario_paths[reference_scenario]["rawcsv"]
soil_path=scenario_paths[reference_scenario]["soil_path"]
out_dir=scenario_outputs[reference_scenario]
experiment_name=reference_scenario
CASE_DIRS={"정적":"model_static","동적":"model_dynamic"}
SCENARIO_MODELS={sc["name"]:sc["model"] for sc in scenario_configs}
selected_ages=tuple(float(x) for x in snaps)

registry={
    "period":period,
    "start_ka":start_ka,
    "end_ka":end_ka,
    "reference_scenario":reference_scenario,
    "virtual_basin":virtual_basin_config,
    "scenarios":scenario_configs,
    "interval_kyr":1.0,
}
registry_path=exp_root/"scenario_registry.json"
registry_path.write_text(json.dumps(registry,ensure_ascii=False,indent=2),encoding="utf-8")

print("\n전체 시나리오 실행 완료")
display(pd.DataFrame(scenario_runtime).assign(elapsed_min=lambda x:x.elapsed_s/60).round(2))
print("기준 시나리오:",reference_scenario)


## 7절 BIOME4 식생모델

### 7.1 식물기능유형(PFT)

식물기능유형(Plant Functional Type, PFT)은 생장형, 잎의 형태와 계절성, 생리적 특성 및 기후 요구조건이 유사한 식물을 묶은 기능적 분류 단위이다(Bigelow et al., 2003).

BIOME4는 13개 PFT의 생물기후적 생육 가능성과 순일차생산량(NPP), 엽면적지수(LAI), 수분수지를 계산하고, PFT 간 경쟁 결과를 바탕으로 생물군계를 판정한다(Kaplan et al., 2003).



#### 7.1.1 BIOME4 식물기능유형 13개

<div style="width:980px;max-width:100%;overflow-x:auto;margin:18px auto;box-sizing:border-box;">
<table width="980" style="width:980px;min-width:980px;max-width:980px;margin:0;box-sizing:border-box;table-layout:fixed;border-collapse:collapse;font-size:0.90em;line-height:1.4;overflow-wrap:anywhere;">
<colgroup><col style="width:5%;"><col style="width:27%;"><col style="width:23%;"><col style="width:45%;"></colgroup>
<tr style="background:#eeeeee;">
<th style="border:1px solid #333;padding:6px;text-align:center;;overflow-wrap:anywhere;word-break:normal;">PFT</th><th style="border:1px solid #333;padding:6px;text-align:center;;overflow-wrap:anywhere;word-break:normal;">BIOME4 원명</th><th style="border:1px solid #333;padding:6px;text-align:center;;overflow-wrap:anywhere;word-break:normal;">국문명</th><th style="border:1px solid #333;padding:6px;text-align:center;;overflow-wrap:anywhere;word-break:normal;">대표 식생·분류군</th>
</tr>
<tr><td style="border:1px solid #999;padding:5px;text-align:center;;overflow-wrap:anywhere;word-break:normal;">1</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">Tropical Evergreen</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">열대 상록교목</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">망고나무 (Mangifera indica)</td></tr>
<tr><td style="border:1px solid #999;padding:5px;text-align:center;;overflow-wrap:anywhere;word-break:normal;">2</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">Tropical Raingreen</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">열대 건기낙엽교목</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">티크 (Tectona grandis)</td></tr>
<tr><td style="border:1px solid #999;padding:5px;text-align:center;;overflow-wrap:anywhere;word-break:normal;">3</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">Temperate Broadleaved Evergreen</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">온대 상록활엽교목</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">구실잣밤나무 (Castanopsis sieboldii)</td></tr>
<tr><td style="border:1px solid #999;padding:5px;text-align:center;;overflow-wrap:anywhere;word-break:normal;">4</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">Temperate Summergreen</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">온대 낙엽활엽교목</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">신갈나무 (Quercus mongolica)</td></tr>
<tr><td style="border:1px solid #999;padding:5px;text-align:center;;overflow-wrap:anywhere;word-break:normal;">5</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">Temperate Evergreen Conifer</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">온대 상록침엽교목</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">소나무 (Pinus densiflora)</td></tr>
<tr><td style="border:1px solid #999;padding:5px;text-align:center;;overflow-wrap:anywhere;word-break:normal;">6</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">Boreal Evergreen</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">아한대 상록침엽교목</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">가문비나무 (Picea jezoensis)</td></tr>
<tr><td style="border:1px solid #999;padding:5px;text-align:center;;overflow-wrap:anywhere;word-break:normal;">7</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">Boreal Deciduous</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">아한대 낙엽교목</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">자작나무 (Betula pendula)</td></tr>
<tr><td style="border:1px solid #999;padding:5px;text-align:center;;overflow-wrap:anywhere;word-break:normal;">8</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">Temperate Grass</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">C3 온대 화본형 초본</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">큰김의털 (Festuca arundinacea)</td></tr>
<tr><td style="border:1px solid #999;padding:5px;text-align:center;;overflow-wrap:anywhere;word-break:normal;">9</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">Tropical/Warm-Temperate Grass</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">C4 열대·난온대 화본형 초본</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">솔새 (Themeda triandra var. japonica)</td></tr>
<tr><td style="border:1px solid #999;padding:5px;text-align:center;;overflow-wrap:anywhere;word-break:normal;">10</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">Desert Woody Plant Type C3 and C4</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">C3/C4 건조지 목본</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">목적마황 (Ephedra equisetina)</td></tr>
<tr><td style="border:1px solid #999;padding:5px;text-align:center;;overflow-wrap:anywhere;word-break:normal;">11</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">Tundra Shrub Type</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">툰드라 관목</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">눈잣나무 (Pinus pumila)</td></tr>
<tr><td style="border:1px solid #999;padding:5px;text-align:center;;overflow-wrap:anywhere;word-break:normal;">12</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">Cold Herbaceous Type</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">한랭 초본</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">좀바늘사초 (Kobresia myosuroides)</td></tr>
<tr><td style="border:1px solid #999;padding:5px;text-align:center;;overflow-wrap:anywhere;word-break:normal;">13</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">Lichen/Forb Type</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">지의류·비화본과 초본</td><td style="border:1px solid #999;padding:5px;overflow-wrap:anywhere;">에델바이스 (Leontopodium alpinum)</td></tr>
</table>
</div>
<div style="font-size:0.82em;margin:4px 0 14px 0;">출처: BIOME4 v4.2b2; Ni (2000); Kaplan et al. (2002, 2003); Bradshaw et al. (2025).</div>


#### 7.1.2 PFT별 대표 식생

<div style="width:980px;max-width:100%;overflow-x:auto;margin:18px auto;box-sizing:border-box;">
<table width="980" style="width:980px;min-width:980px;max-width:980px;table-layout:fixed;border-collapse:separate;border-spacing:8px;margin:0;box-sizing:border-box;">
<colgroup><col style="width:25%;"><col style="width:25%;"><col style="width:25%;"><col style="width:25%;"></colgroup>
<tr style="height:330px;"><td style="width:25%;vertical-align:top;padding:0;overflow-wrap:anywhere;word-break:normal;;height:330px;box-sizing:border-box;"><div style="display:block;width:100%;height:330px;min-height:330px;max-height:330px;border:1px solid #aaa;background:#fff;box-sizing:border-box;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Mango_tree_(Mangifera_indica).jpg" target="_blank" rel="noopener" aria-label="망고나무 전체 수관과 줄기" style="display:block;width:100%;height:180px;overflow:hidden;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Mango_tree_%28Mangifera_indica%29.jpg?width=560" alt="망고나무 전체 수관과 줄기" width="230" height="180" style="display:block;width:100%;height:180px;object-fit:cover;object-position:center;"></a><div style="display:block;width:100%;height:150px;min-height:150px;max-height:150px;overflow:hidden;box-sizing:border-box;">

<div style="background:#eeeeee;padding:7px 6px;font-weight:600;font-size:0.82em;line-height:1.35;text-align:left;">PFT 1. 열대 상록활엽교목</div>
<div style="padding:8px 6px 3px 6px;font-weight:700;font-size:1em;line-height:1.3;text-align:left;">망고나무</div>
<div style="padding:0 6px 8px 6px;font-size:0.78em;line-height:1.35;text-align:left;overflow-wrap:anywhere;"><i>Mangifera indica</i></div>
<div style="font-size:0.70em;border-top:1px solid #ddd;padding:7px 6px 6px 6px;line-height:1.5;text-align:left;"><a href="https://commons.wikimedia.org/wiki/File:Mango_tree_(Mangifera_indica).jpg" target="_blank" rel="noopener">출처: Wikimedia Commons</a></div>
</div></div></td>
<td style="width:25%;vertical-align:top;padding:0;overflow-wrap:anywhere;word-break:normal;;height:330px;box-sizing:border-box;"><div style="display:block;width:100%;height:330px;min-height:330px;max-height:330px;border:1px solid #aaa;background:#fff;box-sizing:border-box;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Teak_Tree.JPG" target="_blank" rel="noopener" aria-label="티크나무 줄기와 가지" style="display:block;width:100%;height:180px;overflow:hidden;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Teak_Tree.JPG?width=560" alt="티크나무 줄기와 가지" width="230" height="180" style="display:block;width:100%;height:180px;object-fit:cover;object-position:center;"></a><div style="display:block;width:100%;height:150px;min-height:150px;max-height:150px;overflow:hidden;box-sizing:border-box;">

<div style="background:#eeeeee;padding:7px 6px;font-weight:600;font-size:0.82em;line-height:1.35;text-align:left;">PFT 2. 열대 건기낙엽교목</div>
<div style="padding:8px 6px 3px 6px;font-weight:700;font-size:1em;line-height:1.3;text-align:left;">티크</div>
<div style="padding:0 6px 8px 6px;font-size:0.78em;line-height:1.35;text-align:left;overflow-wrap:anywhere;"><i>Tectona grandis</i></div>
<div style="font-size:0.70em;border-top:1px solid #ddd;padding:7px 6px 6px 6px;line-height:1.5;text-align:left;"><a href="https://commons.wikimedia.org/wiki/File:Teak_Tree.JPG" target="_blank" rel="noopener">출처: Wikimedia Commons</a></div>
</div></div></td>
<td style="width:25%;vertical-align:top;padding:0;overflow-wrap:anywhere;word-break:normal;;height:330px;box-sizing:border-box;"><div style="display:block;width:100%;height:330px;min-height:330px;max-height:330px;border:1px solid #aaa;background:#fff;box-sizing:border-box;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Castanopsis_sieboldii1.jpg" target="_blank" rel="noopener" aria-label="구실잣밤나무의 잎과 가지" style="display:block;width:100%;height:180px;overflow:hidden;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Castanopsis_sieboldii1.jpg?width=560" alt="구실잣밤나무의 잎과 가지" width="230" height="180" style="display:block;width:100%;height:180px;object-fit:cover;object-position:center;"></a><div style="display:block;width:100%;height:150px;min-height:150px;max-height:150px;overflow:hidden;box-sizing:border-box;">

<div style="background:#eeeeee;padding:7px 6px;font-weight:600;font-size:0.82em;line-height:1.35;text-align:left;">PFT 3. 온대 상록활엽교목</div>
<div style="padding:8px 6px 3px 6px;font-weight:700;font-size:1em;line-height:1.3;text-align:left;">구실잣밤나무</div>
<div style="padding:0 6px 8px 6px;font-size:0.78em;line-height:1.35;text-align:left;overflow-wrap:anywhere;"><i>Castanopsis sieboldii</i></div>
<div style="font-size:0.70em;border-top:1px solid #ddd;padding:7px 6px 6px 6px;line-height:1.5;text-align:left;"><a href="https://commons.wikimedia.org/wiki/File:Castanopsis_sieboldii1.jpg" target="_blank" rel="noopener">출처: Wikimedia Commons</a></div>
</div></div></td>
<td style="width:25%;vertical-align:top;padding:0;overflow-wrap:anywhere;word-break:normal;;height:330px;box-sizing:border-box;"><div style="display:block;width:100%;height:330px;min-height:330px;max-height:330px;border:1px solid #aaa;background:#fff;box-sizing:border-box;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Mongolian_Oak_in_October.jpg" target="_blank" rel="noopener" aria-label="가을철 신갈나무 수관" style="display:block;width:100%;height:180px;overflow:hidden;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Mongolian_Oak_in_October.jpg?width=560" alt="가을철 신갈나무 수관" width="230" height="180" style="display:block;width:100%;height:180px;object-fit:cover;object-position:center;"></a><div style="display:block;width:100%;height:150px;min-height:150px;max-height:150px;overflow:hidden;box-sizing:border-box;">

<div style="background:#eeeeee;padding:7px 6px;font-weight:600;font-size:0.82em;line-height:1.35;text-align:left;">PFT 4. 온대 낙엽활엽교목</div>
<div style="padding:8px 6px 3px 6px;font-weight:700;font-size:1em;line-height:1.3;text-align:left;">신갈나무</div>
<div style="padding:0 6px 8px 6px;font-size:0.78em;line-height:1.35;text-align:left;overflow-wrap:anywhere;"><i>Quercus mongolica</i></div>
<div style="font-size:0.70em;border-top:1px solid #ddd;padding:7px 6px 6px 6px;line-height:1.5;text-align:left;"><a href="https://commons.wikimedia.org/wiki/File:Mongolian_Oak_in_October.jpg" target="_blank" rel="noopener">출처: Wikimedia Commons</a></div>
</div></div></td></tr>
<tr style="height:330px;"><td style="width:25%;vertical-align:top;padding:0;overflow-wrap:anywhere;word-break:normal;;height:330px;box-sizing:border-box;"><div style="display:block;width:100%;height:330px;min-height:330px;max-height:330px;border:1px solid #aaa;background:#fff;box-sizing:border-box;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Pinus_densiflora_01.JPG" target="_blank" rel="noopener" aria-label="소나무의 수형과 가지" style="display:block;width:100%;height:180px;overflow:hidden;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Pinus_densiflora_01.JPG?width=560" alt="소나무의 수형과 가지" width="230" height="180" style="display:block;width:100%;height:180px;object-fit:cover;object-position:center;"></a><div style="display:block;width:100%;height:150px;min-height:150px;max-height:150px;overflow:hidden;box-sizing:border-box;">

<div style="background:#eeeeee;padding:7px 6px;font-weight:600;font-size:0.82em;line-height:1.35;text-align:left;">PFT 5. 온대 상록침엽교목</div>
<div style="padding:8px 6px 3px 6px;font-weight:700;font-size:1em;line-height:1.3;text-align:left;">소나무</div>
<div style="padding:0 6px 8px 6px;font-size:0.78em;line-height:1.35;text-align:left;overflow-wrap:anywhere;"><i>Pinus densiflora</i></div>
<div style="font-size:0.70em;border-top:1px solid #ddd;padding:7px 6px 6px 6px;line-height:1.5;text-align:left;"><a href="https://commons.wikimedia.org/wiki/File:Pinus_densiflora_01.JPG" target="_blank" rel="noopener">출처: Wikimedia Commons</a></div>
</div></div></td>
<td style="width:25%;vertical-align:top;padding:0;overflow-wrap:anywhere;word-break:normal;;height:330px;box-sizing:border-box;"><div style="display:block;width:100%;height:330px;min-height:330px;max-height:330px;border:1px solid #aaa;background:#fff;box-sizing:border-box;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Picea_jezoensis.JPG" target="_blank" rel="noopener" aria-label="가문비나무의 가지와 수형" style="display:block;width:100%;height:180px;overflow:hidden;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Picea_jezoensis.JPG?width=560" alt="가문비나무의 가지와 수형" width="230" height="180" style="display:block;width:100%;height:180px;object-fit:cover;object-position:center;"></a><div style="display:block;width:100%;height:150px;min-height:150px;max-height:150px;overflow:hidden;box-sizing:border-box;">

<div style="background:#eeeeee;padding:7px 6px;font-weight:600;font-size:0.82em;line-height:1.35;text-align:left;">PFT 6. 아한대 상록침엽교목</div>
<div style="padding:8px 6px 3px 6px;font-weight:700;font-size:1em;line-height:1.3;text-align:left;">가문비나무</div>
<div style="padding:0 6px 8px 6px;font-size:0.78em;line-height:1.35;text-align:left;overflow-wrap:anywhere;"><i>Picea jezoensis</i></div>
<div style="font-size:0.70em;border-top:1px solid #ddd;padding:7px 6px 6px 6px;line-height:1.5;text-align:left;"><a href="https://commons.wikimedia.org/wiki/File:Picea_jezoensis.JPG" target="_blank" rel="noopener">출처: Wikimedia Commons</a></div>
</div></div></td>
<td style="width:25%;vertical-align:top;padding:0;overflow-wrap:anywhere;word-break:normal;;height:330px;box-sizing:border-box;"><div style="display:block;width:100%;height:330px;min-height:330px;max-height:330px;border:1px solid #aaa;background:#fff;box-sizing:border-box;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Betula_pendula_tree.jpg" target="_blank" rel="noopener" aria-label="자작나무의 흰 수피와 잎" style="display:block;width:100%;height:180px;overflow:hidden;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Betula_pendula_tree.jpg?width=560" alt="자작나무의 흰 수피와 잎" width="230" height="180" style="display:block;width:100%;height:180px;object-fit:cover;object-position:center;"></a><div style="display:block;width:100%;height:150px;min-height:150px;max-height:150px;overflow:hidden;box-sizing:border-box;">

<div style="background:#eeeeee;padding:7px 6px;font-weight:600;font-size:0.82em;line-height:1.35;text-align:left;">PFT 7. 아한대 낙엽활엽교목</div>
<div style="padding:8px 6px 3px 6px;font-weight:700;font-size:1em;line-height:1.3;text-align:left;">자작나무</div>
<div style="padding:0 6px 8px 6px;font-size:0.78em;line-height:1.35;text-align:left;overflow-wrap:anywhere;"><i>Betula pendula</i></div>
<div style="font-size:0.70em;border-top:1px solid #ddd;padding:7px 6px 6px 6px;line-height:1.5;text-align:left;"><a href="https://commons.wikimedia.org/wiki/File:Betula_pendula_tree.jpg" target="_blank" rel="noopener">출처: Wikimedia Commons</a></div>
</div></div></td>
<td style="width:25%;vertical-align:top;padding:0;overflow-wrap:anywhere;word-break:normal;;height:330px;box-sizing:border-box;"><div style="display:block;width:100%;height:330px;min-height:330px;max-height:330px;border:1px solid #aaa;background:#fff;box-sizing:border-box;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Festuca_arundinacea_kz03.jpg" target="_blank" rel="noopener" aria-label="큰김의털의 잎과 꽃차례" style="display:block;width:100%;height:180px;overflow:hidden;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Festuca_arundinacea_kz03.jpg?width=560" alt="큰김의털의 잎과 꽃차례" width="230" height="180" style="display:block;width:100%;height:180px;object-fit:cover;object-position:center;"></a><div style="display:block;width:100%;height:150px;min-height:150px;max-height:150px;overflow:hidden;box-sizing:border-box;">

<div style="background:#eeeeee;padding:7px 6px;font-weight:600;font-size:0.82em;line-height:1.35;text-align:left;">PFT 8. 온대 C3 화본형 초본</div>
<div style="padding:8px 6px 3px 6px;font-weight:700;font-size:1em;line-height:1.3;text-align:left;">큰김의털</div>
<div style="padding:0 6px 8px 6px;font-size:0.78em;line-height:1.35;text-align:left;overflow-wrap:anywhere;"><i>Festuca arundinacea</i></div>
<div style="font-size:0.70em;border-top:1px solid #ddd;padding:7px 6px 6px 6px;line-height:1.5;text-align:left;"><a href="https://commons.wikimedia.org/wiki/File:Festuca_arundinacea_kz03.jpg" target="_blank" rel="noopener">출처: Wikimedia Commons</a></div>
</div></div></td></tr>
<tr style="height:330px;"><td style="width:25%;vertical-align:top;padding:0;overflow-wrap:anywhere;word-break:normal;;height:330px;box-sizing:border-box;"><div style="display:block;width:100%;height:330px;min-height:330px;max-height:330px;border:1px solid #aaa;background:#fff;box-sizing:border-box;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Themeda_triandra_var._japonica.JPG" target="_blank" rel="noopener" aria-label="솔새의 줄기와 꽃차례" style="display:block;width:100%;height:180px;overflow:hidden;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Themeda_triandra_var._japonica.JPG?width=560" alt="솔새의 줄기와 꽃차례" width="230" height="180" style="display:block;width:100%;height:180px;object-fit:cover;object-position:center;"></a><div style="display:block;width:100%;height:150px;min-height:150px;max-height:150px;overflow:hidden;box-sizing:border-box;">

<div style="background:#eeeeee;padding:7px 6px;font-weight:600;font-size:0.82em;line-height:1.35;text-align:left;">PFT 9. 열대·난온대 C4 화본형 초본</div>
<div style="padding:8px 6px 3px 6px;font-weight:700;font-size:1em;line-height:1.3;text-align:left;">솔새</div>
<div style="padding:0 6px 8px 6px;font-size:0.78em;line-height:1.35;text-align:left;overflow-wrap:anywhere;"><i>Themeda triandra var. japonica</i></div>
<div style="font-size:0.70em;border-top:1px solid #ddd;padding:7px 6px 6px 6px;line-height:1.5;text-align:left;"><a href="https://commons.wikimedia.org/wiki/File:Themeda_triandra_var._japonica.JPG" target="_blank" rel="noopener">출처: Wikimedia Commons</a></div>
</div></div></td>
<td style="width:25%;vertical-align:top;padding:0;overflow-wrap:anywhere;word-break:normal;;height:330px;box-sizing:border-box;"><div style="display:block;width:100%;height:330px;min-height:330px;max-height:330px;border:1px solid #aaa;background:#fff;box-sizing:border-box;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Ephedra_equisetina_1.jpg" target="_blank" rel="noopener" aria-label="목적마황의 녹색 마디줄기" style="display:block;width:100%;height:180px;overflow:hidden;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Ephedra_equisetina_1.jpg?width=560" alt="목적마황의 녹색 마디줄기" width="230" height="180" style="display:block;width:100%;height:180px;object-fit:cover;object-position:center;"></a><div style="display:block;width:100%;height:150px;min-height:150px;max-height:150px;overflow:hidden;box-sizing:border-box;">

<div style="background:#eeeeee;padding:7px 6px;font-weight:600;font-size:0.82em;line-height:1.35;text-align:left;">PFT 10. 건조지 목본·관목</div>
<div style="padding:8px 6px 3px 6px;font-weight:700;font-size:1em;line-height:1.3;text-align:left;">목적마황</div>
<div style="padding:0 6px 8px 6px;font-size:0.78em;line-height:1.35;text-align:left;overflow-wrap:anywhere;"><i>Ephedra equisetina</i></div>
<div style="font-size:0.70em;border-top:1px solid #ddd;padding:7px 6px 6px 6px;line-height:1.5;text-align:left;"><a href="https://commons.wikimedia.org/wiki/File:Ephedra_equisetina_1.jpg" target="_blank" rel="noopener">출처: Wikimedia Commons</a></div>
</div></div></td>
<td style="width:25%;vertical-align:top;padding:0;overflow-wrap:anywhere;word-break:normal;;height:330px;box-sizing:border-box;"><div style="display:block;width:100%;height:330px;min-height:330px;max-height:330px;border:1px solid #aaa;background:#fff;box-sizing:border-box;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Pinus_pumila_Amidadake.jpg" target="_blank" rel="noopener" aria-label="고산 사면에 낮게 퍼진 눈잣나무" style="display:block;width:100%;height:180px;overflow:hidden;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Pinus_pumila_Amidadake.jpg?width=560" alt="고산 사면에 낮게 퍼진 눈잣나무" width="230" height="180" style="display:block;width:100%;height:180px;object-fit:cover;object-position:center;"></a><div style="display:block;width:100%;height:150px;min-height:150px;max-height:150px;overflow:hidden;box-sizing:border-box;">

<div style="background:#eeeeee;padding:7px 6px;font-weight:600;font-size:0.82em;line-height:1.35;text-align:left;">PFT 11. 툰드라 관목</div>
<div style="padding:8px 6px 3px 6px;font-weight:700;font-size:1em;line-height:1.3;text-align:left;">눈잣나무</div>
<div style="padding:0 6px 8px 6px;font-size:0.78em;line-height:1.35;text-align:left;overflow-wrap:anywhere;"><i>Pinus pumila</i></div>
<div style="font-size:0.70em;border-top:1px solid #ddd;padding:7px 6px 6px 6px;line-height:1.5;text-align:left;"><a href="https://commons.wikimedia.org/wiki/File:Pinus_pumila_Amidadake.jpg" target="_blank" rel="noopener">출처: Wikimedia Commons</a></div>
</div></div></td>
<td style="width:25%;vertical-align:top;padding:0;overflow-wrap:anywhere;word-break:normal;;height:330px;box-sizing:border-box;"><div style="display:block;width:100%;height:330px;min-height:330px;max-height:330px;border:1px solid #aaa;background:#fff;box-sizing:border-box;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Kobresia_myosuroides.jpg" target="_blank" rel="noopener" aria-label="좀바늘사초의 낮은 잎 포기" style="display:block;width:100%;height:180px;overflow:hidden;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Kobresia_myosuroides.jpg?width=560" alt="좀바늘사초의 낮은 잎 포기" width="230" height="180" style="display:block;width:100%;height:180px;object-fit:cover;object-position:center;"></a><div style="display:block;width:100%;height:150px;min-height:150px;max-height:150px;overflow:hidden;box-sizing:border-box;">

<div style="background:#eeeeee;padding:7px 6px;font-weight:600;font-size:0.82em;line-height:1.35;text-align:left;">PFT 12. 한랭지 사초형 초본</div>
<div style="padding:8px 6px 3px 6px;font-weight:700;font-size:1em;line-height:1.3;text-align:left;">좀바늘사초</div>
<div style="padding:0 6px 8px 6px;font-size:0.78em;line-height:1.35;text-align:left;overflow-wrap:anywhere;"><i>Kobresia myosuroides</i></div>
<div style="font-size:0.70em;border-top:1px solid #ddd;padding:7px 6px 6px 6px;line-height:1.5;text-align:left;"><a href="https://commons.wikimedia.org/wiki/File:Kobresia_myosuroides.jpg" target="_blank" rel="noopener">출처: Wikimedia Commons</a></div>
</div></div></td></tr>
<tr style="height:330px;"><td style="width:25%;vertical-align:top;padding:0;overflow-wrap:anywhere;word-break:normal;;height:330px;box-sizing:border-box;"><div style="display:block;width:100%;height:330px;min-height:330px;max-height:330px;border:1px solid #aaa;background:#fff;box-sizing:border-box;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Leontopodium_alpinum_detail.jpg" target="_blank" rel="noopener" aria-label="에델바이스의 흰 포엽과 꽃차례" style="display:block;width:100%;height:180px;overflow:hidden;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Leontopodium_alpinum_detail.jpg?width=560" alt="에델바이스의 흰 포엽과 꽃차례" width="230" height="180" style="display:block;width:100%;height:180px;object-fit:cover;object-position:center;"></a><div style="display:block;width:100%;height:150px;min-height:150px;max-height:150px;overflow:hidden;box-sizing:border-box;">

<div style="background:#eeeeee;padding:7px 6px;font-weight:600;font-size:0.82em;line-height:1.35;text-align:left;">PFT 13. 고산성 비화본과 초본</div>
<div style="padding:8px 6px 3px 6px;font-weight:700;font-size:1em;line-height:1.3;text-align:left;">에델바이스</div>
<div style="padding:0 6px 8px 6px;font-size:0.78em;line-height:1.35;text-align:left;overflow-wrap:anywhere;"><i>Leontopodium alpinum</i></div>
<div style="font-size:0.70em;border-top:1px solid #ddd;padding:7px 6px 6px 6px;line-height:1.5;text-align:left;"><a href="https://commons.wikimedia.org/wiki/File:Leontopodium_alpinum_detail.jpg" target="_blank" rel="noopener">출처: Wikimedia Commons</a></div>
</div></div></td><td style="width:25%;vertical-align:top;padding:0;border:0;;overflow-wrap:anywhere;word-break:normal;;height:330px;box-sizing:border-box;"></td><td style="width:25%;vertical-align:top;padding:0;border:0;;overflow-wrap:anywhere;word-break:normal;;height:330px;box-sizing:border-box;"></td><td style="width:25%;vertical-align:top;padding:0;border:0;;overflow-wrap:anywhere;word-break:normal;;height:330px;box-sizing:border-box;"></td></tr>
</table>
</div>

<h2 style="font-size:1.55em;font-weight:700;line-height:1.35;margin:8px 0 24px;padding:0;">7.2 PFT 경쟁과 생물군계 판정</h2>

<div style="width:100%;max-width:1320px;overflow-x:auto;margin:18px auto;box-sizing:border-box;">
<table style="width:100%;min-width:1180px;max-width:1320px;margin:0;box-sizing:border-box;table-layout:fixed;border-collapse:collapse;font-size:0.94em;line-height:1.5;">
<colgroup><col style="width:24%;"><col style="width:76%;"></colgroup>
<tr style="background:#eeeeee;">
<th style="width:24%;border:1px solid #777;padding:8px 10px;text-align:center;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;">용어</th>
<th style="width:76%;border:1px solid #777;padding:8px 10px;text-align:center;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;">설명</th>
</tr>
<tr>
<td style="width:24%;border:1px solid #bbb;padding:8px 10px;vertical-align:top;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere;">PFT<br>식물기능유형</td>
<td style="width:76%;border:1px solid #bbb;padding:8px 10px;vertical-align:top;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere;">잎의 형태와 계절성, 생장 형태, 기후 적응 특성이 비슷한 식물을 묶은 분류 단위이다. BIOME4에서는 각 PFT에 생물기후적 제약조건을 적용하고, 가능한 유형의 생장량을 계산한다.</td>
</tr>
<tr style="background:#f5f5f5;">
<td style="width:24%;border:1px solid #bbb;padding:8px 10px;vertical-align:top;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;">NPP<br>순일차생산량</td>
<td style="width:76%;border:1px solid #bbb;padding:8px 10px;vertical-align:top;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere;">식물이 광합성으로 고정한 탄소량에서 식물의 호흡으로 소비한 탄소량을 뺀 값이다. 식물체의 생장에 순수하게 남는 탄소량을 나타낸다.</td>
</tr>
<tr>
<td style="width:24%;border:1px solid #bbb;padding:8px 10px;vertical-align:top;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;">optPFT<br>우점 식물기능유형</td>
<td style="width:76%;border:1px solid #bbb;padding:8px 10px;vertical-align:top;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere;">생물기후적 제약을 통과한 목본 PFT 가운데 NPP가 가장 크게 계산된 유형이다. 예를 들어 optPFT = 4이면 온대 낙엽활엽교목(PFT 4)이 우점 목본 유형으로 선택된 것이다.</td>
</tr>
<tr style="background:#f5f5f5;">
<td style="width:24%;border:1px solid #bbb;padding:8px 10px;vertical-align:top;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;">subPFT<br>부우점 식물기능유형</td>
<td style="width:76%;border:1px solid #bbb;padding:8px 10px;vertical-align:top;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere;">우점 목본 PFT 다음으로 NPP가 크게 계산된 유형이다.</td>
</tr>
<tr>
<td style="width:24%;border:1px solid #bbb;padding:8px 10px;vertical-align:top;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;">present(n)</td>
<td style="width:76%;border:1px solid #bbb;padding:8px 10px;vertical-align:top;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere;">PFT n의 NPP가 0보다 크면 참, 0이면 거짓으로 판정하는 생육 가능성 지표이다. 해당 기후와 토양수분 조건에서 그 PFT가 생육할 수 있는지를 나타낸다. PFT 12는 별도 기준으로 처리한다.</td>
</tr>
<tr style="background:#f5f5f5;">
<td style="width:24%;border:1px solid #bbb;padding:8px 10px;vertical-align:top;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;">GDD5<br>기준온도 5°C 초과 적산온도(성장도일)</td>
<td style="width:76%;border:1px solid #bbb;padding:8px 10px;vertical-align:top;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere;">날짜별 일평균기온에서 기준온도 5°C를 뺀 값 중 양수만 1년 동안 합산한 값이다. 예를 들어 일평균기온이 10°C인 날은 5°C·일, 4°C인 날은 0°C·일로 계산한다. 값이 클수록 식물 생육에 필요한 열량이 많이 축적되었음을 뜻한다.</td>
</tr>
<tr>
<td style="width:24%;border:1px solid #bbb;padding:8px 10px;vertical-align:top;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;">TCM<br>최한월 평균기온</td>
<td style="width:76%;border:1px solid #bbb;padding:8px 10px;vertical-align:top;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere;">12개월 가운데 월평균기온이 가장 낮은 달의 평균기온이다. 가장 추운 하루의 최저기온이 아니라 가장 추운 달의 평균적인 기온을 뜻하며, 값이 높을수록 겨울이 온화하다.</td>
</tr>
</table>
</div>

<h3 style="font-size:1.28em;font-weight:700;line-height:1.4;margin:34px 0 16px;padding:0;">7.2.1 PFT 4 우점 시 생물군계 판정</h3>

<div style="width:100%;max-width:1320px;overflow-x:auto;margin:18px auto 34px;box-sizing:border-box;">
<table style="width:100%;min-width:980px;max-width:1320px;margin:0;box-sizing:border-box;table-layout:fixed;border-collapse:collapse;background:#fff;border:1px solid #333;font-size:0.90em;">
<tr><th style="width:50.0%;padding:5px 7px;border:1px solid #333;background:#eeeeee;text-align:center;;overflow-wrap:anywhere;word-break:normal;">대표 PFT 4 수종</th><th style="width:50.0%;padding:5px 7px;border:1px solid #333;background:#eeeeee;text-align:center;;overflow-wrap:anywhere;word-break:normal;">생물군계 경관</th></tr>
<tr><td style="padding:0;border:1px solid #777;text-align:center;vertical-align:middle;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Mongolian%20Oak%20in%20October.jpg" target="_blank" rel="noopener" style="display:block;width:100%;height:220px;min-height:220px;max-height:220px;overflow:hidden;line-height:0;padding:0;margin:0;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Mongolian%20Oak%20in%20October.jpg?width=440" style="display:block!important;width:100%!important;height:220px!important;min-height:220px!important;max-height:220px!important;max-width:100%!important;object-fit:contain!important;object-position:center!important;margin:0!important;padding:0!important;border:0!important;background:#eeeeee;" width="440" height="220"" alt="신갈나무"></a></td><td style="padding:0;border:1px solid #777;text-align:center;vertical-align:middle;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Temperate%20deciduous%20forest.tif" target="_blank" rel="noopener" style="display:block;width:100%;height:220px;min-height:220px;max-height:220px;overflow:hidden;line-height:0;padding:0;margin:0;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Temperate%20deciduous%20forest.tif?width=440" style="display:block!important;width:100%!important;height:220px!important;min-height:220px!important;max-height:220px!important;max-width:100%!important;object-fit:contain!important;object-position:center!important;margin:0!important;padding:0!important;border:0!important;background:#eeeeee;" width="440" height="220"" alt="온대 낙엽활엽수림"></a></td></tr>
<tr><td style="padding:6px 8px;border:1px solid #777;vertical-align:top;line-height:1.4;word-break:keep-all;;overflow-wrap:anywhere;word-break:normal;">신갈나무 (Quercus mongolica)<br>PFT 4 · 온대 낙엽활엽교목<br>넓은 잎 · 겨울 낙엽<br><a href="https://commons.wikimedia.org/wiki/File:Mongolian%20Oak%20in%20October.jpg" target="_blank" rel="noopener" style="font-size:0.86em;">출처: Wikimedia Commons</a></td><td style="padding:6px 8px;border:1px solid #777;vertical-align:top;line-height:1.4;word-break:keep-all;;overflow-wrap:anywhere;word-break:normal;">온대 낙엽활엽수림<br>낙엽활엽교목이 상층 수관을 주로 형성<br><a href="https://commons.wikimedia.org/wiki/File:Temperate%20deciduous%20forest.tif" target="_blank" rel="noopener" style="font-size:0.86em;">출처: Wikimedia Commons</a></td></tr>
</table>
</div>

<p style="margin:0;padding:0;font-size:16px;line-height:36px;height:36px;">&nbsp;</p>

<div style="display:block;width:100%;max-width:1320px;min-width:0;overflow:visible;margin:0 auto 18px;box-sizing:border-box;">
<table style="display:table!important;width:100%!important;min-width:0!important;max-width:1320px!important;margin:0;box-sizing:border-box;table-layout:fixed!important;border-collapse:collapse;font-size:0.94em;line-height:1.5;white-space:normal!important;"><colgroup><col style="width:24%;"><col style="width:54%;"><col style="width:22%;"></colgroup><tr style="background:#eeeeee;"><th style="padding:8px 10px;border:1px solid #777;background:#eeeeee;text-align:left;vertical-align:top;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.4;">판정 조건</th><th style="padding:8px 10px;border:1px solid #777;background:#eeeeee;text-align:left;vertical-align:top;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.4;">식생·기후가 뜻하는 바</th><th style="padding:8px 10px;border:1px solid #777;background:#eeeeee;text-align:left;vertical-align:top;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.4;">분류 결과</th></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">optPFT = 4</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">온대 낙엽활엽교목(PFT 4)이 우점</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">PFT 6·3·5의 생육 가능성과 기온으로 세분</td></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">present(6) = 참<br>TCM (최한월 평균기온) &lt; −15°C</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">PFT 6도 생육할 수 있지만, 가장 추운 달의 평균기온이 −15°C보다 낮아 겨울이 춥다.</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">한랭 혼효림<br>(Biome 9)</td></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">present(6) = 참<br>TCM (최한월 평균기온) ≥ −15°C</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">PFT 4와 PFT 6이 모두 생육할 수 있으며, 가장 추운 달의 평균기온이 −15°C 이상으로 겨울이 비교적 온화하다.</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">냉온대 혼효림<br>(Biome 7)</td></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">present(6) = 거짓<br>present(3) = 참</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">PFT 6은 생육하지 못하지만 PFT 3은 생육할 수 있다. 낙엽활엽교목이 우세하고 상록활엽교목도 자랄 수 있는 조건이다.</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">난온대 혼효림<br>(Biome 6)</td></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">present(6) = 거짓<br>present(3) = 거짓<br>present(5) = 참<br>GDD5 (기준온도 5°C 초과 적산온도, 성장도일) &gt; 3,000°C·일<br>TCM (최한월 평균기온) &gt; 3°C</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">PFT 5가 생육할 수 있다. GDD5가 3,000°C·일을 넘고 최한월 평균기온이 3°C보다 높아, 생육기 열량이 충분하고 겨울도 온화하다.</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">난온대 혼효림<br>(Biome 6)</td></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">그 밖의 경우</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">PFT 6·3·5의 혼효림 판정 조건에 해당하지 않는다.</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">온대 낙엽활엽수림<br>(Biome 4)</td></tr></table>
</div>

<h3 style="font-size:1.28em;font-weight:700;line-height:1.4;margin:34px 0 16px;padding:0;">7.2.2 난온대 혼효림의 판정 조건</h3>

<p style="margin:0 0 20px;line-height:1.75;">난온대 혼효림(Biome 6)은 온대 상록활엽교목·낙엽활엽교목·상록침엽교목의 우점 관계와 생육 가능성을 바탕으로 판정한다. 다음 네 가지 식생·기후 조합 가운데 하나에 해당하면 Biome 6으로 분류한다.</p>

<div style="width:100%;max-width:1320px;overflow-x:auto;margin:18px auto 34px;box-sizing:border-box;">
<table style="width:100%;min-width:980px;max-width:1320px;margin:0;box-sizing:border-box;table-layout:fixed;border-collapse:collapse;background:#fff;border:1px solid #333;font-size:0.90em;">
<tr><th style="width:33.3%;padding:5px 7px;border:1px solid #333;background:#eeeeee;text-align:center;;overflow-wrap:anywhere;word-break:normal;">온대 상록활엽교목</th><th style="width:33.3%;padding:5px 7px;border:1px solid #333;background:#eeeeee;text-align:center;;overflow-wrap:anywhere;word-break:normal;">온대 낙엽활엽교목</th><th style="width:33.3%;padding:5px 7px;border:1px solid #333;background:#eeeeee;text-align:center;;overflow-wrap:anywhere;word-break:normal;">생물군계 경관</th></tr>
<tr><td style="padding:0;border:1px solid #777;text-align:center;vertical-align:middle;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Castanopsis%20sieboldii1.jpg" target="_blank" rel="noopener" style="display:block;width:100%;height:220px;min-height:220px;max-height:220px;overflow:hidden;line-height:0;padding:0;margin:0;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Castanopsis%20sieboldii1.jpg?width=440" style="display:block!important;width:100%!important;height:220px!important;min-height:220px!important;max-height:220px!important;max-width:100%!important;object-fit:contain!important;object-position:center!important;margin:0!important;padding:0!important;border:0!important;background:#eeeeee;" width="320" height="220"" alt="구실잣밤나무"></a></td><td style="padding:0;border:1px solid #777;text-align:center;vertical-align:middle;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Mongolian%20Oak%20in%20October.jpg" target="_blank" rel="noopener" style="display:block;width:100%;height:220px;min-height:220px;max-height:220px;overflow:hidden;line-height:0;padding:0;margin:0;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Mongolian%20Oak%20in%20October.jpg?width=440" style="display:block!important;width:100%!important;height:220px!important;min-height:220px!important;max-height:220px!important;max-width:100%!important;object-fit:contain!important;object-position:center!important;margin:0!important;padding:0!important;border:0!important;background:#eeeeee;" width="320" height="220"" alt="신갈나무"></a></td><td style="padding:0;border:1px solid #777;text-align:center;vertical-align:middle;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Forest%20in%20Yakushima%2028.jpg" target="_blank" rel="noopener" style="display:block;width:100%;height:220px;min-height:220px;max-height:220px;overflow:hidden;line-height:0;padding:0;margin:0;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Forest%20in%20Yakushima%2028.jpg?width=440" style="display:block!important;width:100%!important;height:220px!important;min-height:220px!important;max-height:220px!important;max-width:100%!important;object-fit:contain!important;object-position:center!important;margin:0!important;padding:0!important;border:0!important;background:#eeeeee;" width="320" height="220"" alt="난온대 혼효림"></a></td></tr>
<tr><td style="padding:6px 8px;border:1px solid #777;vertical-align:top;line-height:1.4;word-break:keep-all;;overflow-wrap:anywhere;word-break:normal;">구실잣밤나무 (Castanopsis sieboldii)<br>PFT 3 · 온대 상록활엽교목<br>넓은 잎 · 겨울에도 잎 유지<br><a href="https://commons.wikimedia.org/wiki/File:Castanopsis%20sieboldii1.jpg" target="_blank" rel="noopener" style="font-size:0.86em;">출처: Wikimedia Commons</a></td><td style="padding:6px 8px;border:1px solid #777;vertical-align:top;line-height:1.4;word-break:keep-all;;overflow-wrap:anywhere;word-break:normal;">신갈나무 (Quercus mongolica)<br>PFT 4 · 온대 낙엽활엽교목<br>넓은 잎 · 겨울 낙엽<br><a href="https://commons.wikimedia.org/wiki/File:Mongolian%20Oak%20in%20October.jpg" target="_blank" rel="noopener" style="font-size:0.86em;">출처: Wikimedia Commons</a></td><td style="padding:6px 8px;border:1px solid #777;vertical-align:top;line-height:1.4;word-break:keep-all;;overflow-wrap:anywhere;word-break:normal;">난온대 혼효림<br>상록활엽교목과 다른 온대 교목이 함께 수관을 형성<br><a href="https://commons.wikimedia.org/wiki/File:Forest%20in%20Yakushima%2028.jpg" target="_blank" rel="noopener" style="font-size:0.86em;">출처: Wikimedia Commons</a></td></tr>
</table>
</div>

<p style="margin:0;padding:0;font-size:16px;line-height:36px;height:36px;">&nbsp;</p>

<div style="display:block;width:100%;max-width:1320px;min-width:0;overflow:visible;margin:0 auto 18px;box-sizing:border-box;">
<table style="display:table!important;width:100%!important;min-width:0!important;max-width:1320px!important;margin:0;box-sizing:border-box;table-layout:fixed!important;border-collapse:collapse;font-size:0.94em;line-height:1.5;white-space:normal!important;"><colgroup><col style="width:24%;"><col style="width:54%;"><col style="width:22%;"></colgroup><tr style="background:#eeeeee;"><th style="padding:8px 10px;border:1px solid #777;background:#eeeeee;text-align:left;vertical-align:top;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.4;">판정 조건</th><th style="padding:8px 10px;border:1px solid #777;background:#eeeeee;text-align:left;vertical-align:top;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.4;">식생·기후가 뜻하는 바</th><th style="padding:8px 10px;border:1px solid #777;background:#eeeeee;text-align:left;vertical-align:top;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.4;">분류 결과</th></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">optPFT = 3</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">온대 상록활엽교목(PFT 3)의 NPP가 목본 PFT 가운데 가장 크다. 상록활엽교목이 우점하는 식생으로 판정한다.</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">난온대 혼효림<br>(Biome 6)</td></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">optPFT = 4<br>present(6) = 거짓<br>present(3) = 참</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">온대 낙엽활엽교목(PFT 4)의 NPP가 가장 크다. 온대 상록활엽교목(PFT 3)도 자랄 수 있지만, 아한대 상록침엽교목(PFT 6)은 자라기 어렵다. 낙엽활엽교목이 우세하고 상록활엽교목이 함께 자랄 수 있는 조합이다.</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">난온대 혼효림<br>(Biome 6)</td></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">optPFT = 4<br>present(6) = 거짓<br>present(3) = 거짓<br>present(5) = 참<br>GDD5 &gt; 3,000°C·일<br>TCM &gt; 3°C</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">낙엽활엽교목(PFT 4)이 우세하고 온대 상록침엽교목(PFT 5)도 자랄 수 있다. GDD5가 3,000°C·일을 넘는 것은 생육기에 5°C를 초과하는 열량이 충분히 쌓였다는 뜻이다. TCM이 3°C보다 높으면 가장 추운 달의 평균기온도 3°C를 넘으므로 겨울이 온화하다.</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">난온대 혼효림<br>(Biome 6)</td></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">optPFT = 5<br>present(3) = 참</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">온대 상록침엽교목(PFT 5)의 NPP가 가장 크고, 온대 상록활엽교목(PFT 3)도 자랄 수 있다. 상록침엽교목이 우세하면서 상록활엽교목이 함께 자랄 수 있는 조합이다.</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">난온대 혼효림<br>(Biome 6)</td></tr></table>
</div>

<h3 style="font-size:1.28em;font-weight:700;line-height:1.4;margin:34px 0 16px;padding:0;">7.2.3 PFT 4·6 조합에 따른 생물군계 판정</h3>

<div style="width:100%;max-width:1320px;overflow-x:auto;margin:18px auto 34px;box-sizing:border-box;">
<table style="width:100%;min-width:980px;max-width:1320px;margin:0;box-sizing:border-box;table-layout:fixed;border-collapse:collapse;background:#fff;border:1px solid #333;font-size:0.90em;">
<tr><th style="width:33.3%;padding:5px 7px;border:1px solid #333;background:#eeeeee;text-align:center;;overflow-wrap:anywhere;word-break:normal;">온대 낙엽활엽교목</th><th style="width:33.3%;padding:5px 7px;border:1px solid #333;background:#eeeeee;text-align:center;;overflow-wrap:anywhere;word-break:normal;">아한대 상록침엽교목</th><th style="width:33.3%;padding:5px 7px;border:1px solid #333;background:#eeeeee;text-align:center;;overflow-wrap:anywhere;word-break:normal;">생물군계 경관</th></tr>
<tr><td style="padding:0;border:1px solid #777;text-align:center;vertical-align:middle;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Mongolian%20Oak%20in%20October.jpg" target="_blank" rel="noopener" style="display:block;width:100%;height:220px;min-height:220px;max-height:220px;overflow:hidden;line-height:0;padding:0;margin:0;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Mongolian%20Oak%20in%20October.jpg?width=440" style="display:block!important;width:100%!important;height:220px!important;min-height:220px!important;max-height:220px!important;max-width:100%!important;object-fit:contain!important;object-position:center!important;margin:0!important;padding:0!important;border:0!important;background:#eeeeee;" width="320" height="220"" alt="신갈나무"></a></td><td style="padding:0;border:1px solid #777;text-align:center;vertical-align:middle;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Picea%20jezoensis%2002.jpg" target="_blank" rel="noopener" style="display:block;width:100%;height:220px;min-height:220px;max-height:220px;overflow:hidden;line-height:0;padding:0;margin:0;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Picea%20jezoensis%2002.jpg?width=440" style="display:block!important;width:100%!important;height:220px!important;min-height:220px!important;max-height:220px!important;max-width:100%!important;object-fit:contain!important;object-position:center!important;margin:0!important;padding:0!important;border:0!important;background:#eeeeee;" width="320" height="220"" alt="가문비나무"></a></td><td style="padding:0;border:1px solid #777;text-align:center;vertical-align:middle;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Mixed-forest.jpg" target="_blank" rel="noopener" style="display:block;width:100%;height:220px;min-height:220px;max-height:220px;overflow:hidden;line-height:0;padding:0;margin:0;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Mixed-forest.jpg?width=440" style="display:block!important;width:100%!important;height:220px!important;min-height:220px!important;max-height:220px!important;max-width:100%!important;object-fit:contain!important;object-position:center!important;margin:0!important;padding:0!important;border:0!important;background:#eeeeee;" width="320" height="220"" alt="냉온대 혼효림"></a></td></tr>
<tr><td style="padding:6px 8px;border:1px solid #777;vertical-align:top;line-height:1.4;word-break:keep-all;;overflow-wrap:anywhere;word-break:normal;">신갈나무 (Quercus mongolica)<br>PFT 4 · 온대 낙엽활엽교목<br><a href="https://commons.wikimedia.org/wiki/File:Mongolian%20Oak%20in%20October.jpg" target="_blank" rel="noopener" style="font-size:0.86em;">출처: Wikimedia Commons</a></td><td style="padding:6px 8px;border:1px solid #777;vertical-align:top;line-height:1.4;word-break:keep-all;;overflow-wrap:anywhere;word-break:normal;">가문비나무 (Picea jezoensis)<br>PFT 6 · 아한대 상록침엽교목<br><a href="https://commons.wikimedia.org/wiki/File:Picea%20jezoensis%2002.jpg" target="_blank" rel="noopener" style="font-size:0.86em;">출처: Wikimedia Commons</a></td><td style="padding:6px 8px;border:1px solid #777;vertical-align:top;line-height:1.4;word-break:keep-all;;overflow-wrap:anywhere;word-break:normal;">냉온대 혼효림<br>낙엽활엽교목과 상록침엽교목이 함께 수관을 형성<br><a href="https://commons.wikimedia.org/wiki/File:Mixed-forest.jpg" target="_blank" rel="noopener" style="font-size:0.86em;">출처: Wikimedia Commons</a></td></tr>
</table>
</div>

<p style="margin:0;padding:0;font-size:16px;line-height:36px;height:36px;">&nbsp;</p>

<div style="display:block;width:100%;max-width:1320px;min-width:0;overflow:visible;margin:0 auto 18px;box-sizing:border-box;">
<table style="display:table!important;width:100%!important;min-width:0!important;max-width:1320px!important;margin:0;box-sizing:border-box;table-layout:fixed!important;border-collapse:collapse;font-size:0.94em;line-height:1.5;white-space:normal!important;"><colgroup><col style="width:24%;"><col style="width:54%;"><col style="width:22%;"></colgroup><tr style="background:#eeeeee;"><th style="padding:8px 10px;border:1px solid #777;background:#eeeeee;text-align:left;vertical-align:top;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.4;">판정 조건</th><th style="padding:8px 10px;border:1px solid #777;background:#eeeeee;text-align:left;vertical-align:top;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.4;">식생·기후가 뜻하는 바</th><th style="padding:8px 10px;border:1px solid #777;background:#eeeeee;text-align:left;vertical-align:top;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.4;">분류 결과</th></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">optPFT = 4<br>present(6) = 참<br>TCM (최한월 평균기온) ≥ −15°C</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">PFT 4가 우점하고 PFT 6도 생육 가능. TCM ≥ −15°C로 겨울이 상대적으로 온화함</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">냉온대 혼효림<br>(Biome 7)</td></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">optPFT = 4<br>present(6) = 참<br>TCM (최한월 평균기온) &lt; −15°C</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">PFT 4와 PFT 6이 함께 생육하지만 TCM < −15°C로 겨울이 더 추움</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">한랭 혼효림<br>(Biome 9)</td></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">optPFT = 6<br>present(4) = 참<br>GDD5 (기준온도 5°C 초과 적산온도, 성장도일) &gt; 900°C·일<br>TCM (최한월 평균기온) &gt; −19°C</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">PFT 6이 우점하고 PFT 4도 생육 가능. 생육기 열량이 충분하고 겨울 한랭도가 상대적으로 낮음</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">냉온대 혼효림<br>(Biome 7)</td></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">optPFT = 6<br>present(4) = 거짓<br>GDD5 (기준온도 5°C 초과 적산온도, 성장도일) &gt; 900°C·일<br>TCM (최한월 평균기온) &gt; −19°C</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">기후조건은 충족하지만 PFT 4는 생육하지 못함</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">냉온대 침엽수림<br>(Biome 8)</td></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">optPFT = 6<br>present(4) = 참<br>GDD5 (기준온도 5°C 초과 적산온도, 성장도일) ≤ 900°C·일 또는<br>TCM (최한월 평균기온) ≤ −19°C</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">PFT 4는 생육 가능하지만 생육기 열량이 부족하거나 겨울 한랭도가 높음</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">한랭 혼효림<br>(Biome 9)</td></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">optPFT = 6<br>present(4) = 거짓<br>GDD5 (기준온도 5°C 초과 적산온도, 성장도일) ≤ 900°C·일 또는<br>TCM (최한월 평균기온) ≤ −19°C</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">생육기 열량이 부족하거나 겨울이 춥고, PFT 4도 생육하지 못함</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">상록 타이가·산지림<br>(Biome 10)</td></tr></table>
</div>

<h3 style="font-size:1.28em;font-weight:700;line-height:1.4;margin:34px 0 16px;padding:0;">7.2.4 PFT 5 우점 시 생물군계 판정</h3>

<div style="width:100%;max-width:1320px;overflow-x:auto;margin:18px auto 34px;box-sizing:border-box;">
<table style="width:100%;min-width:980px;max-width:1320px;margin:0;box-sizing:border-box;table-layout:fixed;border-collapse:collapse;background:#fff;border:1px solid #333;font-size:0.90em;">
<tr><th style="width:50.0%;padding:5px 7px;border:1px solid #333;background:#eeeeee;text-align:center;;overflow-wrap:anywhere;word-break:normal;">대표 PFT 5 수종</th><th style="width:50.0%;padding:5px 7px;border:1px solid #333;background:#eeeeee;text-align:center;;overflow-wrap:anywhere;word-break:normal;">생물군계 경관</th></tr>
<tr><td style="padding:0;border:1px solid #777;text-align:center;vertical-align:middle;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Abies%20holophylla%201.JPG" target="_blank" rel="noopener" style="display:block;width:100%;height:220px;min-height:220px;max-height:220px;overflow:hidden;line-height:0;padding:0;margin:0;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Abies%20holophylla%201.JPG?width=440" style="display:block!important;width:100%!important;height:220px!important;min-height:220px!important;max-height:220px!important;max-width:100%!important;object-fit:contain!important;object-position:center!important;margin:0!important;padding:0!important;border:0!important;background:#eeeeee;" width="440" height="220"" alt="전나무"></a></td><td style="padding:0;border:1px solid #777;text-align:center;vertical-align:middle;overflow:hidden;"><a href="https://commons.wikimedia.org/wiki/File:Misty%20coniferous%20forest%20on%20a%20slope%20(Unsplash).jpg" target="_blank" rel="noopener" style="display:block;width:100%;height:220px;min-height:220px;max-height:220px;overflow:hidden;line-height:0;padding:0;margin:0;"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Misty%20coniferous%20forest%20on%20a%20slope%20(Unsplash).jpg?width=440" style="display:block!important;width:100%!important;height:220px!important;min-height:220px!important;max-height:220px!important;max-width:100%!important;object-fit:contain!important;object-position:center!important;margin:0!important;padding:0!important;border:0!important;background:#eeeeee;" width="440" height="220"" alt="온대 침엽수림"></a></td></tr>
<tr><td style="padding:6px 8px;border:1px solid #777;vertical-align:top;line-height:1.4;word-break:keep-all;;overflow-wrap:anywhere;word-break:normal;">전나무 (Abies holophylla)<br>PFT 5 · 온대 상록침엽교목<br>바늘잎 · 겨울에도 잎 유지<br><a href="https://commons.wikimedia.org/wiki/File:Abies%20holophylla%201.JPG" target="_blank" rel="noopener" style="font-size:0.86em;">출처: Wikimedia Commons</a></td><td style="padding:6px 8px;border:1px solid #777;vertical-align:top;line-height:1.4;word-break:keep-all;;overflow-wrap:anywhere;word-break:normal;">온대 침엽수림<br>상록침엽교목이 상층 수관을 주로 형성<br><a href="https://commons.wikimedia.org/wiki/File:Misty%20coniferous%20forest%20on%20a%20slope%20(Unsplash).jpg" target="_blank" rel="noopener" style="font-size:0.86em;">출처: Wikimedia Commons</a></td></tr>
</table>
</div>

<p style="margin:0;padding:0;font-size:16px;line-height:36px;height:36px;">&nbsp;</p>

<div style="display:block;width:100%;max-width:1320px;min-width:0;overflow:visible;margin:0 auto 18px;box-sizing:border-box;">
<table style="display:table!important;width:100%!important;min-width:0!important;max-width:1320px!important;margin:0;box-sizing:border-box;table-layout:fixed!important;border-collapse:collapse;font-size:0.94em;line-height:1.5;white-space:normal!important;"><colgroup><col style="width:24%;"><col style="width:54%;"><col style="width:22%;"></colgroup><tr style="background:#eeeeee;"><th style="padding:8px 10px;border:1px solid #777;background:#eeeeee;text-align:left;vertical-align:top;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.4;">판정 조건</th><th style="padding:8px 10px;border:1px solid #777;background:#eeeeee;text-align:left;vertical-align:top;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.4;">식생·기후가 뜻하는 바</th><th style="padding:8px 10px;border:1px solid #777;background:#eeeeee;text-align:left;vertical-align:top;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.4;">분류 결과</th></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">optPFT = 5</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">온대 상록침엽교목(PFT 5)이 우점</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">PFT 3의 생육 가능성과 부우점 PFT로 세분</td></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">present(3) = 참</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">PFT 3도 생육 가능하여 상록활엽교목이 공존할 만큼 겨울이 온화함</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">난온대 혼효림<br>(Biome 6)</td></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">present(3) = 거짓<br>subPFT = 4<br>우점 PFT NPP − 부우점 PFT NPP &lt; 50 g C m⁻² yr⁻¹</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">PFT 3은 생육하지 못함. PFT 4가 차순위이며 두 PFT의 NPP 차이가 50 g C m⁻² yr⁻¹ 미만이어도 PFT 5 우점으로 분류됨</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">온대 침엽수림<br>(Biome 5)</td></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">present(3) = 거짓<br>subPFT = 7</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">PFT 3은 생육하지 못하고 PFT 7이 차순위임</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">한랭 혼효림<br>(Biome 9)</td></tr>
<tr><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">그 밖의 경우</td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;"><div style="display:block;width:100%;min-width:0;max-width:100%;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;"><div style="display:block!important;width:100%;min-width:0;height:auto!important;min-height:2.9em;max-height:none!important;white-space:normal!important;overflow:visible!important;text-overflow:clip!important;overflow-wrap:anywhere!important;word-break:keep-all!important;line-height:1.45!important;-webkit-line-clamp:unset!important;">PFT 5 우점이며 난온대·한랭 혼효림 조건에 해당하지 않음</div></div></td><td style="padding:8px 10px;border:1px solid #bbb;vertical-align:top;text-align:left;white-space:normal!important;overflow-wrap:break-word;word-break:normal;line-height:1.5;height:auto;">온대 침엽수림<br>(Biome 5)</td></tr></table>
</div>

### 7.3 BIOME4 생물군계

<div style="width:100%;max-width:100%;overflow-x:visible;margin:18px auto;box-sizing:border-box;">
<table style="width:100%;min-width:100%;max-width:100%;margin:0;box-sizing:border-box;table-layout:fixed;border-collapse:collapse;font-size:0.88em;line-height:1.4;">
<colgroup><col style="width:4%;"><col style="width:26%;"><col style="width:70%;"></colgroup>
<tr style="background:#eeeeee;"><th style="border:1px solid #555;padding:6px;overflow-wrap:break-word;white-space:normal;vertical-align:middle;">ID</th><th style="border:1px solid #555;padding:6px;overflow-wrap:break-word;white-space:normal;vertical-align:middle;">생물군계</th><th style="border:1px solid #555;padding:6px;overflow-wrap:break-word;white-space:normal;vertical-align:middle;">주요 식생 구조</th></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">1</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">열대 상록수림<br><span style="font-size:0.88em;">Tropical evergreen forest</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">연중 고온다습하여 상록활엽수가 밀집한 숲</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">2</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">열대 반낙엽수림<br><span style="font-size:0.88em;">Tropical semi-deciduous forest</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">열대의 계절적 건기에 일부 교목이 잎을 떨어뜨리는 숲</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">3</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">열대 낙엽수림 및 소림<br><span style="font-size:0.88em;">Tropical deciduous forest/woodland</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">건기가 뚜렷하여 낙엽교목이 우세하고 수관이 비교적 개방된 숲</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">4</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">온대 낙엽활엽수림<br><span style="font-size:0.88em;">Temperate deciduous forest</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">계절별 기온 변화에 따라 활엽교목이 겨울철에 낙엽하는 숲</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">5</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">온대 침엽수림<br><span style="font-size:0.88em;">Temperate conifer forest</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">온대 기후에서 침엽교목이 우세한 숲</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">6</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">난온대 혼효림<br><span style="font-size:0.88em;">Warm mixed forest</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">비교적 온난한 온대에서 상록성·낙엽성 교목 등이 함께 나타나는 숲</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">7</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">냉온대 혼효림<br><span style="font-size:0.88em;">Cool mixed forest</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">냉량한 온대에서 낙엽활엽수와 침엽수가 함께 우세한 숲</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">8</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">냉온대 침엽수림<br><span style="font-size:0.88em;">Cool conifer forest</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">냉량한 기후에서 침엽교목이 우세한 숲</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">9</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">한랭 혼효림<br><span style="font-size:0.88em;">Cold mixed forest</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">한랭한 지역에서 침엽교목과 낙엽교목이 혼재하는 숲</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">10</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">상록 타이가 및 산지림<br><span style="font-size:0.88em;">Evergreen taiga/montane forest</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">아한대 또는 고산의 긴 추위에 적응한 상록침엽수림</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">11</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">낙엽 타이가 및 산지림<br><span style="font-size:0.88em;">Deciduous taiga/montane forest</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">아한대 또는 고산에서 낙엽성 교목이 우세한 숲</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">12</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">열대 사바나<br><span style="font-size:0.88em;">Tropical savanna</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">열대의 계절적 건조 환경에서 초본층과 드문드문 선 교목이 공존하는 식생</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">13</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">열대 건생관목림<br><span style="font-size:0.88em;">Tropical xerophytic shrubland</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">강수량이 적은 열대에서 건조에 적응한 관목이 우세한 식생</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">14</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">온대 건생관목림<br><span style="font-size:0.88em;">Temperate xerophytic shrubland</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">건조한 온대에서 내건성 관목이 우세한 식생</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">15</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">온대 경엽수 소림<br><span style="font-size:0.88em;">Temperate sclerophyll woodland</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">여름철 건조 등에 적응한 단단한 잎의 목본이 형성한 개방된 숲</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">16</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">온대 활엽수 사바나<br><span style="font-size:0.88em;">Temperate broadleaved savanna</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">온대에서 초본층과 개방된 활엽교목층이 함께 나타나는 식생</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">17</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">개방 침엽수림<br><span style="font-size:0.88em;">Open conifer woodland</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">침엽교목이 낮은 밀도로 분포하여 수관 사이가 개방된 숲</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">18</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">아한대 소림 및 개방식생<br><span style="font-size:0.88em;">Boreal parkland</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">아한대 수목한계 부근에서 드문드문 선 교목과 저목·초본이 공존하는 식생</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">19</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">열대 초원<br><span style="font-size:0.88em;">Tropical grassland</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">열대에서 화본과 초본이 주로 우세하고 교목 피복이 낮은 초원</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">20</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">온대 초원<br><span style="font-size:0.88em;">Temperate grassland</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">온대에서 초본류가 우세하며 교목층이 거의 발달하지 않은 초원</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">21</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">사막<br><span style="font-size:0.88em;">Desert</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">건조지 목본 우점 또는 매우 낮은 최적 NPP로 판정되는 저생산성 식생 상태</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">22</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">스텝 툰드라<br><span style="font-size:0.88em;">Steppe tundra</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">한랭하고 비교적 건조한 조건에서 초본과 낮은 툰드라 식생이 혼재하는 지역</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">23</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">관목 툰드라<br><span style="font-size:0.88em;">Shrub tundra</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">수목한계 너머의 한랭 지역에서 관목이 우세한 툰드라</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">24</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">왜성관목 툰드라<br><span style="font-size:0.88em;">Dwarf shrub tundra</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">낮은 온도와 짧은 생육기에 적응한 왜성관목이 우세한 툰드라</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">25</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">포복관목 툰드라<br><span style="font-size:0.88em;">Prostrate shrub tundra</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">지표 가까이 기는 관목이 우세한 한랭지 툰드라</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">26</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">방석형 초본·지의류·선태류 식생<br><span style="font-size:0.88em;">Cushion-forbs, lichen and moss</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">매우 한랭한 환경에서 방석형 초본, 지의류 및 선태류가 드물게 분포하는 식생</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">27</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">무식생지<br><span style="font-size:0.88em;">Barren</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">선택 가능한 PFT가 없는 무식생 상태</td></tr>
<tr><td style="border:1px solid #aaa;padding:5px;text-align:center;;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">28</td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">빙설 피복지<br><span style="font-size:0.88em;">Land ice</span></td><td style="border:1px solid #aaa;padding:5px;overflow-wrap:break-word;word-break:keep-all;white-space:normal;vertical-align:middle;">지표가 지속적인 얼음으로 덮여 육상식생이 형성되지 않는 지역</td></tr>
</table>
</div>


#### 7.3.1 BIOME4 생물군계별 대표 사진

<div>
<table width="100%" border="1" bordercolor="#777777" cellpadding="6" cellspacing="0">
<tr>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Khao%20Yai%2C%20Thailand%2C%20Tropical%20moist%20evergreen%20forest.jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Khao%20Yai%2C%20Thailand%2C%20Tropical%20moist%20evergreen%20forest.jpg?width=560" height="180" alt="열대 상록수림"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Subtropical%20semi-evergreen%20seasonal%20forest%20in%20Northern%20Thailand.JPG" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Subtropical%20semi-evergreen%20seasonal%20forest%20in%20Northern%20Thailand.JPG?width=560" height="180" alt="열대 반낙엽수림"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Southern%20tropical%20dry%20deciduous%20forest.jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Southern%20tropical%20dry%20deciduous%20forest.jpg?width=560" height="180" alt="열대 낙엽수림 및 소림"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Temperate%20deciduous%20forest.tif" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Temperate%20deciduous%20forest.tif?width=560" height="180" alt="온대 낙엽활엽수림"></a></td>
</tr>
<tr>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>1. 열대 상록수림</b><br><i>Tropical evergreen forest</i><br><a href="https://commons.wikimedia.org/wiki/File:Khao%20Yai%2C%20Thailand%2C%20Tropical%20moist%20evergreen%20forest.jpg" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>2. 열대 반낙엽수림</b><br><i>Tropical semi-deciduous forest</i><br><a href="https://commons.wikimedia.org/wiki/File:Subtropical%20semi-evergreen%20seasonal%20forest%20in%20Northern%20Thailand.JPG" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>3. 열대 낙엽수림 및 소림</b><br><i>Tropical deciduous forest/woodland</i><br><a href="https://commons.wikimedia.org/wiki/File:Southern%20tropical%20dry%20deciduous%20forest.jpg" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>4. 온대 낙엽활엽수림</b><br><i>Temperate deciduous forest</i><br><a href="https://commons.wikimedia.org/wiki/File:Temperate%20deciduous%20forest.tif" target="_blank">출처: Wikimedia Commons</a></td>
</tr>
<tr>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Misty%20coniferous%20forest%20on%20a%20slope%20(Unsplash).jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Misty%20coniferous%20forest%20on%20a%20slope%20(Unsplash).jpg?width=560" height="180" alt="온대 침엽수림"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Forest%20in%20Yakushima%2028.jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Forest%20in%20Yakushima%2028.jpg?width=560" height="180" alt="난온대 혼효림"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Mixed-forest.jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Mixed-forest.jpg?width=560" height="180" alt="냉온대 혼효림"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Abies%20koreana%20Halla%20San%202.jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Abies%20koreana%20Halla%20San%202.jpg?width=560" height="180" alt="냉온대 침엽수림"></a></td>
</tr>
<tr>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>5. 온대 침엽수림</b><br><i>Temperate conifer forest</i><br><a href="https://commons.wikimedia.org/wiki/File:Misty%20coniferous%20forest%20on%20a%20slope%20(Unsplash).jpg" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>6. 난온대 혼효림</b><br><i>Warm mixed forest</i><br><a href="https://commons.wikimedia.org/wiki/File:Forest%20in%20Yakushima%2028.jpg" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>7. 냉온대 혼효림</b><br><i>Cool mixed forest</i><br><a href="https://commons.wikimedia.org/wiki/File:Mixed-forest.jpg" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>8. 냉온대 침엽수림</b><br><i>Cool conifer forest</i><br><a href="https://commons.wikimedia.org/wiki/File:Abies%20koreana%20Halla%20San%202.jpg" target="_blank">출처: Wikimedia Commons</a></td>
</tr>
<tr>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Lake%20Baikal%2C%20Siberian%20mixed%20taiga%20forest%2C%20Siberia%2C%20Russia.jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Lake%20Baikal%2C%20Siberian%20mixed%20taiga%20forest%2C%20Siberia%2C%20Russia.jpg?width=560" height="180" alt="한랭 혼효림"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Ergaki%2C%20Taiga%20in%20Siberia%2C%20Coniferous%20forest%2C%20Sayan%20Mountains%2C%20Siberia%2C%20Russia.jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Ergaki%2C%20Taiga%20in%20Siberia%2C%20Coniferous%20forest%2C%20Sayan%20Mountains%2C%20Siberia%2C%20Russia.jpg?width=560" height="180" alt="상록 타이가 및 산지림"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Siberian%20Larch%20Forest%20369857661.jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Siberian%20Larch%20Forest%20369857661.jpg?width=560" height="180" alt="낙엽 타이가 및 산지림"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Savanna%20landscape%20with%20trees%2C%20Northern%20Mara%2C%20Kenya.jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Savanna%20landscape%20with%20trees%2C%20Northern%20Mara%2C%20Kenya.jpg?width=560" height="180" alt="열대 사바나"></a></td>
</tr>
<tr>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>9. 한랭 혼효림</b><br><i>Cold mixed forest</i><br><a href="https://commons.wikimedia.org/wiki/File:Lake%20Baikal%2C%20Siberian%20mixed%20taiga%20forest%2C%20Siberia%2C%20Russia.jpg" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>10. 상록 타이가 및 산지림</b><br><i>Evergreen taiga/montane forest</i><br><a href="https://commons.wikimedia.org/wiki/File:Ergaki%2C%20Taiga%20in%20Siberia%2C%20Coniferous%20forest%2C%20Sayan%20Mountains%2C%20Siberia%2C%20Russia.jpg" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>11. 낙엽 타이가 및 산지림</b><br><i>Deciduous taiga/montane forest</i><br><a href="https://commons.wikimedia.org/wiki/File:Siberian%20Larch%20Forest%20369857661.jpg" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>12. 열대 사바나</b><br><i>Tropical savanna</i><br><a href="https://commons.wikimedia.org/wiki/File:Savanna%20landscape%20with%20trees%2C%20Northern%20Mara%2C%20Kenya.jpg" target="_blank">출처: Wikimedia Commons</a></td>
</tr>
<tr>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Thicket%20of%20bush%20on%20rangeland%20in%20Namibia.jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Thicket%20of%20bush%20on%20rangeland%20in%20Namibia.jpg?width=560" height="180" alt="열대 건생관목림"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Sagebrush%20landscape%20(34535852192).jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Sagebrush%20landscape%20(34535852192).jpg?width=560" height="180" alt="온대 건생관목림"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Bosque%20escler%C3%B3filo%2C%20Chile%20001%20%E2%80%94%20Inao%20V%C3%A1squez.jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Bosque%20escler%C3%B3filo%2C%20Chile%20001%20%E2%80%94%20Inao%20V%C3%A1squez.jpg?width=560" height="180" alt="온대 경엽수 소림"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Bur-oak-savanna.jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Bur-oak-savanna.jpg?width=560" height="180" alt="온대 활엽수 사바나"></a></td>
</tr>
<tr>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>13. 열대 건생관목림</b><br><i>Tropical xerophytic shrubland</i><br><a href="https://commons.wikimedia.org/wiki/File:Thicket%20of%20bush%20on%20rangeland%20in%20Namibia.jpg" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>14. 온대 건생관목림</b><br><i>Temperate xerophytic shrubland</i><br><a href="https://commons.wikimedia.org/wiki/File:Sagebrush%20landscape%20(34535852192).jpg" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>15. 온대 경엽수 소림</b><br><i>Temperate sclerophyll woodland</i><br><a href="https://commons.wikimedia.org/wiki/File:Bosque%20escler%C3%B3filo%2C%20Chile%20001%20%E2%80%94%20Inao%20V%C3%A1squez.jpg" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>16. 온대 활엽수 사바나</b><br><i>Temperate broadleaved savanna</i><br><a href="https://commons.wikimedia.org/wiki/File:Bur-oak-savanna.jpg" target="_blank">출처: Wikimedia Commons</a></td>
</tr>
<tr>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Ponderosa%20Forest.JPG" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Ponderosa%20Forest.JPG?width=560" height="180" alt="개방 침엽수림"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Tundra%20landscape%20with%20trees%20in%20Ivvavik%20National%20Park%2C%20YT.jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Tundra%20landscape%20with%20trees%20in%20Ivvavik%20National%20Park%2C%20YT.jpg?width=560" height="180" alt="아한대 소림 및 개방식생"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Serengeti%20-%20plains.jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Serengeti%20-%20plains.jpg?width=560" height="180" alt="열대 초원"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Mongolia%20steppe.JPG" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Mongolia%20steppe.JPG?width=560" height="180" alt="온대 초원"></a></td>
</tr>
<tr>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>17. 개방 침엽수림</b><br><i>Open conifer woodland</i><br><a href="https://commons.wikimedia.org/wiki/File:Ponderosa%20Forest.JPG" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>18. 아한대 소림 및 개방식생</b><br><i>Boreal parkland</i><br><a href="https://commons.wikimedia.org/wiki/File:Tundra%20landscape%20with%20trees%20in%20Ivvavik%20National%20Park%2C%20YT.jpg" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>19. 열대 초원</b><br><i>Tropical grassland</i><br><a href="https://commons.wikimedia.org/wiki/File:Serengeti%20-%20plains.jpg" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>20. 온대 초원</b><br><i>Temperate grassland</i><br><a href="https://commons.wikimedia.org/wiki/File:Mongolia%20steppe.JPG" target="_blank">출처: Wikimedia Commons</a></td>
</tr>
<tr>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Haloxylon%20persicum%20kz17.jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Haloxylon%20persicum%20kz17.jpg?width=560" height="180" alt="사막"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Karakoram-West%20Tibetan%20Plateau%20alpine%20steppe.jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Karakoram-West%20Tibetan%20Plateau%20alpine%20steppe.jpg?width=560" height="180" alt="스텝 툰드라"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Arctic%20Tundra%20(8540635536).jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Arctic%20Tundra%20(8540635536).jpg?width=560" height="180" alt="관목 툰드라"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Cassiope%20tetragona%2001(js)%2C%20Greenland.jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Cassiope%20tetragona%2001(js)%2C%20Greenland.jpg?width=560" height="180" alt="왜성관목 툰드라"></a></td>
</tr>
<tr>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>21. 사막</b><br><i>Desert</i><br><a href="https://commons.wikimedia.org/wiki/File:Haloxylon%20persicum%20kz17.jpg" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>22. 스텝 툰드라</b><br><i>Steppe tundra</i><br><a href="https://commons.wikimedia.org/wiki/File:Karakoram-West%20Tibetan%20Plateau%20alpine%20steppe.jpg" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>23. 관목 툰드라</b><br><i>Shrub tundra</i><br><a href="https://commons.wikimedia.org/wiki/File:Arctic%20Tundra%20(8540635536).jpg" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>24. 왜성관목 툰드라</b><br><i>Dwarf shrub tundra</i><br><a href="https://commons.wikimedia.org/wiki/File:Cassiope%20tetragona%2001(js)%2C%20Greenland.jpg" target="_blank">출처: Wikimedia Commons</a></td>
</tr>
<tr>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Mountain%20avens%20grow%20along%20the%20tundra%20at%20the%20top%20of%20the%20Mount%20Healy%20Trail%20(4ba5c750-cce5-45a1-96df-69bfc5d6c2f6).JPG" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Mountain%20avens%20grow%20along%20the%20tundra%20at%20the%20top%20of%20the%20Mount%20Healy%20Trail%20(4ba5c750-cce5-45a1-96df-69bfc5d6c2f6).JPG?width=560" height="180" alt="포복관목 툰드라"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Close%20up%20of%20Saxifraga%20oppositifolia%20(purple%20mountain%20saxifrage)%20(51444964715).jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Close%20up%20of%20Saxifraga%20oppositifolia%20(purple%20mountain%20saxifrage)%20(51444964715).jpg?width=560" height="180" alt="방석형 초본·지의류·선태류 식생"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Barren%20landscape.jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Barren%20landscape.jpg?width=560" height="180" alt="무식생지"></a></td>
<td width="25%" height="180" align="center" valign="middle"><a href="https://commons.wikimedia.org/wiki/File:Greenland-ice%20sheet%20hg.jpg" target="_blank"><img src="https://commons.wikimedia.org/wiki/Special:FilePath/Greenland-ice%20sheet%20hg.jpg?width=560" height="180" alt="빙설 피복지"></a></td>
</tr>
<tr>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>25. 포복관목 툰드라</b><br><i>Prostrate shrub tundra</i><br><a href="https://commons.wikimedia.org/wiki/File:Mountain%20avens%20grow%20along%20the%20tundra%20at%20the%20top%20of%20the%20Mount%20Healy%20Trail%20(4ba5c750-cce5-45a1-96df-69bfc5d6c2f6).JPG" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>26. 방석형 초본·지의류·선태류 식생</b><br><i>Cushion-forbs, lichen and moss</i><br><a href="https://commons.wikimedia.org/wiki/File:Close%20up%20of%20Saxifraga%20oppositifolia%20(purple%20mountain%20saxifrage)%20(51444964715).jpg" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>27. 무식생지</b><br><i>Barren</i><br><a href="https://commons.wikimedia.org/wiki/File:Barren%20landscape.jpg" target="_blank">출처: Wikimedia Commons</a></td>
<td width="25%" valign="top" bgcolor="#f2f2f2"><b>28. 빙설 피복지</b><br><i>Land ice</i><br><a href="https://commons.wikimedia.org/wiki/File:Greenland-ice%20sheet%20hg.jpg" target="_blank">출처: Wikimedia Commons</a></td>
</tr>
</table>
</div>


## 8절 시계열 비교


In [ ]:
#@title 8.1 시계열 비교
PFT_NAMES_EN={
    1:"Tropical Evergreen",
    2:"Tropical Raingreen",
    3:"Temperate Broadleaved Evergreen",
    4:"Temperate Summergreen",
    5:"Temperate Evergreen Conifer",
    6:"Boreal Evergreen",
    7:"Boreal Deciduous",
    8:"Temperate Grass",
    9:"Tropical/Warm-Temperate Grass",
    10:"Desert Woody Plant Type C3 and C4",
    11:"Tundra Shrub Type",
    12:"Cold Herbaceous Type",
    13:"Lichen/Forb Type",
}
PFT_NAMES={
    1:"열대 상록교목",
    2:"열대 건기낙엽교목",
    3:"온대 상록활엽교목",
    4:"온대 낙엽활엽교목",
    5:"온대 상록침엽교목",
    6:"아한대 상록침엽교목",
    7:"아한대 낙엽교목",
    8:"C3 온대 화본형 초본",
    9:"C4 열대·난온대 화본형 초본",
    10:"C3/C4 건조지 목본",
    11:"툰드라 관목",
    12:"한랭 초본",
    13:"지의류·비화본과 초본",
}

PFT_EXAMPLES_KO={
    1:"망고나무 (Mangifera indica) — 열대 상록활엽교목; 현재 실행 비활성",
    2:"티크 (Tectona grandis) — 열대 건기낙엽교목",
    3:"구실잣밤나무 (Castanopsis sieboldii) — 온대 상록활엽교목",
    4:"신갈나무 (Quercus mongolica) — 온대 낙엽활엽교목",
    5:"소나무 (Pinus densiflora) — 온대 상록침엽교목",
    6:"가문비나무 (Picea jezoensis) — 아한대 상록침엽교목",
    7:"자작나무 (Betula pendula) — 아한대 낙엽활엽교목",
    8:"큰김의털 (Festuca arundinacea) — 온대 C3 화본형 초본",
    9:"솔새 (Themeda triandra var. japonica) — 열대·난온대 C4 화본형 초본",
    10:"목적마황 (Ephedra equisetina) — 건조지 목본·관목",
    11:"눈잣나무 (Pinus pumila) — 툰드라 관목",
    12:"좀바늘사초 (Kobresia myosuroides) — 한랭지 사초형 초본",
    13:"에델바이스 (Leontopodium alpinum) — 고산성 비화본과 초본",
}

PFT_EXAMPLE_SOURCE={
    1:"BIOME4 v4.2b2; Bradshaw et al. (2025); Ni (2000)",
    2:"BIOME4 v4.2b2; Bradshaw et al. (2025); Ni (2000)",
    3:"BIOME4 v4.2b2; Bradshaw et al. (2025); Ni (2000)",
    4:"BIOME4 v4.2b2; Bradshaw et al. (2025); Ni (2000)",
    5:"BIOME4 v4.2b2; Bradshaw et al. (2025); Ni (2000)",
    6:"BIOME4 v4.2b2; Bradshaw et al. (2025); Ni (2000)",
    7:"BIOME4 v4.2b2; Bradshaw et al. (2025); Ni (2000)",
    8:"BIOME4 v4.2b2; Kaplan et al. (2003); Ni (2000)",
    9:"BIOME4 v4.2b2; Kaplan et al. (2002); Edwards et al. (2010)",
    10:"BIOME4 v4.2b2; Kaplan et al. (2002); Ni (2000)",
    11:"BIOME4 v4.2b2; Kaplan et al. (2003) Table 1",
    12:"BIOME4 v4.2b2; Kaplan et al. (2003) Table 1; Ni (2000)",
    13:"BIOME4 v4.2b2; Kaplan et al. (2003) Table 1",
}

BIOME4_NAMES={
    1:"Tropical evergreen forest",
    2:"Tropical semi-deciduous forest",
    3:"Tropical deciduous forest/woodland",
    4:"Temperate deciduous forest",
    5:"Temperate conifer forest",
    6:"Warm mixed forest",
    7:"Cool mixed forest",
    8:"Cool conifer forest",
    9:"Cold mixed forest",
    10:"Evergreen taiga/montane forest",
    11:"Deciduous taiga/montane forest",
    12:"Tropical savanna",
    13:"Tropical xerophytic shrubland",
    14:"Temperate xerophytic shrubland",
    15:"Temperate sclerophyll woodland",
    16:"Temperate broadleaved savanna",
    17:"Open conifer woodland",
    18:"Boreal parkland",
    19:"Tropical grassland",
    20:"Temperate grassland",
    21:"Desert",
    22:"Steppe tundra",
    23:"Shrub tundra",
    24:"Dwarf shrub tundra",
    25:"Prostrate shrub tundra",
    26:"Cushion-forbs, lichen and moss",
    27:"Barren",
    28:"Land ice",
}
BIOME4_NAMES_KO={
    1:"열대 상록수림",
    2:"열대 반낙엽수림",
    3:"열대 낙엽수림 및 소림",
    4:"온대 낙엽활엽수림",
    5:"온대 침엽수림",
    6:"난온대 혼효림",
    7:"냉온대 혼효림",
    8:"냉온대 침엽수림",
    9:"한랭 혼효림",
    10:"상록 타이가 및 산지림",
    11:"낙엽 타이가 및 산지림",
    12:"열대 사바나",
    13:"열대 건생관목림",
    14:"온대 건생관목림",
    15:"온대 경엽수 소림",
    16:"온대 활엽수 사바나",
    17:"개방 침엽수림",
    18:"아한대 소림 및 개방식생",
    19:"열대 초원",
    20:"온대 초원",
    21:"사막",
    22:"스텝 툰드라",
    23:"관목 툰드라",
    24:"왜성관목 툰드라",
    25:"포복관목 툰드라",
    26:"방석형 초본·지의류·선태류 식생",
    27:"무식생지",
    28:"빙설 피복지",
}
BIOME4_DESCRIPTIONS_KO={
    1:"연중 고온다습하여 상록활엽수가 밀집한 숲.",
    2:"열대의 계절적 건기에 일부 교목이 잎을 떨어뜨리는 숲.",
    3:"건기가 뚜렷하여 낙엽교목이 우세하고 수관이 비교적 개방된 숲.",
    4:"계절별 기온 변화에 따라 활엽교목이 겨울철에 낙엽하는 숲.",
    5:"온대 기후에서 침엽교목이 우세한 숲.",
    6:"비교적 온난한 온대에서 상록성·낙엽성 교목 등이 함께 나타나는 숲.",
    7:"냉량한 온대에서 낙엽활엽수와 침엽수가 함께 우세한 숲.",
    8:"냉량한 기후에서 침엽교목이 우세한 숲.",
    9:"한랭한 지역에서 침엽교목과 낙엽교목이 혼재하는 숲.",
    10:"아한대 또는 고산의 긴 추위에 적응한 상록침엽수림.",
    11:"아한대 또는 고산에서 낙엽성 교목이 우세한 숲.",
    12:"열대의 계절적 건조 환경에서 초본층과 드문드문 선 교목이 공존하는 식생.",
    13:"강수량이 적은 열대에서 건조에 적응한 관목이 우세한 식생.",
    14:"건조한 온대에서 내건성 관목이 우세한 식생.",
    15:"여름철 건조 등에 적응한 단단한 잎의 목본이 형성한 개방된 숲.",
    16:"온대에서 초본층과 개방된 활엽교목층이 함께 나타나는 식생.",
    17:"침엽교목이 낮은 밀도로 분포하여 수관 사이가 개방된 숲.",
    18:"아한대 수목한계 부근에서 드문드문 선 교목과 저목·초본이 공존하는 식생.",
    19:"열대에서 화본과 초본이 주로 우세하고 교목 피복이 낮은 초원.",
    20:"온대에서 초본류가 우세하며 교목층이 거의 발달하지 않은 초원.",
    21:"건조지 목본 우점 또는 매우 낮은 최적 NPP로 판정되는 저생산성 식생 상태.",
    22:"한랭하고 비교적 건조한 조건에서 초본과 낮은 툰드라 식생이 혼재하는 지역.",
    23:"수목한계 너머의 한랭 지역에서 관목이 우세한 툰드라.",
    24:"낮은 온도와 짧은 생육기에 적응한 왜성관목이 우세한 툰드라.",
    25:"지표 가까이 기는 관목이 우세한 한랭지 툰드라.",
    26:"매우 한랭한 환경에서 방석형 초본, 지의류 및 선태류가 드물게 분포하는 식생.",
    27:"선택 가능한 PFT가 없는 무식생 상태.",
    28:"지표가 지속적인 얼음으로 덮여 육상식생이 형성되지 않는 지역.",
}
BIOME4_EXAMPLES_KO={
    1:"Rubroshorea curtisii (=Shorea curtisii), Rubroshorea leprosula (=Shorea leprosula), Rubroshorea parvifolia (=Shorea parvifolia)",
    2:"Anthoshorea hypochra (=Shorea hypochra), Anthoshorea roxburghii (=Shorea roxburghii), Dipterocarpus kerrii",
    3:"티크(Tectona grandis), Anogeissus latifolia, Shorea robusta",
    4:"신갈나무(Quercus mongolica), 피나무(Tilia amurensis), 고로쇠나무(Acer pictum var. mono)",
    5:"전나무(Abies holophylla), 가문비나무(Picea jezoensis), 솔송나무(Tsuga sieboldii)",
    6:"구실잣밤나무(Castanopsis sieboldii), 붉가시나무(Quercus acuta), 후박나무(Machilus thunbergii)",
    7:"잣나무(Pinus koraiensis), 신갈나무(Quercus mongolica), 피나무(Tilia amurensis)",
    8:"가문비나무(Picea jezoensis), 분비나무(Abies nephrolepis), 구상나무(Abies koreana)",
    9:"Picea obovata, Larix sibirica, 자작나무(Betula pendula)",
    10:"Picea obovata, Abies sibirica, Pinus sibirica",
    11:"Larix gmelinii, Larix sibirica, 자작나무(Betula pendula)",
    12:"Vachellia tortilis, Burkea africana, Themeda triandra",
    13:"Vachellia reficiens, Senegalia mellifera, Commiphora africana",
    14:"Artemisia tridentata, Atriplex canescens, Ericameria nauseosa",
    15:"Quercus ilex, 코르크참나무(Quercus suber), Olea europaea",
    16:"Quercus macrocarpa, Andropogon gerardii, Sorghastrum nutans",
    17:"Pinus ponderosa, 로키향나무(Juniperus scopulorum), Bouteloua gracilis",
    18:"Picea mariana, Picea glauca, Larix laricina",
    19:"Themeda triandra, Heteropogon contortus, Hyparrhenia rufa",
    20:"Leymus chinensis, Stipa capillata, Festuca ovina",
    21:"Haloxylon persicum, Nitraria sphaerocarpa, Reaumuria soongarica",
    22:"Stipa purpurea, Carex moorcroftii, Artemisia stracheyi",
    23:"눈잣나무(Pinus pumila), 좀자작나무(Betula nana), Salix glauca",
    24:"북극종꽃나무(Cassiope tetragona), 들쭉나무(Vaccinium uliginosum), Empetrum nigrum",
    25:"북극버들(Salix arctica), 담자리꽃나무(Dryas octopetala), 북극콩버들(Salix polaris)",
    26:"자주범의귀(Saxifraga oppositifolia), 북극이끼장구채(Silene acaulis), 사슴지의(Cladonia rangiferina)",
    27:"해당 없음",
    28:"해당 없음",
}

BIOME4_EXAMPLE_SOURCE={
    1:"Prentice 1992 tropical rain forest; Ni 2000 tropical rain forest; FAO lowland evergreen rainforest dominants",
    2:"Prentice 1992 tropical seasonal forest; Ni 2000 tropical seasonal forest; FAO semi-evergreen rainforest dominants",
    3:"Prentice 1992 tropical raingreen; FAO Asian dry-deciduous forests",
    4:"Prentice 1992 temperate summergreen; Ni 2000 temperate deciduous forest",
    5:"Ni 2000 temperate conifer forest: Picea-Abies-Tsuga",
    6:"Prentice 1992 broad-leaved evergreen/warm mixed; Ni 2000 subtropical evergreen/mixed forest",
    7:"Prentice 1992 cool mixed; Ni 2000 temperate/boreal mixed forest",
    8:"Prentice 1992 cool conifer; Ni 2000 Picea-Abies-Tsuga; Korean subalpine conifer study",
    9:"Prentice 1992 cold mixed: cool-temperate conifer + boreal summergreen",
    10:"Prentice 1992 taiga; BIOME4 boreal evergreen tree PFT",
    11:"Prentice 1992 cold deciduous; Ni 2000 Larix-Betula-Populus",
    12:"Prentice 1992 tropical dry forest/savanna; African savanna vegetation references",
    13:"Prentice 1992 xerophytic woods/scrub; dry thornbush references",
    14:"Prentice 1992 xerophytic woods/scrub; Kaplan 2003 temperate xerophytic shrubland",
    15:"Prentice 1992 sclerophyll/succulent; Mediterranean evergreen woodland",
    16:"BIOME4 open temperate broadleaf woodland; USDA oak-savanna dominant species",
    17:"BIOME4 open conifer woodland; USNVC/USDA ponderosa-juniper woodland",
    18:"BIOME4 boreal parkland; forest-tundra/treeline vegetation",
    19:"Prentice 1992 warm grass/shrub; BIOME4 C4 tropical grass PFT",
    20:"Prentice 1992 cool grass/shrub; Ni 2000 Stipa-Leymus/Festuca steppe",
    21:"Prentice 1992 desert/semidesert; Ni 2000 desert/arid shrubland",
    22:"Kaplan 2003 graminoid/forb tundra; Ni 2000 alpine steppe/meadow",
    23:"Kaplan 2003 Table 1 low/high shrub tundra Typical Taxa",
    24:"Kaplan 2003 Table 1 erect dwarf-shrub tundra Typical Taxa",
    25:"Kaplan 2003 Table 1 prostrate dwarf-shrub tundra Typical Taxa",
    26:"Kaplan 2003 Table 1 cushion-forb, lichen and moss tundra Typical Taxa",
    27:"BIOME4 v4.2b2: optpft=0 -> Barren",
    28:"BIOME4 v4.2b2: Land ice",
}


import matplotlib.pyplot as plt
_configure_korean_plotting()
import pandas as pd
from IPython.display import display

timeseries_rows=[]

for sc in scenario_configs:
    name=sc["name"]

    for model,subdir in [(sc["model"],CASE_DIRS[sc["model"]])]:
        df=pd.read_csv(scenario_outputs[name]/subdir/"summary_timeseries.csv")
        tmp=df.copy()
        tmp["scenario"]=name
        tmp["model"]=model
        timeseries_rows.append(tmp)

scenario_timeseries=pd.concat(timeseries_rows,ignore_index=True)

comparison_report=exp_root/"comparison_report"
comparison_report.mkdir(parents=True,exist_ok=True)
for old_name in (
    "within_scenario_dynamic_minus_static.csv",
    "between_scenario_minus_reference.csv",
    "grid_biome4_agreement.csv",
    "grid_difference_statistics.csv",
    "scenario_end_comparison.csv",
):
    old_file=comparison_report/old_name
    if old_file.exists():
        old_file.unlink()

scenario_timeseries.to_csv(
    comparison_report/"all_scenarios_timeseries.csv",
    index=False,
    encoding="utf-8-sig"
)

fig,axes=plt.subplots(1,3,figsize=(16,4.8),constrained_layout=True)

metrics=[
    ("mean_elevation_m","Mean elevation (m)"),
    ("mean_soil_depth_m","Mean soil depth (m)"),
    ("mean_npp","Mean NPP (gC m$^{-2}$ yr$^{-1}$)")
]

for ax,(col,ylabel) in zip(axes,metrics):

    for (scenario,model),g in scenario_timeseries.groupby(["scenario","model"]):
        g=g.sort_values("ka_bp",ascending=False)
        ls="-" if model=="동적" else "--"
        ax.plot(
            g["ka_bp"],
            g[col],
            linestyle=ls,
            lw=1.8,
            label=f"{scenario} / {model}"
        )

    ax.invert_xaxis()
    ax.set_xlabel("Age (ka BP)")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=.2)

axes[0].legend(frameon=False,fontsize=8)
plt.show()

end_summary=scenario_timeseries[
    np.isclose(
        scenario_timeseries["ka_bp"].to_numpy(float),
        float(end_ka),
        atol=1e-6
    )
][[
    "scenario","model",
    "mean_elevation_m","mean_soil_depth_m","mean_npp"
]]

print("종료시점")
display(end_summary.round(3))


## 9절 5 kyr 간격 공간분포


In [ ]:
#@title 9.1 5 kyr 간격 공간분포 비교
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
BIOME4_COLORS={
    1:"#00661a",2:"#4d801a",3:"#99661a",4:"#00c800",
    5:"#006666",6:"#001a80",7:"#ccffcc",8:"#009933",
    9:"#80334d",10:"#1a1ae6",11:"#33ccff",12:"#b3e61a",
    13:"#ffb399",14:"#ffd9b3",15:"#669900",16:"#cccc00",
    17:"#ff99e6",18:"#9b7fff",19:"#e6b34d",20:"#ffe57f",
    21:"#ffffe6",22:"#e6e600",23:"#66e680",24:"#808033",
    25:"#cc9999",26:"#b399ff",27:"#ccccb3",28:"#ccffff",
}
biome4_cmap=ListedColormap([BIOME4_COLORS[i] for i in range(1,29)])
biome4_norm=BoundaryNorm(np.arange(0.5,29.5,1),biome4_cmap.N)

def _read_map(scenario,model,var,map_age):
    p=scenario_outputs[scenario]/CASE_DIRS[model]/snapshot_dir_name(float(map_age))/(var+".tif")
    if not p.is_file():
        raise FileNotFoundError(
            f"스냅샷 파일 없음: {p}. 2.1 모델 및 입력자료 준비부터 6.1 시나리오 실행까지 다시 실행하세요."
        )
    with rasterio.open(p) as src:
        return src.read(1,masked=True).filled(np.nan).astype(float)

map_variables=[
    ("soil_depth_m","토심 (m)"),
    ("biome4_full_id","BIOME4 생물군계"),
    ("elevation_m","고도 (m)"),
    ("npp","NPP")
]
map_output_dir=comparison_report/"maps_5k_by_variable"
map_output_dir.mkdir(parents=True,exist_ok=True)

ages=list(selected_ages)
ages_per_page=5
print(
    "공간분포는 연도별 묶음이 아니라 항목별로 정리합니다. "
    "각 항목에서 5 kyr 간격 시점을 순서대로 비교합니다."
)

for var,label in map_variables:
    item_dir=map_output_dir/var
    item_dir.mkdir(parents=True,exist_ok=True)
    for old_png in item_dir.glob("*.png"):
        old_png.unlink()

    print(f"\n===== {label} =====")

    # 항목 전체 시점·시나리오에 공통 연속형 색범위 적용
    common_vmin=common_vmax=None
    if var!="biome4_full_id":
        vals=[]
        for age in ages:
            for sc in scenario_configs:
                a=_read_map(sc["name"],sc["model"],var,age)
                v=a[np.isfinite(a)]
                if v.size:
                    vals.append(v)
        if not vals:
            raise RuntimeError(f"{label}: 유효 격자값이 없습니다.")
        all_valid=np.concatenate(vals)
        common_vmin=float(all_valid.min())
        common_vmax=float(all_valid.max())
        if common_vmin==common_vmax:
            common_vmax=common_vmin+1e-8

    for page_no,start_idx in enumerate(range(0,len(ages),ages_per_page),1):
        page_ages=ages[start_idx:start_idx+ages_per_page]
        nrows=len(scenario_configs)
        ncols=len(page_ages)
        fig,axes=plt.subplots(
            nrows,ncols,
            figsize=(3.35*ncols,3.15*nrows),
            squeeze=False,
            constrained_layout=True
        )
        present=set()
        last_im=None

        for r,sc in enumerate(scenario_configs):
            for c,age in enumerate(page_ages):
                ax=axes[r,c]
                a=_read_map(sc["name"],sc["model"],var,age)

                if var=="biome4_full_id":
                    present.update(
                        int(x) for x in np.unique(a[np.isfinite(a)])
                        if 1<=int(x)<=28
                    )
                    last_im=ax.imshow(
                        np.ma.masked_invalid(a),
                        cmap=biome4_cmap,norm=biome4_norm,
                        interpolation="nearest"
                    )
                else:
                    last_im=ax.imshow(
                        np.ma.masked_invalid(a),
                        vmin=common_vmin,vmax=common_vmax,
                        interpolation="nearest"
                    )

                if r==0:
                    ax.set_title(f"{age:g} ka BP",fontsize=10)
                if c==0:
                    ax.set_ylabel(
                        f'{sc["name"]}\n{sc["model"]}',
                        fontsize=9
                    )
                ax.set_xticks([])
                ax.set_yticks([])

        if var!="biome4_full_id" and last_im is not None:
            fig.colorbar(
                last_im,ax=list(axes.ravel()),
                fraction=.018,pad=.012,label=label
            )
        elif present:
            handles=[
                Patch(
                    facecolor=BIOME4_COLORS[i],edgecolor="0.35",
                    label=f"{i}. {BIOME4_NAMES_KO[i]}"
                )
                for i in sorted(present)
            ]
            fig.legend(
                handles=handles,loc="outside lower center",
                ncol=min(4,len(handles)),
                frameon=False,fontsize=8
            )

        fig.suptitle(
            f"{label} — 5 kyr 간격 비교 "
            f"({page_ages[0]:g}–{page_ages[-1]:g} ka BP)",
            fontsize=13
        )
        out_png=item_dir/f"{var}_page_{page_no:02d}.png"
        fig.savefig(out_png,dpi=160,bbox_inches="tight")
        plt.show()
        plt.close(fig)

print("항목별 5 kyr 지도 저장:",map_output_dir)


### 9.2 하도·곡저와 사면


In [ ]:
#@title 9.2.1 시간에 따른 하도·곡저–사면 구분
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
_configure_korean_plotting()

landform_rows=[]
landform_output_dir=comparison_report/"valley_hillslope_5k"
landform_output_dir.mkdir(parents=True,exist_ok=True)

for sc in scenario_configs:
    scenario_name=sc["name"]
    model_name=sc["model"]
    for age in selected_ages:
        valley_raw=_read_map(
            scenario_name,model_name,"valley_mask",age
        )
        biome=_read_map(
            scenario_name,model_name,"biome4_full_id",age
        )

        valid=np.isfinite(valley_raw)
        valley=valid & (valley_raw>=0.5)
        hillslope=valid & (~valley)
        desert=valid & np.isclose(biome,21)
        barren=valid & np.isclose(biome,27)

        n_valid=int(np.sum(valid))
        n_valley=int(np.sum(valley))
        n_hillslope=int(np.sum(hillslope))
        n_desert=int(np.sum(desert))
        n_barren=int(np.sum(barren))

        landform_rows.append({
            "scenario":scenario_name,
            "model":model_name,
            "ka_bp":float(age),
            "valid_cells":n_valid,
            "valley_cells":n_valley,
            "hillslope_cells":n_hillslope,
            "valley_area_percent":100.0*n_valley/n_valid if n_valid else np.nan,
            "hillslope_area_percent":100.0*n_hillslope/n_valid if n_valid else np.nan,
            "desert_cells":n_desert,
            "desert_in_valley_percent":(
                100.0*np.sum(desert & valley)/n_desert
                if n_desert else 0.0
            ),
            "valley_desert_percent":(
                100.0*np.sum(desert & valley)/n_valley
                if n_valley else 0.0
            ),
            "barren_cells":n_barren,
            "barren_in_valley_percent":(
                100.0*np.sum(barren & valley)/n_barren
                if n_barren else 0.0
            ),
            "valley_barren_percent":(
                100.0*np.sum(barren & valley)/n_valley
                if n_valley else 0.0
            ),
        })

landform_timeseries=pd.DataFrame(landform_rows).sort_values(
    ["scenario","ka_bp"],ascending=[True,False]
).reset_index(drop=True)
landform_timeseries.to_csv(
    comparison_report/"valley_hillslope_biome_overlap_5k.csv",
    index=False,encoding="utf-8-sig"
)

print("하도·곡저/사면 및 BIOME4 21·27 중첩")
display(
    landform_timeseries[[
        "scenario","model","ka_bp",
        "valley_area_percent","hillslope_area_percent",
        "desert_cells","desert_in_valley_percent","valley_desert_percent",
        "barren_cells","barren_in_valley_percent","valley_barren_percent"
    ]].round(2)
)

# 5 kyr 간격 하도·곡저 / 사면 지도
landform_cmap=ListedColormap(["#d9d9d9","#4c78a8"])
landform_handles=[
    Patch(facecolor="#d9d9d9",edgecolor="0.4",label="사면"),
    Patch(facecolor="#4c78a8",edgecolor="0.4",label="하도·곡저"),
]

ages=list(selected_ages)
ages_per_page=5
for old_png in landform_output_dir.glob("*.png"):
    old_png.unlink()

for page_no,start_idx in enumerate(range(0,len(ages),ages_per_page),1):
    page_ages=ages[start_idx:start_idx+ages_per_page]
    fig,axes=plt.subplots(
        len(scenario_configs),len(page_ages),
        figsize=(3.2*len(page_ages),3.0*len(scenario_configs)),
        squeeze=False,constrained_layout=True
    )

    for r,sc in enumerate(scenario_configs):
        for c,age in enumerate(page_ages):
            ax=axes[r,c]
            valley_raw=_read_map(
                sc["name"],sc["model"],"valley_mask",age
            )
            valid=np.isfinite(valley_raw)
            landform=np.where(
                valid,
                np.where(valley_raw>=0.5,1.0,0.0),
                np.nan
            )
            ax.imshow(
                np.ma.masked_invalid(landform),
                cmap=landform_cmap,vmin=0,vmax=1,
                interpolation="nearest"
            )
            if r==0:
                ax.set_title(f"{age:g} ka BP",fontsize=10)
            if c==0:
                ax.set_ylabel(
                    f'{sc["name"]}\n{sc["model"]}',
                    fontsize=9
                )
            ax.set_xticks([])
            ax.set_yticks([])

    fig.legend(
        handles=landform_handles,
        loc="outside lower center",
        ncol=2,frameon=False
    )
    fig.suptitle(
        f"하도·곡저 / 사면 — {page_ages[0]:g}–{page_ages[-1]:g} ka BP",
        fontsize=13
    )
    out_png=landform_output_dir/f"valley_hillslope_page_{page_no:02d}.png"
    fig.savefig(out_png,dpi=160,bbox_inches="tight")
    plt.show()
    plt.close(fig)

# 하도·곡저 면적비율 변화
fig,ax=plt.subplots(figsize=(10,4.6),constrained_layout=True)
for sc in scenario_configs:
    sub=landform_timeseries[
        (landform_timeseries.scenario==sc["name"]) &
        (landform_timeseries.model==sc["model"])
    ].sort_values("ka_bp",ascending=False)
    ax.plot(
        sub["ka_bp"],sub["valley_area_percent"],
        marker="o",markersize=3,linewidth=1.6,
        label=f'{sc["name"]} / {sc["model"]}'
    )
ax.set_xlim(float(start_ka),float(end_ka))
ax.set_ylim(0,100)
ax.set_xlabel("시기 (ka BP)")
ax.set_ylabel("하도·곡저 면적비율 (%)")
ax.grid(alpha=.2)
ax.legend(frameon=False,fontsize=8)
fig.savefig(
    comparison_report/"valley_area_percent_timeseries.png",
    dpi=180,bbox_inches="tight"
)
plt.show()


## 10절 BIOME4 생물군계 면적비율


In [ ]:
#@title 10.1 BIOME4 생물군계 면적비율
import numpy as np, pandas as pd
from IPython.display import display
report_dir=comparison_report

if "biome4_full_id_counts" not in scenario_timeseries.columns:
    raise RuntimeError("BIOME4 생물군계 집계가 없습니다.")

def parse_biome_counts(value):
    counts={code:0 for code in BIOME4_NAMES}
    for token in str(value).split(";"):
        if ":" in token:
            code,count=token.strip().split(":",1)
            code=int(code)
            if code in counts:
                counts[code]+=int(float(count))
    return counts

records=[]
for row in scenario_timeseries.itertuples(index=False):
    counts=parse_biome_counts(row.biome4_full_id_counts)
    for code,num in counts.items():
        records.append({
            "scenario":row.scenario,"model":row.model,
            "ka_bp":float(row.ka_bp),"biome_id":code,
            "biome_name":BIOME4_NAMES[code],
            "biome_name_ko":BIOME4_NAMES_KO[code],
            "biome_description_ko":BIOME4_DESCRIPTIONS_KO[code],
            "biome_example_ko":BIOME4_EXAMPLES_KO[code],
            "biome_example_source":BIOME4_EXAMPLE_SOURCE[code],
            "cells":num,
            "area_percent":100.0*num/int(row.n_cells)
        })

biome_ratios=pd.DataFrame(records)
biome_ratios.to_csv(report_dir/"biome4_biome_ratio_all_times.csv",index=False,encoding="utf-8-sig")

# 시나리오별 BIOME4 생물군계 면적비율 시계열
import matplotlib.pyplot as plt
_configure_korean_plotting()
from matplotlib.patches import Patch

present_ids=sorted(
    biome_ratios.loc[biome_ratios.cells>0,"biome_id"].unique().astype(int)
)
fig,axes=plt.subplots(
    len(scenario_configs),1,
    figsize=(12,max(3.2,3.1*len(scenario_configs))),
    sharex=True,squeeze=False,constrained_layout=True
)
for ax,sc in zip(axes[:,0],scenario_configs):
    sub=biome_ratios[
        (biome_ratios.scenario==sc["name"]) &
        (biome_ratios.model==sc["model"])
    ]
    wide=sub.pivot(index="ka_bp",columns="biome_id",values="area_percent")
    wide=wide.reindex(columns=present_ids,fill_value=0).sort_index(ascending=False).fillna(0)
    ax.stackplot(
        wide.index.to_numpy(float),
        *[wide[k].to_numpy(float) for k in present_ids],
        colors=[BIOME4_COLORS[k] for k in present_ids],
        linewidth=0
    )
    ax.set_xlim(start_ka,end_ka)
    ax.set_ylim(0,100)
    ax.set_ylabel("면적비율 (%)")
    ax.set_title(f'{sc["name"]} / {sc["model"]}',loc="left",fontsize=10)
    ax.grid(alpha=.15)
axes[-1,0].set_xlabel("시기 (ka BP)")
if present_ids:
    fig.legend(
        handles=[Patch(color=BIOME4_COLORS[k],label=f"{k}. {BIOME4_NAMES_KO[k]}")
                 for k in present_ids],
        loc="outside lower center",ncol=min(4,len(present_ids)),frameon=False,fontsize=8
    )
fig.savefig(report_dir/"biome4_area_percent_timeseries.png",dpi=180,bbox_inches="tight")
plt.show()


## 11절 PFT 진단


In [ ]:
#@title 11.1 PFT 진단
_required_pft_cols=[]
for pid in range(1,14):
    _required_pft_cols += [
        f"pft{pid:02d}_npp_mean",
        f"pft{pid:02d}_lai_mean",
        f"pft{pid:02d}_constraint_pass_fraction",
    ]
_required_pft_cols += [f"selected_pft{pid:02d}_count" for pid in range(0,15)]
_missing=[c for c in _required_pft_cols if c not in scenario_timeseries.columns]
if _missing:
    raise RuntimeError(
        "현재 결과는 PFT 진단 열이 없는 이전 실행 결과입니다. "
        "2.1 모델 및 입력자료 준비를 실행한 뒤 6.1 시나리오 실행부터 다시 실행하십시오. "
        f"누락 예: {_missing[:6]}"
    )

pft_rows=[]
chosen_rows=[]
for _,row in scenario_timeseries.iterrows():
    n_cells=int(row["n_cells"])
    for pid,pname in PFT_NAMES.items():
        prefix=f"pft{pid:02d}_"
        pft_rows.append({
            "scenario":row["scenario"],
            "model":row["model"],
            "ka_bp":float(row["ka_bp"]),
            "pft_id":pid,
            "pft_name":pname,
            "pft_example_ko":PFT_EXAMPLES_KO[pid],
            "pft_example_source":PFT_EXAMPLE_SOURCE[pid],
            "potential_npp":float(row[prefix+"npp_mean"]),
            "potential_lai":float(row[prefix+"lai_mean"]),
            "constraint_pass_fraction":float(row[prefix+"constraint_pass_fraction"])
        })

    counts=[int(row[f"selected_pft{pid:02d}_count"]) for pid in range(0,15)]
    total=sum(counts)
    if total!=n_cells:
        raise RuntimeError(
            f'{row["scenario"]} {row["ka_bp"]} ka: '
            f'BIOME4 optPFT 집계 격자수({total})와 유효 격자수({n_cells})가 일치하지 않습니다.'
        )
    optpft_special={
        0:"미선택(optPFT=0)",
        14:"목본·초본 공우점(optPFT=14; 별도 PFT가 아님)",
    }
    for pid,num in enumerate(counts):
        chosen_rows.append({
            "scenario":row["scenario"],
            "model":row["model"],
            "ka_bp":float(row["ka_bp"]),
            "pft_id":pid,
            "pft_name":PFT_NAMES.get(pid,optpft_special.get(pid,f"optPFT {pid}")),
            "pft_example_ko":PFT_EXAMPLES_KO.get(pid,""),
            "pft_example_source":PFT_EXAMPLE_SOURCE.get(pid,""),
            "chosen_cells":num,
            "chosen_cells_percent":100.0*num/total
        })

pft_long=pd.DataFrame(pft_rows)
chosen_pft=pd.DataFrame(chosen_rows)
pft_long.to_csv(report_dir/"pft_all_times.csv",index=False,encoding="utf-8-sig")
chosen_pft.to_csv(report_dir/"selected_pft_grid_ratio_all_times.csv",index=False,encoding="utf-8-sig")

# 5 kyr 간격 진단표
_pft_5k_mask=np.zeros(len(pft_long),dtype=bool)
for age in selected_ages:
    _pft_5k_mask |= np.isclose(pft_long.ka_bp.to_numpy(float),float(age),atol=1e-6)
pft_5k=pft_long.loc[_pft_5k_mask].copy()
chosen_pft_5k=chosen_pft.loc[
    np.logical_or.reduce([
        np.isclose(chosen_pft.ka_bp.to_numpy(float),float(age),atol=1e-6)
        for age in selected_ages
    ])
].copy()
pft_5k.to_csv(report_dir/"pft_5k_diagnostics.csv",index=False,encoding="utf-8-sig")
chosen_pft_5k.to_csv(report_dir/"selected_pft_5k_ratio.csv",index=False,encoding="utf-8-sig")

print("PFT 잠재생산·LAI·제약 통과율: 5 kyr 간격")
display(pft_5k.round(4))
print("BIOME4 원본 optPFT 격자비율: 5 kyr 간격")
display(chosen_pft_5k.round(3))

# PFT별 생물기후 제약 통과율 시계열
import matplotlib.pyplot as plt
_configure_korean_plotting()

fig,axes=plt.subplots(
    4,4,
    figsize=(16,12),
    sharex=True,sharey=True,
    constrained_layout=True
)
axes=axes.ravel()

series_keys=[
    (sc["name"],sc["model"])
    for sc in scenario_configs
]
for pid in range(1,14):
    ax=axes[pid-1]
    for scenario_name,model_name in series_keys:
        sub=pft_long[
            (pft_long.scenario==scenario_name) &
            (pft_long.model==model_name) &
            (pft_long.pft_id==pid)
        ].sort_values("ka_bp",ascending=False)
        if sub.empty:
            continue
        ax.plot(
            sub["ka_bp"].to_numpy(float),
            100.0*sub["constraint_pass_fraction"].to_numpy(float),
            lw=1.8,
            label=f"{scenario_name} / {model_name}"
        )
    ax.set_xlim(float(start_ka),float(end_ka))
    ax.set_ylim(-2,102)
    ax.set_title(f"PFT {pid}. {PFT_NAMES[pid]}",fontsize=10)
    ax.grid(alpha=.2)
    if (pid-1)%4==0:
        ax.set_ylabel("제약 통과율 (%)")
    if pid>=10:
        ax.set_xlabel("시기 (ka BP)")

for ax in axes[13:]:
    ax.axis("off")

handles,labels=axes[0].get_legend_handles_labels()
if handles:
    fig.legend(
        handles,labels,
        loc="outside lower center",
        ncol=min(4,len(labels)),
        frameon=False,
        fontsize=8
    )
fig.savefig(
    report_dir/"pft_constraint_pass_lines.png",
    dpi=180,bbox_inches="tight"
)
plt.show()



## 12절 시나리오 5 kyr 간격 비교


In [ ]:
#@title 12.1 시나리오 설정과 5 kyr 간격 결과
settings_table=pd.DataFrame(scenario_configs)

_result_candidates=[
    "scenario","model","ka_bp",
    "mean_elevation_m","mean_soil_depth_m","mean_npp","mean_eemt",
    "mean_lai","mean_aet_mm_yr","mean_runoff_mm_yr",
    "mean_agb_proxy_kg_m2",
    "dominant_biome4_id"
]
result_columns=[c for c in _result_candidates if c in scenario_timeseries.columns]

_5k_mask=np.zeros(len(scenario_timeseries),dtype=bool)
for age in selected_ages:
    _5k_mask |= np.isclose(
        scenario_timeseries.ka_bp.to_numpy(float),
        float(age),atol=1e-6
    )

comparison_5k_table=(
    scenario_timeseries.loc[_5k_mask,result_columns]
    .sort_values(["ka_bp","scenario"],ascending=[False,True])
    .reset_index(drop=True)
)

end_table=(
    scenario_timeseries.loc[
        np.isclose(
            scenario_timeseries.ka_bp.to_numpy(float),
            float(end_ka),atol=1e-6
        ),
        result_columns
    ]
    .copy()
    .reset_index(drop=True)
)

print("시나리오 설정")
display(settings_table.round(4))
print("5 kyr 간격 결과")
display(comparison_5k_table.round(4))
print("종료시점 결과")
display(end_table.round(4))

settings_table.to_csv(
    report_dir/"scenario_settings.csv",index=False,encoding="utf-8-sig"
)
comparison_5k_table.to_csv(
    report_dir/"scenario_5k_results.csv",index=False,encoding="utf-8-sig"
)
end_table.to_csv(
    report_dir/"scenario_end_results.csv",index=False,encoding="utf-8-sig"
)


## 13절 보고서 파일


In [ ]:
#@title 13.1 전체 실험 보고서 자료 압축
# 보고서의 식생 참고자료 CSV
report_dir=comparison_report
pft_reference_table=pd.DataFrame([
    {
        "PFT":pid,
        "BIOME4 원명":PFT_NAMES_EN[pid],
        "국문명":PFT_NAMES[pid],
        "대표 식생·분류군 예시":PFT_EXAMPLES_KO[pid],
    }
    for pid in range(1,14)
])


pft_reference_export=pd.DataFrame([
    {
        "PFT":pid,
        "BIOME4 원명":PFT_NAMES_EN[pid],
        "국문명":PFT_NAMES[pid],
        "대표 식생·분류군 예시":PFT_EXAMPLES_KO[pid],
        "근거":PFT_EXAMPLE_SOURCE[pid],
    }
    for pid in range(1,14)
])
pft_reference_export.to_csv(
    report_dir/"pft_reference_examples.csv",
    index=False,encoding="utf-8-sig"
)

biome_reference_table=pd.DataFrame([
    {
        "ID":i,
        "BIOME4 원명":BIOME4_NAMES[i],
        "국문명":BIOME4_NAMES_KO[i],
        "대표 식생 예시":BIOME4_EXAMPLES_KO[i],
        "설명":BIOME4_DESCRIPTIONS_KO[i],
        "근거":BIOME4_EXAMPLE_SOURCE[i],
    }
    for i in range(1,29)
])
biome_reference_table.to_csv(
    comparison_report/"biome4_reference_examples.csv",
    index=False,encoding="utf-8-sig"
)

VEGETATION_SOURCE_NOTES={
    "Bradshaw et al. (2025)":"BIOME4 13개 PFT와 28개 biome 명칭",
    "Kaplan et al. (2002)":"BIOME4 PFT와 biome 판정",
    "Kaplan et al. (2003)":"고위도 PFT와 tundra Typical Taxa",
    "Ni (2000)":"BIOME 계열 식생형과 대표 우점분류군",
    "Ni & Herzschuh (2011)":"Tibetan steppe/tundra 식생과 PFT",
    "Edwards et al. (2010)":"열대·아열대 C4 초원·사바나",
}
vegetation_source_table=pd.DataFrame([
    {"인용":k,"확인 내용":v}
    for k,v in VEGETATION_SOURCE_NOTES.items()
])
vegetation_source_table.to_csv(
    comparison_report/"vegetation_example_sources.csv",
    index=False,encoding="utf-8-sig"
)

download_report_zip = False #@param {type:"boolean"}
from zipfile import ZipFile,ZIP_DEFLATED
from pathlib import Path

template_path=comparison_report/"student_report_template.md"
template_text=f"""# 제1절 생태환경과 지리: 실습 4 경관진화모델링 보고서

## 1. 실험 목적
- 시나리오 수: {scenario_count}

## 2. 시나리오 설정
- 표: scenario_settings.csv
- 계산기간: {start_ka:g}–{end_ka:g} ka BP
- 시나리오별 모델 유형과 설정값:

## 3. 시계열 비교
- 표: all_scenarios_timeseries.csv
- 시나리오별 토심·고도·NPP 변화:

## 4. BIOME4 biome 및 PFT
- 표: biome4_biome_ratio_all_times.csv
- 참고표: biome4_reference_examples.csv
- 표: pft_all_times.csv
- 표: selected_pft_grid_ratio_all_times.csv
- 참고표: pft_reference_examples.csv
- 출처표: vegetation_example_sources.csv
- BIOME4 biome과 최적 PFT의 변화:

## 5. 5 kyr 간격 및 종료시점 비교
- 표: valley_hillslope_biome_overlap_5k.csv
- 그림: valley_area_percent_timeseries.png
- 표: scenario_5k_results.csv
- 표: scenario_end_results.csv
- 시나리오별 결과 비교:

## 6. 토의
- 기후·식생·토심·사면과정의 상호작용:
- 시나리오에서 변경한 조건과 유지한 조건:
- 모형 결과 해석의 한계:

## 7. 사용 모형 및 기후자료
- 모형: BIOME4–Pelletier
- 기후자료: Beyer 120–0 ka BP
- EEMT 저온 처리: {EEMT_COLD_PHASE_METHOD}

"""
template_path.write_text(template_text,encoding="utf-8")


# 결과 엑셀 및 변수 설명
import re
import pandas as pd

variable_defs={
    "scenario":("시나리오","시나리오 이름",""),
    "model":("모델 유형","정적 또는 동적 모델",""),
    "scenario_number":("시나리오 번호","설정한 시나리오의 순서",""),
    "scenario_count":("시나리오 수","전체 시나리오 개수","개"),
    "target_basin_area_km2":("목표 유역면적","가상유역 생성 시 사용자가 지정한 목표 면적","km²"),
    "actual_basin_area_km2":("실제 유역면적","선택된 D8 유출구의 상류 기여역 면적","km²"),
    "grid_size":("격자 배열 크기","정사각형 계산영역의 한 변 격자 수","개"),
    "cell_size_m":("격자 해상도","격자 한 변의 길이","m"),
    "bounding_box_width_m":("경계상자 폭","가상유역 계산영역의 한 변 길이","m"),
    "valid_cell_count":("유역 내부 격자 수","D8로 delineation된 유역 내부 유효 격자 수","개"),
    "basin_area_m2":("실제 유역면적","유효 격자 수와 해상도로 계산한 유역면적","m²"),
    "name":("시나리오 이름","시나리오를 구별하는 이름",""),
    "koppen":("쾨펜 기후형","현재 기준 쾨펜 기후형 코드",""),
    "representative_id":("기후 지점 ID","기후자료 지점 식별자",""),
    "기후 지역":("기후 지역","선택한 기후자료의 지역명",""),
    "기후 지점":("기후 지점","지역명·좌표·ID",""),
    "soil_depth_multiplier":("초기 토심 배율","초기 토심 자료에 적용한 배율","배"),
    "uplift_m_per_kyr":("융기율","천 년당 지표 융기량","m/kyr"),
    "soil_production_a":("토양생산 계수 a","EEMT 기반 토양생산식의 계수 a",""),
    "hillslope_c":("사면수송 계수 c","EEMT에 대응하는 사면수송식의 계수 c",""),
    "hillslope_d":("사면수송 계수 d","지상부 바이오매스에 대응하는 사면수송식의 계수 d",""),
    "fluvial_K0":("유수침식 계수 K0","유수침식 계산에 사용하는 기준 계수",""),
    "mean_temp_C":("연평균 기온","12개월 평균기온의 산술평균","°C"),
    "annual_precip_mm":("연강수량","12개월 월강수량의 합","mm/yr"),
    "mean_cloud_pct":("연평균 운량","월평균 운량의 연평균","%"),
    "ka_bp":("현재 이전 시기","현재를 기준으로 한 과거 시점","ka BP"),
    "resolution_m":("격자 해상도","공간격자의 한 변 길이","m"),
    "n_cells":("유효 격자 수","모의에 포함된 유효 격자의 개수","개"),
    "area_m2":("유효 면적","유효 격자의 총면적","m²"),
    "relief_m":("기복량","최고 고도와 최저 고도의 차이","m"),
    "dominant_vegetation_code":("우점 식생 코드","PB4Studio 내부 집계에 사용된 우점 식생 코드",""),
    "dominant_biome4_id":("우점 BIOME4 생물군계","BIOME4 원본 생물군계의 최빈 ID",""),
    "biome4_full_id_counts":("BIOME4 생물군계별 격자 수","생물군계 ID와 격자 수를 콜론 및 세미콜론으로 구분한 값",""),
    "vegetation_class_richness":("식생 유형 수","모의 영역에 나타난 식생 유형의 개수","개"),
    "vegetation_min_code":("최소 식생 코드","PB4Studio 내부 식생 코드의 최솟값",""),
    "vegetation_max_code":("최대 식생 코드","PB4Studio 내부 식생 코드의 최댓값",""),
    "vegetation_class_counts":("식생 유형별 격자 수","PB4Studio 내부 식생 유형의 격자 수 집계",""),
    "vegetation_nonvegetated_cell_count":("비식생 격자 수","내부 식생 분류상 비식생에 해당하는 격자 수","개"),
    "vegetation_unclassified_cell_count":("미분류 격자 수","식생 유형이 판정되지 않은 격자 수","개"),
    "vegetation_unclassified_fraction":("미분류 격자 비율","유효 격자 중 식생 미분류 비중","비율"),
    "case_label":("실행 구분","모델 실행의 내부 구분",""),
    "biome_id":("BIOME4 생물군계 ID","BIOME4 생물군계 분류 코드(1–28)",""),
    "biome_name":("생물군계 영문명","BIOME4 생물군계의 영문 명칭",""),
    "biome_name_ko":("생물군계 국문명","BIOME4 생물군계의 국문 명칭",""),
    "biome_description_ko":("생물군계 설명","생물군계의 생태적 특성에 관한 국문 설명",""),
    "biome_example_ko":("생물군계 대표 식생 예시","해당 BIOME4 생물군계의 대표 식생 예",""),
    "biome_example_source":("생물군계 예시 근거","대표 식생 예시를 확인한 문헌",""),
    "pft_example_ko":("PFT 대표 식생 예시","해당 PFT의 대표 식생·분류군 예",""),
    "pft_example_source":("PFT 예시 근거","대표 식생 예시를 확인한 문헌",""),
    "cells":("생물군계 격자 수","해당 생물군계가 나타난 격자의 개수","개"),
    "area_percent":("생물군계 면적비율","유효 격자에서 해당 생물군계가 차지하는 면적비율","%"),
    "pft_id":("식물기능유형 ID","BIOME4 식물기능유형(PFT) 번호",""),
    "pft_name":("식물기능유형명","해당 식물기능유형의 국문 명칭",""),
    "potential_npp":("잠재 순일차생산량","해당 PFT의 잠재 순일차생산량","gC m⁻² yr⁻¹"),
    "potential_lai":("잠재 엽면적지수","해당 PFT의 잠재 엽면적지수","m²/m²"),
    "constraint_pass_fraction":("생물기후 제약 통과 비율","해당 PFT가 생물기후적 제약조건을 만족한 격자 비율","비율"),
    "chosen_cells":("optPFT 격자 수","해당 BIOME4 optPFT 코드가 나타난 격자의 수","개"),
    "chosen_cells_percent":("optPFT 격자 비율","해당 BIOME4 optPFT 코드가 나타난 격자의 비율","%"),
    "mean_agb_proxy_kg_m2":("평균 지상부 바이오매스 지표","Pelletier 결합에 사용되는 NPP 기반 지상부 바이오매스 지표의 공간평균","kg/m²"),
    "mean_lai":("평균 엽면적지수","유효 격자의 평균 LAI","m²/m²"),
    "mean_aet_mm_yr":("평균 실제증발산량","연간 실제증발산량의 공간평균","mm/yr"),
    "mean_runoff_mm_yr":("평균 유출량","연간 지표 유출량의 공간평균","mm/yr"),
}
physical={
    "elevation_m":("고도","m"),
    "soil_depth_m":("토심","m"),
    "slope":("경사도",""),
    "npp":("순일차생산량","gC m⁻² yr⁻¹"),
    "eemt":("유효 에너지 및 물질 전달량","MJ m⁻² yr⁻¹"),
    "lai":("엽면적지수","m²/m²"),
    "agb":("지상부 바이오매스","kg/m²"),
}
for key,(ko,unit) in physical.items():
    for prefix,stat in (
        ("mean_", "평균"),("sd_","표준편차"),
        ("min_","최솟값"),("max_","최댓값")
    ):
        variable_defs[prefix+key]=(stat+" "+ko,ko+"의 공간 "+stat,unit)
    variable_defs[key+"_mean_sd"]=(ko+" 평균 ± 표준편차",ko+"의 공간평균 및 표준편차를 함께 표시한 문자열",unit)

def describe_variable(col):
    if col in variable_defs:
        return variable_defs[col]
    m=re.fullmatch(r"pft(\d{2})_(npp_mean|lai_mean|constraint_pass_fraction)",col)
    if m:
        pid=int(m.group(1))
        part=m.group(2)
        ko={"npp_mean":"잠재 순일차생산량 평균",
            "lai_mean":"잠재 엽면적지수 평균",
            "constraint_pass_fraction":"생물기후 제약 통과 비율"}[part]
        unit={"npp_mean":"gC m⁻² yr⁻¹","lai_mean":"m²/m²",
              "constraint_pass_fraction":"비율"}[part]
        return (f"PFT {pid} {ko}",f"식물기능유형 {pid} ({PFT_NAMES.get(pid,'')})의 {ko}",unit)
    m=re.fullmatch(r"selected_pft(\d{2})_count",col)
    if m:
        pid=int(m.group(1))
        if pid==0:
            return ("optPFT 0 격자 수","BIOME4 원본 optPFT가 0(미선택)인 격자 수","개")
        if pid==14:
            return ("optPFT 14 격자 수","BIOME4 원본 optPFT가 14(목본·초본 공우점 코드)인 격자 수","개")
        return (f"PFT {pid} 선택 격자 수",f"BIOME4 원본 optPFT에서 PFT {pid}가 선택된 격자 수","개")
    if col.startswith("geomorph_"):
        gloss={
            "accepted_substeps_to_next":("허용된 계산 소단계 수","개"),
            "rejected_substeps_to_next":("기각된 계산 소단계 수","개"),
            "rejected_fluvial_trials_to_next":("기각된 유수침식 계산 횟수","회"),
            "min_dt_kyr_to_next":("최소 내부 시간간격","kyr"),
            "max_dt_kyr_to_next":("최대 내부 시간간격","kyr"),
            "fluvial_tolerance_m":("유수침식 허용오차","m"),
            "max_accepted_fluvial_change_m_to_next":("최대 허용 유수침식 고도변화","m"),
        }
        suffix=col[len("geomorph_"):]
        if suffix in gloss:
            ko,unit=gloss[suffix]
            return (ko,"지형발달 계산 중 "+ko,unit)
    return ("변수 정의 확인 필요","원본 모델 출력 변수: "+col,"")

export_tables={
    "가상유역 설정":virtual_basin_table,
    "시나리오 설정":settings_table,
    "생물군계 참고표":biome_reference_table,
    "PFT 참고표":pft_reference_table,
    "식생 예시 출처":vegetation_source_table,
    "5천년 간격":comparison_5k_table,
    "종료시점":end_table,
    "시계열":scenario_timeseries,
    "생물군계 비율":biome_ratios,
    "PFT 잠재생산":pft_long,
    "PFT 선택비율":chosen_pft,
    "PFT 5천년 진단":pft_5k,
    "PFT 5천년 선택":chosen_pft_5k,
    "하도사면 5천년":landform_timeseries,
    "연간 기후":scenario_climate_annual,
}
dictionary_rows=[]
for sheet,frame in export_tables.items():
    for column in frame.columns:
        ko,meaning,unit=describe_variable(str(column))
        dictionary_rows.append({
            "자료 시트":sheet,
            "변수명":str(column),
            "국문명":ko,
            "국문 설명":meaning,
            "단위":unit
        })
variable_dictionary=pd.DataFrame(dictionary_rows)
xlsx_path=comparison_report/"global_climate_scenario_results.xlsx"
with pd.ExcelWriter(xlsx_path,engine="xlsxwriter") as writer:
    variable_dictionary.to_excel(writer,sheet_name="변수 설명",index=False)
    for sheet,frame in export_tables.items():
        frame.to_excel(writer,sheet_name=sheet,index=False)
    for sheet in ["변수 설명",*export_tables]:
        worksheet=writer.sheets[sheet]
        worksheet.freeze_panes(1,0)
        worksheet.autofilter(0,0,
            (len(variable_dictionary) if sheet=="변수 설명" else len(export_tables[sheet])),
            (len(variable_dictionary.columns) if sheet=="변수 설명" else len(export_tables[sheet].columns))-1
        )
        if sheet=="변수 설명":
            worksheet.set_column(0,0,19)
            worksheet.set_column(1,1,37)
            worksheet.set_column(2,2,34)
            worksheet.set_column(3,3,75)
            worksheet.set_column(4,4,20)
        else:
            worksheet.set_column(0,0,19)
            worksheet.set_column(1,min(24,len(export_tables[sheet].columns)-1),17)
print("결과 엑셀:",xlsx_path)

export_zip=exp_root/"global_climate_virtual_basin_lab_report.zip"
with ZipFile(export_zip,"w",ZIP_DEFLATED) as archive:
    for file in sorted(comparison_report.rglob("*")):
        if file.is_file() and file.suffix.lower() in (".csv",".png",".md",".json",".xlsx"):
            archive.write(
                file,
                Path("comparison_report")/file.relative_to(comparison_report)
            )
    climate_summary=exp_root/"climate_forcing_annual.csv"
    if climate_summary.is_file():
        archive.write(climate_summary,"climate_forcing_annual.csv")
    if registry_path.is_file():
        archive.write(registry_path,"scenario_registry.json")
    for sc in scenario_configs:
        inputs=scenario_paths[sc["name"]]["inputs"]
        for p in sorted(inputs.glob("*")):
            if p.is_file() and p.suffix.lower() in (".csv",".geojson",".tif"):
                archive.write(p,Path(sc["name"])/"inputs"/p.name)
        w=scenario_paths[sc["name"]]["work"]
        for fn in ("teaching_settings.json","run_config.json"):
            p=w/fn
            if p.is_file():
                archive.write(p,Path(sc["name"])/fn)

print("보고서 자료 ZIP:",export_zip)
if download_report_zip:
    try:
        from google.colab import files
        files.download(str(export_zip))
    except ImportError:
        from IPython.display import FileLink
        display(FileLink(str(export_zip)))
else:
    print("내려받으려면 download_report_zip에 체크하고 이 셀을 다시 실행합니다.")
